# ROGII 最終提出 — A/B 2モード(SUBMIT_MODE を A/B に変え2回実行)
全脚を test で再現し blend_final.json の重みで合成。**SUBMIT_MODE を A/B に変え2回実行**で2ファイル。
- **A(v104無関係)** = L5grid + TCN + anchor
- **B(最良・v104少量)** = A + greedy(v104 貪欲サブセット, decaf含む=dec10af 生成)
脚: L5grid=v103 grid(91候補) / greedy=v104 keep27(103候補) / TCN=v103 / anchor=物理錨δ。各モード後に expansion 後処理。
持ち出し = `final_submit_dataset` + 競技データ。GPU=**T4**推奨。


In [ ]:
# ===== 提出モード選択(A/B を2回実行して2ファイル作る)=====
SUBMIT_MODE = "A"   # ← 1回目 "B"(最良)→提出、2回目 "A"(v104無関係)に変え再実行→提出


In [ ]:
# ===== 環境 + 検出 + env =====
import os, sys, glob
from pathlib import Path
WORK = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path(".").resolve()
def detect_data():
    for c in ["/kaggle/input/rogii-wellbore-geology-prediction","/kaggle/input/competitions/rogii-wellbore-geology-prediction"]:
        if os.path.isdir(os.path.join(c,"test")): return c
    for g in sorted(glob.glob("/kaggle/input/**/sample_submission.csv",recursive=True)): return os.path.dirname(g)
    return r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction"
def detect_ds():
    for g in sorted(glob.glob("/kaggle/input/datasets/k256net/rogii-artifact-v103-final/final_submit_dataset/blend_final.json",recursive=True)): return os.path.dirname(g)
    for g in sorted(glob.glob("/kaggle/input/datasets/k256net/rogii-artifact-v103-final/final_submit_dataset/dec10/pf_banks_config.json",recursive=True)): return os.path.dirname(os.path.dirname(g))
    return None
DATA=detect_data(); DSPATH=detect_ds()
print("DATA=",DATA,"| final_submit_dataset=",DSPATH,"| SUBMIT_MODE=",SUBMIT_MODE)
assert SUBMIT_MODE in ("A","B"), "SUBMIT_MODE は A か B"
assert os.path.isdir(os.path.join(DATA,"test")), "競技データ未追加"
assert DSPATH is not None, "final_submit_dataset 未追加"
assert os.path.isfile(os.path.join(DSPATH,"blend_final.json")), "blend_final.json 未添付"
os.chdir(WORK); sys.path.insert(0,str(WORK))
import json as _json
# L5grid = 存在する grid ckpt(+ det)
_cfg = [[-1,5]] + [c for c in [[0, 5], [1, 5], [2, 5], [0, 8], [1, 8], [2, 8], [0, 10], [1, 10], [2, 10]] if os.path.isdir(os.path.join(DSPATH, f"l5_v103g{c[0]}k{c[1]}_ckpt"))]
os.environ.update(dict(ROGII_DATA=str(DATA), ROGII_PROJ=str(WORK), ROGII_SCRIPTS=str(WORK), ROGII_PY=sys.executable,
    V100SUB=str(DSPATH), V103SUB=str(DSPATH), V104SUB=str(DSPATH), FULL_VRAM_GB="8.0", PYTHONUNBUFFERED="1",
    V103_L5_CONFIGS=_json.dumps(_cfg), NN_SEEDS="5", N_SPLITS="5", SUBMIT_MODE=SUBMIT_MODE, V103_TCN="1"))
assert os.path.isdir(os.path.join(DSPATH,"dec10af")), "dec10af 未添付(greedy の 103候補再現に必須)"
import torch as _t
_ng=_t.cuda.device_count(); os.environ["PF_NGPU"]=str(max(1,_ng))
print(f"[env] L5grid構成={len(_cfg)} greedy={'有' if os.path.isdir(os.path.join(DSPATH,'l5_v104greedyg-1_ckpt')) else '無'} dec10af={'有' if os.path.isdir(os.path.join(DSPATH,'dec10af')) else '無'}")
print("[GPU]",_ng,[_t.cuda.get_device_name(i) for i in range(_ng)])


In [ ]:
%%writefile gen_grfree_anchor.py
import io,sys,os,pickle
try: sys.stdout=io.open(1,"w",encoding="utf-8",closefd=False)
except: pass
# -*- coding: utf-8 -*-
"""GRフリー自己完結 v2 — ★val完全除外の正しい5-fold。
   fold毎に [場F・κ・近傍プール・GRU] の全てを train foldの井だけで構築 → val fold予測。
   完全ランダムshuffle分割。fold別val CVを表示。GR一切不使用(X,Y,Z,TVT,TVT_inputのみ)。
   test = 5fold×3seed=15セット(各foldのF/κ/GRU)平均。"""
import os, glob, time, io, sys
import numpy as np, pandas as pd
t00 = time.time()
KAGGLE = os.path.exists('/kaggle/input')
def _resolve_root():
    # ★呼び出し元(ノート/submit)が渡す ROGII_DATA を最優先
    r = os.environ.get('ROGII_DATA')
    if r and os.path.isdir(os.path.join(r, 'train')):
        return r
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(os.path.join(c, 'train')):
            return c
    g = glob.glob('/kaggle/input/*rogii*wellbore*') or glob.glob('/kaggle/input/**/train', recursive=True)
    if g:
        return g[0][:-6] if g[0].endswith('/train') else g[0]
    return r'c:/Users/kosaka256/Documents/rogii_claude/rogii-wellbore-geology-prediction'
ROOT = _resolve_root()
TRAIN_DIR = os.path.join(ROOT, 'train'); TEST_DIR = os.path.join(ROOT, 'test'); SAMPLE = os.path.join(ROOT, 'sample_submission.csv')
PGATE = 0.35; THETA0 = 118.4; KNN_K, KNN_H = 15, 500.0
KBINS = [0.0, 750.0, 1500.0, 2500.0, 4000.0, 1e18]; KAPPA_REGIMES = [0.0, 1000.0, 1500.0, 2000.0]
NBIN = len(KBINS) - 1; CMAX = 0.30; K_FIX = 16; CLIP_DRIFT = 150.0
N_FOLDS = 5; NBK = 3; STRIDE = 2; HID = 96; C = 37
FT_EP = int(os.environ.get('FT_EP', '10')); FT_LR = 1e-3; SEEDS = int(os.environ.get('SEEDS', '3'))
SPLIT_SEED = int(os.environ.get('SPLIT_SEED', '42'))
WINS = [301, 1001, 2001, 4001]; _C0, _S0 = np.cos(np.radians(THETA0)), np.sin(np.radians(THETA0))
def lowpass(x, win): return pd.Series(x).rolling(int(win) | 1, center=True, min_periods=1).mean().to_numpy()

# ===== エンジン(phaseG lean, GR不使用) =====
def make_gdict(X, Y, Z, ti, tvt=None):
    fin = np.where(np.isfinite(ti))[0]
    if len(fin) == 0: return None
    s = int(fin.max()); n = len(X) - 1 - s
    if n < 2: return None
    dX = np.diff(X)[s:]; dY = np.diff(Y)[s:]; Lxy = np.maximum(np.hypot(dX, dY), 1e-3)
    p_row = (dX * _C0 + dY * _S0) / Lxy
    d = dict(s=s, n=n, dX=dX, dY=dY, ndz=-np.diff(Z)[s:], Lxy=Lxy, arc=np.cumsum(Lxy),
             Xl=X[s:], Yl=Y[s:], anchor=float(ti[s]), perp=np.abs(p_row) < PGATE)
    if tvt is not None: d['R0'] = tvt[s + 1:] - tvt[s]
    return d
def segment_well(wd, with_slopes):
    n, A = wd['n'], wd['arc']; K = min(K_FIX, max(1, n // 4)); total = float(A[-1])
    edges = np.linspace(0.0, total, K + 1); segid = np.clip(np.searchsorted(edges[1:], A, side='left'), 0, K - 1)
    mid = np.empty((K, 2)); az = np.empty(K); Xl, Yl = wd['Xl'], wd['Yl']
    for j in range(K):
        rows = np.where(segid == j)[0]; j0, j1 = (int(rows[0]), int(rows[-1])) if len(rows) else (0, n - 1)
        az[j] = np.arctan2(Yl[j1 + 1] - Yl[j0], Xl[j1 + 1] - Xl[j0]); mid[j] = ((Xl[j0] + Xl[j1 + 1]) / 2.0, (Yl[j0] + Yl[j1 + 1]) / 2.0)
    if not with_slopes: return segid, mid, az, None, None
    phi = np.column_stack([np.clip(A - edges[j], 0.0, edges[j + 1] - edges[j]) for j in range(K)])
    y = wd['R0'] - np.cumsum(wd['ndz']); m = np.isfinite(y)
    if m.sum() < K + 2: return segid, mid, az, None, None
    c, res, rank, _ = np.linalg.lstsq(phi[m], y[m], rcond=None); dof = max(int(m.sum()) - K, 1)
    ssr = float(res[0]) if len(res) else float(np.sum((phi[m] @ c - y[m]) ** 2))
    cov = (ssr / dof) * np.linalg.pinv(phi[m].T @ phi[m]); se = np.sqrt(np.maximum(np.diag(cov), 1e-12))
    return segid, mid, az, c, se
def query_well(F, mids, own, mdbuf=0.0):
    keep = F['wi'] != own; fx, fy = F['x'][keep], F['y'][keep]; fc = F['c'][keep]; fp0 = F['p0'][keep]; fqw = F['qw'][keep]
    dx = fx[None, :] - mids[:, 0:1]; dy = fy[None, :] - mids[:, 1:2]; d2 = dx * dx + dy * dy
    if mdbuf > 0: d2 = np.where(d2 >= mdbuf * mdbuf, d2, np.inf)
    kk = min(KNN_K, d2.shape[1] - 1) if d2.shape[1] > 1 else 1
    idx = np.argpartition(d2, kk - 1, axis=1)[:, :kk]; r = np.arange(len(mids))[:, None]
    d2s = d2[r, idx]; c_s, p0_s = fc[idx], fp0[idx]
    w = np.exp(np.maximum(-d2s / (2 * KNN_H ** 2), -700.0)) * fqw[idx]; w[~np.isfinite(d2s)] = 0.0
    dead = w.sum(1) <= 1e-300
    with np.errstate(invalid='ignore'): dd = np.sqrt(np.nanmedian(np.where(np.isfinite(d2s), d2s, np.nan), axis=1))
    dd[~np.isfinite(dd)] = KBINS[-2] * 2
    s_hat = (w * c_s * p0_s).sum(1) / ((w * p0_s ** 2).sum(1) + 1e-9); g = np.column_stack([s_hat * _C0, s_hat * _S0]); g[dead] = 0.0
    gn = np.hypot(g[:, 0], g[:, 1]); big = gn > CMAX
    if big.any(): g[big] *= (CMAX / gn[big])[:, None]
    pred_don = g[:, 0:1] * np.cos(F['az'][keep][idx]) + g[:, 1:2] * np.sin(F['az'][keep][idx])
    spread = np.sqrt(np.maximum((w * (c_s - pred_don) ** 2).sum(1) / np.maximum(w.sum(1), 1e-12), 0.0)); spread[dead] = 0.0
    return g, dd, spread
def design_blocks(F, wd, mdbuf):
    g, dd, _ = query_well(F, wd['seg'][1], wd['wi'], mdbuf=mdbuf); sid = wd['seg'][0]
    f = g[sid, 0] * wd['dX'] + g[sid, 1] * wd['dY']; a = wd['ndz']; par = ~wd['perp']
    b_row = np.digitize(dd, KBINS[1:-1])[sid]; cols = []
    for b in range(NBIN):
        m = (b_row == b) & par; cols.append(np.cumsum(np.where(m, a, 0.0))); cols.append(np.cumsum(np.where(m, f, 0.0)))
    cols.append(np.cumsum(np.where(wd['perp'], a, 0.0))); cols.append(np.cumsum(np.where(wd['perp'], f, 0.0)))
    G = np.column_stack(cols); ok = np.isfinite(wd['R0']); G = np.nan_to_num(G[ok]); return G.T @ G, G.T @ wd['R0'][ok]
def solve_kappa(A, yv):
    npar = 2 * NBIN; ncol = npar + 2; A = A.copy(); lam_p = 0.02 * float(np.trace(A)) / ncol
    A[npar, npar] += lam_p; A[npar + 1, npar + 1] += lam_p; coef = np.linalg.lstsq(A, yv, rcond=None)[0]
    return dict(alpha=np.clip(coef[0:npar:2], -0.25, 1.5), beta=np.clip(coef[1:npar:2], -0.25, 1.5),
                ap=float(np.clip(coef[npar], -0.25, 1.5)), bp=float(np.clip(coef[npar + 1], -0.25, 1.5)))
def engine_predict(F, wd, K, own=-1):
    g, dd, spread = query_well(F, wd['seg'][1], own, mdbuf=0.0); sid = wd['seg'][0]
    f = g[sid, 0] * wd['dX'] + g[sid, 1] * wd['dY']; b_row = np.digitize(dd, KBINS[1:-1])[sid]
    av = K['alpha'][b_row]; bv = K['beta'][b_row]; av = np.where(wd['perp'], K['ap'], av); bv = np.where(wd['perp'], K['bp'], bv)
    rate = av * wd['ndz'] + bv * f; resid = np.clip(np.cumsum(rate), -CLIP_DRIFT, CLIP_DRIFT)
    return wd['anchor'] + resid, rate, dd[sid], spread[sid]

def build_field(wells):
    px, py, pc, paz, pwi = [], [], [], [], []
    for wd in wells:
        if wd['seg'][3] is None: continue
        segid, mid, az, c, se = wd['seg']; ok = np.isfinite(c) & (np.abs(c) <= CMAX) & np.isfinite(se)
        px.append(mid[ok, 0]); py.append(mid[ok, 1]); pc.append(c[ok]); paz.append(az[ok]); pwi.append(np.full(int(ok.sum()), wd['wi'], float))
    F = dict(x=np.concatenate(px), y=np.concatenate(py), c=np.concatenate(pc), az=np.concatenate(paz), wi=np.concatenate(pwi))
    F['qw'] = np.ones(len(F['x'])); F['p0'] = np.cos(F['az']) * _C0 + np.sin(F['az']) * _S0
    return F
def fit_kappa(F, wells):
    A = 0; yv = 0
    for wd in wells:
        if wd['seg'][3] is None: continue
        for R in KAPPA_REGIMES:
            Ab, yb = design_blocks(F, wd, R); A = A + Ab; yv = yv + yb
    return solve_kappa(A, yv)
def build_pool(wells):
    ids = [wd['wid'] for wd in wells]
    cent = np.array([[np.nanmean(wd['_X']), np.nanmean(wd['_Y'])] for wd in wells])
    from scipy.spatial import cKDTree
    trees = {wd['wid']: cKDTree(np.c_[wd['_X'], wd['_Y']][::2]) for wd in wells}
    gks = {wd['wid']: lowpass((wd['_tvt'] + wd['_Z']).astype(float), 201) for wd in wells}
    mdc = {wd['wid']: np.arange(len(wd['_X']), dtype=float) for wd in wells}
    yx = {wd['wid']: (wd['_X'].astype(float), wd['_Y'].astype(float)) for wd in wells}
    return dict(ids=ids, cent=cent, trees=trees, gks=gks, md=mdc, yx=yx)

def build_channels(md, x, y, z, tin, t0, d7_drift, rates_full, conf_full, POOL, self_wid=None):
    n = len(md); ei = np.arange(t0 + 1, n)[::STRIDE].astype(int); tvt_last = float(tin[t0])
    d7 = d7_drift[(ei - (t0 + 1))]; rates = rates_full[ei]; conf = conf_full[ei]
    g = np.where(np.isfinite(tin), tin, np.nan) + z; ki = np.where(np.isfinite(tin))[0]
    zh301 = z - lowpass(z, 301); tl = ki[-min(len(ki), 1000):]; gt = (g - lowpass(g, 301))[tl]; zt = zh301[tl]
    fin = np.isfinite(gt); vz = float(np.dot(zt[fin], zt[fin])); k_pref = float(np.clip(np.dot(zt[fin], gt[fin]) / vz, 0, 1.5)) if vz > 1e-9 else 0.0
    az = np.arctan2(np.gradient(y), np.gradient(x)); az_s = np.arctan2(lowpass(np.sin(az), 301), lowpass(np.cos(az), 301))
    bands = {W: lowpass(z, W) for W in WINS}; msb = (np.arange(n) - t0).astype(float)
    dh = np.hypot(np.gradient(lowpass(x, 101)), np.gradient(lowpass(y, 101))) + 1e-9
    incl = np.arctan2(np.gradient(lowpass(z, 301)), dh); build_r = np.gradient(lowpass(incl, 201)) * 1000
    g_s = lowpass(g, 201); r_all = np.gradient(np.nan_to_num(g_s, nan=0.0)) / np.maximum(dh, 1e-6)
    tlp = ki[(ki >= t0 - 1500)]; rvals = r_all[tlp]; rvals = rvals[np.isfinite(g_s[tlp])]
    r_pref = float(np.clip(np.median(rvals), -0.08, 0.08)) if len(rvals) > 100 else 0.0
    dprior = (r_pref * (md - md[t0]) - (z - z[t0])); kn_flag = np.zeros(len(ei), np.float32)
    ch = [d7 / 20.0, rates * 20.0, conf, kn_flag, msb[ei] / 3000.0, np.sin(az_s[ei]), np.cos(az_s[ei]),
          np.full(len(ei), k_pref), incl[ei] * 3, build_r[ei], (incl[ei] - incl[t0]) * 3, np.clip(dprior[ei], -120, 120) / 30.0]
    for W in WINS: ch.append((bands[W][ei] - bands[W][t0]) / 50.0); ch.append((z - bands[W])[ei] / 10.0)
    cent = POOL['cent']; ids = POOL['ids']
    dd0 = np.hypot(cent[:, 0] - np.nanmean(x), cent[:, 1] - np.nanmean(y)); order = np.argsort(dd0); nbrs = []
    for j in order:
        if self_wid is not None and ids[j] == self_wid: continue
        nbrs.append(ids[j])
        if len(nbrs) == NBK: break
    p0 = np.array([x[t0], y[t0]]); g0 = float(tin[t0] + z[t0])
    for nw in nbrs:
        tr = POOL['trees'][nw]; gks = POOL['gks'][nw]; mdn = POOL['md'][nw]; nx, ny = POOL['yx'][nw]
        _, j0 = tr.query(p0); j0 = int(j0) * 2
        dq, jj = tr.query(np.c_[x[ei], y[ei]]); jj = (np.asarray(jj) * 2).astype(int)
        dipk = np.gradient(gks, mdn, edge_order=1)
        ch += [np.clip(gks[jj] - gks[j0], -120, 120) / 30.0, np.asarray(dq) / 1000.0, dipk[jj] * 30,
               np.cos(az_s[ei] - np.arctan2(np.gradient(ny)[jj], np.gradient(nx)[jj])),
               np.full(len(ei), float(np.clip(gks[j0] - g0, -120, 120)) / 30.0)]
    ch += [np.zeros(len(ei), np.float32), np.zeros(len(ei), np.float32)]
    X = np.nan_to_num(np.stack(ch, 1).astype(np.float32)); return X, ei, tvt_last

def wseq(gd, F, K, POOL, own):
    X_, Y_, Z_, ti, tvt = gd['_X'], gd['_Y'], gd['_Z'], gd['_ti'], gd['_tvt']
    n = len(X_); d7_full, rate, ddr, spr = engine_predict(F, gd, K, own=own); t0w = gd['s']
    rates_full = np.zeros(n, np.float32); rates_full[t0w + 1:] = rate
    conf_full = np.zeros(n, np.float32)
    conf_full[t0w + 1:] = np.exp(-np.maximum(ddr - 800.0, 0.0) / 1000.0) * np.exp(-np.nan_to_num(spr, nan=0.1) / 0.06)
    md = np.arange(n, dtype=float)
    X, ei, tvt_last = build_channels(md, X_, Y_, Z_, ti, t0w, (d7_full - float(ti[t0w])), rates_full, conf_full, POOL, self_wid=gd['wid'])
    target = (tvt[ei] - d7_full[(ei - (t0w + 1))]).astype(np.float32)
    return X, ei, target, d7_full, t0w

# ===== torch / モデル定義(ロード経路でも必要なので先に定義) =====
import torch, torch.nn as nn
from sklearn.model_selection import KFold
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
class Net(nn.Module):
    def __init__(s):
        super().__init__(); s.inp = nn.Linear(C + 1, HID); s.gru = nn.GRU(HID, HID, 2, batch_first=True, bidirectional=True, dropout=0.1); s.out = nn.Linear(HID * 2, 1)
    def forward(s, x):
        h = torch.relu(s.inp(x)); h, _ = s.gru(h); return s.out(h)[..., 0]
def train_loop(net, items, seed):
    opt = torch.optim.AdamW(net.parameters(), lr=FT_LR, weight_decay=1e-4); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, FT_EP)
    rng = np.random.default_rng(seed); BS = 12
    for ep in range(FT_EP):
        net.train(); order = rng.permutation(len(items))
        for b0 in range(0, len(order), BS):
            batch = [items[i] for i in order[b0:b0 + BS]]; L = max(len(b[1]) for b in batch)
            xb = np.zeros((len(batch), L, C + 1), np.float32); yb = np.zeros((len(batch), L), np.float32); mb = np.zeros((len(batch), L), np.float32)
            for i, (Xx, yv, fl) in enumerate(batch):
                xb[i, :len(yv), :C] = Xx; xb[i, :len(yv), C] = fl; yb[i, :len(yv)] = yv; mb[i, :len(yv)] = 1.0
            xb = torch.from_numpy(xb).to(dev); yb = torch.from_numpy(yb).to(dev); mb = torch.from_numpy(mb).to(dev)
            loss = (nn.functional.huber_loss(net(xb), yb, delta=8.0, reduction='none') * mb).sum() / mb.sum()
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step()
        sch.step()
    return net

def _rebuild_pool_trees(P):   # 保存時に外した cKDTree を yx から再構築(scipyバージョン非依存で再現一致)
    from scipy.spatial import cKDTree
    if P.get('trees') is None:
        P['trees'] = {wid: cKDTree(np.c_[x, y][::2]) for wid, (x, y) in P['yx'].items()}
    return P

# ===== 事前学習済み FOLD_ART があればロード → train全ロード+GRU学習を丸ごとスキップ =====
FOLD_ART_PKL = os.environ.get('GRFREE_FOLD_ART')   # (F_k,K_k,POOL_k,GRU重み)×5fold の保存/ロード先
# ★FORCE_ANCHOR(最終錨pklの再生成)とは独立。FOLD_ARTは存在すれば常にロード(FORCE_FOLD_ART=1で強制再学習)
_LOAD_FA = bool(FOLD_ART_PKL) and os.path.exists(FOLD_ART_PKL) and os.environ.get('FORCE_FOLD_ART', '0') != '1'
if _LOAD_FA:
    blob = pickle.loads(open(FOLD_ART_PKL, 'rb').read())
    FOLD_ART = []
    for (F_k, K_k, POOL_k, states) in blob:
        _rebuild_pool_trees(POOL_k)
        nets = []
        for st in states:
            net = Net().to(dev); net.load_state_dict({k: v.to(dev) for k, v in st.items()}); net.eval(); nets.append(net)
        FOLD_ART.append((F_k, K_k, POOL_k, nets))
    print(f'[FOLD_ART] 事前学習済みロード {FOLD_ART_PKL} ({len(FOLD_ART)}fold) = train全ロード+GRU学習スキップ  ({time.time()-t00:.0f}s)', flush=True)
else:
    # ===== train 全ロード =====
    wids = sorted(set(os.path.basename(f).split('__')[0] for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))))
    WELLS = []
    for wi, w in enumerate(wids):
        hw = pd.read_csv(os.path.join(TRAIN_DIR, f'{w}__horizontal_well.csv'), usecols=['X', 'Y', 'Z', 'TVT', 'TVT_input'])
        X = hw['X'].to_numpy(float); Y = hw['Y'].to_numpy(float); Z = hw['Z'].to_numpy(float); tvt = hw['TVT'].to_numpy(float); ti = hw['TVT_input'].to_numpy(float)
        gd = make_gdict(X, Y, Z, ti, tvt)
        if gd is None: continue
        gd['wi'] = wi; gd['wid'] = w; gd['seg'] = segment_well(gd, with_slopes=True)
        gd['_X'] = X; gd['_Y'] = Y; gd['_Z'] = Z; gd['_ti'] = ti; gd['_tvt'] = tvt
        WELLS.append(gd)
    print(f'train wells={len(WELLS)}  ({time.time()-t00:.0f}s)', flush=True)

    # ===== ★完全ランダムshuffle 5分割 → fold毎に[場・κ・プール・GRU] =====
    idx = np.arange(len(WELLS))
    FOLDS = list(KFold(N_FOLDS, shuffle=True, random_state=SPLIT_SEED).split(idx))   # ★完全ランダム
    FOLD_ART = []   # (F_k, K_k, POOL_k, nets)
    oof_e = []; oof_base = []; fold_cv = []
    for fold, (tr_i, va_i) in enumerate(FOLDS):
        tr_wells = [WELLS[i] for i in tr_i]; va_wells = [WELLS[i] for i in va_i]
        # ★train foldの井だけで 場・κ・近傍プール(val完全除外)
        F_k = build_field(tr_wells); K_k = fit_kappa(F_k, tr_wells); POOL_k = build_pool(tr_wells)
        # train seq(own=wi で自井donor除外) → GRU学習items
        items = []
        for gd in tr_wells:
            try:
                X, ei, tg, d7, t0 = wseq(gd, F_k, K_k, POOL_k, own=gd['wi'])
                items.append((X, tg, 0.0)); items.append((X[::-1].copy(), (tg[::-1] - tg[-1]).copy(), 1.0))
            except Exception: pass
        nets = []
        for sd in range(SEEDS):
            torch.manual_seed(sd + 100 + fold); net = Net().to(dev); net = train_loop(net, items, sd + fold * 7); net.eval(); nets.append(net)
        FOLD_ART.append((F_k, K_k, POOL_k, nets))
        # val予測(val井は F_k/POOL_k に不在 → own=-1)
        fe = []; fb = []
        for gd in va_wells:
            try:
                X, ei, tg, d7, t0 = wseq(gd, F_k, K_k, POOL_k, own=-1)
            except Exception: continue
            xb = np.zeros((1, len(ei), C + 1), np.float32); xb[0, :, :C] = X
            with torch.no_grad():
                tb = torch.from_numpy(xb).to(dev); p = np.mean([nt(tb)[0].cpu().numpy() for nt in nets], axis=0)
            tvt = gd['_tvt']; pred = d7[(ei - (t0 + 1))] + p
            fe.append(pred - tvt[ei]); fb.append(d7[(ei - (t0 + 1))] - tvt[ei])
        fe = np.concatenate(fe); fb = np.concatenate(fb); oof_e.append(fe); oof_base.append(fb)
        cvf = float(np.sqrt(np.mean(fe ** 2))); fold_cv.append(cvf)
        print(f'  fold{fold}: val井={len(va_wells)}  val CV(GRU)={cvf:.4f}  (engine単体 {float(np.sqrt(np.mean(fb**2))):.4f})  ({time.time()-t00:.0f}s)', flush=True)
    oof_e = np.concatenate(oof_e); oof_base = np.concatenate(oof_base)
    CV = float(np.sqrt(np.mean(oof_e ** 2))); CVeng = float(np.sqrt(np.mean(oof_base ** 2)))
    print(f'\n★★ OOF CV(GRフリー GRU, val完全除外) = {CV:.4f}  (engine単体 {CVeng:.4f})', flush=True)
    print(f'   fold別 val CV = {[round(c,3) for c in fold_cv]}', flush=True)
    # ===== FOLD_ART 保存(GRU重み込み) → 次回以降はロードで学習スキップ =====
    if FOLD_ART_PKL:
        blob = []
        for (F_k, K_k, POOL_k, nets) in FOLD_ART:
            P2 = dict(POOL_k); P2['trees'] = None      # cKDTreeは保存せずロード時にyxから再構築
            states = [{k: v.detach().cpu() for k, v in nt.state_dict().items()} for nt in nets]
            blob.append((F_k, K_k, P2, states))
        with open(FOLD_ART_PKL, 'wb') as fh: pickle.dump(blob, fh, protocol=4)
        print(f'[FOLD_ART] 保存 {FOLD_ART_PKL} ({len(blob)}fold, GRU重み込み)', flush=True)

# ===== test/対象SPLIT のアンカー生成(tvt=15本平均, sig=15本std) =====
import glob as _glob
SPLIT=os.environ.get("GRF_SPLIT","test")
DIR = TEST_DIR if SPLIT=="test" else TRAIN_DIR
OUTPKL=os.environ.get("GRF_ANCHOR_OUT") or os.path.join(os.environ.get("ROGII_ART95", "."), f"grfree_anchor_{SPLIT}.pkl")
tids=sorted(set(os.path.basename(f).split("__")[0] for f in _glob.glob(os.path.join(DIR,"*__horizontal_well.csv"))))
ANCH={}
for w in tids:
    hw=pd.read_csv(os.path.join(DIR,f"{w}__horizontal_well.csv"),usecols=["X","Y","Z","TVT_input"])
    X_=hw["X"].to_numpy(float);Y_=hw["Y"].to_numpy(float);Z_=hw["Z"].to_numpy(float);ti=hw["TVT_input"].to_numpy(float);n=len(X_)
    finr=np.where(np.isfinite(ti))[0]
    if len(finr)==0: continue
    gd=make_gdict(X_,Y_,Z_,ti)
    if gd is None: continue
    gd["seg"]=segment_well(gd,with_slopes=False);gd["wid"]=w;t0w=gd["s"];tvt_last=float(ti[t0w]);md=np.arange(n,dtype=float)
    seedpreds=[]
    for (F_k,K_k,POOL_k,nets) in FOLD_ART:
        d7_full,rate,ddr,spr=engine_predict(F_k,gd,K_k,own=-1)
        rates_full=np.zeros(n,np.float32);rates_full[t0w+1:]=rate
        conf_full=np.zeros(n,np.float32);conf_full[t0w+1:]=np.exp(-np.maximum(ddr-800.0,0.0)/1000.0)*np.exp(-np.nan_to_num(spr,nan=0.1)/0.06)
        Xc,ei,_=build_channels(md,X_,Y_,Z_,ti,t0w,(d7_full-tvt_last),rates_full,conf_full,POOL_k,self_wid=None)
        xb=np.zeros((1,len(ei),C+1),np.float32);xb[0,:,:C]=Xc
        with torch.no_grad():
            tb=torch.from_numpy(xb).to(dev)
            for nt in nets:
                pp=nt(tb)[0].cpu().numpy()
                seedpreds.append(np.interp(np.arange(t0w+1,n),ei,d7_full[(ei-(t0w+1))]+pp))  # 全eval行TVT予測(1net)
    if not seedpreds: continue
    S=np.stack(seedpreds,0)                                # (15, eval行)
    ANCH[w]=dict(tvt=S.mean(0).astype(np.float32), sig=np.clip(S.std(0),0.5,60.0).astype(np.float32))
pickle.dump(ANCH,open(OUTPKL,"wb"))
print(f"saved {OUTPKL}  {len(ANCH)}井 (GRフリー {SPLIT} アンカー tvt/sig)",flush=True)
sys.stdout.flush()
os._exit(0)   # ★torch/CUDA 終了時クラッシュ(Windows STATUS_STACK_BUFFER_OVERRUN)回避=保存後に即クリーン終了


In [ ]:
%%writefile gen_grfree_test_anchor_v95.py
# -*- coding: utf-8 -*-
"""v95: GRフリー TEST 錨の生成ラッパ(v93 版の v95 パス対応コピー)。

v92/gen_grfree_anchor.py を GRF_SPLIT=test で実行(TRAINで場/κ/GRUを構築しTEST井を予測)し、
その出力錨を読み込んで sig を定数 9.0 に上書きして保存する。v95 の PF パラメータ/錨は v93 と同一。

env:
  GRFREE_GEN     : gen_grfree_anchor.py のパス(既定 v92/gen_grfree_anchor.py。Kaggle nb では cwd に writefile)
  ROGII_ART95    : 出力先ディレクトリ(既定 v95/artifacts_v95)。grfree_anchor_test.pkl をここに保存
出力: <ROGII_ART95>/grfree_anchor_test.pkl = {wid: {tvt(eval行,), sig(eval行,)=9.0}}
"""
import os, sys, io, pickle, subprocess
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
ART = Path(os.environ.get("ROGII_ART95", str(PROJ / "v95" / "artifacts_v95")))
GEN = Path(os.environ.get("GRFREE_GEN", str(PROJ / "v92" / "gen_grfree_anchor.py")))
RAW_OUT = ART / "grfree_anchor_test_raw.pkl"          # gen の生出力(sig=3-seed-std)
FINAL_OUT = ART / "grfree_anchor_test.pkl"            # sig=9.0 に修正後
SIG_CONST = 9.0


def main():
    py = sys.executable
    env = dict(os.environ)
    env["GRF_SPLIT"] = "test"
    env["GRF_ANCHOR_OUT"] = str(RAW_OUT)               # ★必須(未設定だと gen が未定義 P を参照して落ちる)
    ART.mkdir(parents=True, exist_ok=True)
    print(f"[step] run {GEN.name} (GRF_SPLIT=test) -> {RAW_OUT.name}", flush=True)
    r = subprocess.run([py, str(GEN)], env=env)
    if r.returncode != 0:
        raise SystemExit(f"gen_grfree_anchor.py が異常終了 (code={r.returncode})")

    anch = pickle.loads(RAW_OUT.read_bytes())
    print(f"[load] raw test 錨 {len(anch)}井 (sig=3-seed-std を 9.0 へ上書き)", flush=True)
    fixed = {}
    for wid, d in anch.items():
        tvt = np.asarray(d["tvt"], np.float32)
        fixed[wid] = dict(tvt=tvt, sig=np.full(len(tvt), SIG_CONST, np.float32))   # ★sig=9.0 定数
    FINAL_OUT.write_bytes(pickle.dumps(fixed, protocol=4))
    sig_vals = np.unique(np.concatenate([f["sig"] for f in fixed.values()])) if fixed else np.array([])
    print(f"saved: {FINAL_OUT.name}  {len(fixed)}井  sig unique={sig_vals.tolist()}", flush=True)
    return FINAL_OUT


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gen_grfree_test_anchor_v97.py
# -*- coding: utf-8 -*-
"""v95: GRフリー TEST 錨の生成ラッパ(v93 版の v95 パス対応コピー)。

v92/gen_grfree_anchor.py を GRF_SPLIT=test で実行(TRAINで場/κ/GRUを構築しTEST井を予測)し、
その出力錨を読み込んで sig を定数 9.0 に上書きして保存する。v95 の PF パラメータ/錨は v93 と同一。

env:
  GRFREE_GEN     : gen_grfree_anchor.py のパス(既定 v92/gen_grfree_anchor.py。Kaggle nb では cwd に writefile)
  ROGII_ART95    : 出力先ディレクトリ(既定 v95/artifacts_v95)。grfree_anchor_test.pkl をここに保存
出力: <ROGII_ART95>/grfree_anchor_test.pkl = {wid: {tvt(eval行,), sig(eval行,)=9.0}}
"""
import os, sys, io, pickle, subprocess
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
ART = Path(os.environ.get("ROGII_ART95", str(PROJ / "v95" / "artifacts_v95")))
GEN = Path(os.environ.get("GRFREE_GEN", str(PROJ / "v92" / "gen_grfree_anchor.py")))
RAW_OUT = ART / "grfree_anchor_test_raw.pkl"          # gen の生出力(sig=3-seed-std)
FINAL_OUT = ART / "grfree_anchor_test.pkl"            # sig=9.0 に修正後
SIG_CONST = 9.0


def main():
    py = sys.executable
    env = dict(os.environ)
    env["GRF_SPLIT"] = "test"
    env["GRF_ANCHOR_OUT"] = str(RAW_OUT)               # ★必須(未設定だと gen が未定義 P を参照して落ちる)
    ART.mkdir(parents=True, exist_ok=True)
    print(f"[step] run {GEN.name} (GRF_SPLIT=test) -> {RAW_OUT.name}", flush=True)
    r = subprocess.run([py, str(GEN)], env=env)
    if r.returncode != 0:
        raise SystemExit(f"gen_grfree_anchor.py が異常終了 (code={r.returncode})")

    anch = pickle.loads(RAW_OUT.read_bytes())
    print(f"[load] raw test 錨 {len(anch)}井 (sig=3-seed-std を 9.0 へ上書き)", flush=True)
    fixed = {}
    for wid, d in anch.items():
        tvt = np.asarray(d["tvt"], np.float32)
        fixed[wid] = dict(tvt=tvt, sig=np.full(len(tvt), SIG_CONST, np.float32))   # ★sig=9.0 定数
    FINAL_OUT.write_bytes(pickle.dumps(fixed, protocol=4))
    sig_vals = np.unique(np.concatenate([f["sig"] for f in fixed.values()])) if fixed else np.array([])
    print(f"saved: {FINAL_OUT.name}  {len(fixed)}井  sig unique={sig_vals.tolist()}", flush=True)
    return FINAL_OUT


if __name__ == "__main__":
    main()


In [ ]:
%%writefile nn_emission_v97.py
# -*- coding: utf-8 -*-
"""v93 module2: NN-emission sim 生成(★帯中心=GRフリー錨。過去struct不使用)。

v50/v51 の学習emission(TCN cosine類似度)を、候補帯の中心だけ struct→GRフリー錨tvt に差替。
  - encoder(stageA_enccapaug_f{f}.pt)は凍結再利用(v93/artifacts_v93/nn50 に複製済)。
  - OOF fold: sorted train井の strided 5-fold(wells[f::5])。fold f の井は fold f を hold-out した
    encoder f を使う=OOF-safe(v50/v51と同一分割)。Fh/Ft は encoder が期待する入力なので
    v50 cacheA と数値一致を assert してから使用。
  - 出力: v93/artifacts_v93/sim_grfree_v97.pkl = {wid: {sim(T,181)fp16, st(T,)=GRフリー錨tvt}}。
create 側は各PF入力dictに _sim/_st を添付して smoother に渡す(module1 _pad が消費)。
"""
import os, sys, io, glob, pickle
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJ = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
DATA_DIR = PROJ / "rogii-wellbore-geology-prediction"
ART = Path(os.environ.get("ROGII_ART97", str(PROJ / "v97" / "artifacts_v97")))
NN50 = ART / "nn50"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SIM_STEP = 0.5
NBAND = 90                        # ±45ft / 0.5ft

# ---- split 対応(env NN_SPLIT) ----
#   train: 錨=grfree_anchor_train.pkl / OOF fold(wells[f::5]) の encoder f を使用。
#   test : 錨=env V93_ANCHOR_PKL(=GRフリー test 錨, module1 と同一 pkl)。
#          test 井はどの学習fold にも入っていないため encoder は 5本を平均する(OOF ではなく全平均)。
NN_SPLIT = os.environ.get("NN_SPLIT", "train")
if NN_SPLIT == "test":
    _ANCHOR_PKL = Path(os.environ["V93_ANCHOR_PKL"])   # test 錨(必須)
    SIM_FP = ART / "sim_grfree_test_v97.pkl"
else:
    _ANCHOR_PKL = ART / "grfree_anchor_train.pkl"
    SIM_FP = ART / "sim_grfree_v97.pkl"
_ANCHOR = pickle.loads(_ANCHOR_PKL.read_bytes())        # {wid:{tvt,sig}}


# ---- GR特徴(v50/v52 と同一。encoder入力) ----
def _rz(v):
    med = np.nanmedian(v); iqr = np.nanpercentile(v, 75) - np.nanpercentile(v, 25)
    return np.clip((v - med) / max(iqr / 1.349, 1e-6), -8, 8)


def _rm(v, w):
    return pd.Series(v).rolling(w, center=True, min_periods=1).mean().to_numpy()


def _rs(v, w):
    return pd.Series(v).rolling(w, center=True, min_periods=2).std().fillna(0).to_numpy()


def _detr(v, w=301):
    m = _rm(v, w); sd = _rs(v, w)
    return np.clip((v - m) / np.maximum(sd, 1e-3), -6, 6)


def feats_h(gr, md, z):
    g = pd.Series(gr).interpolate(limit_direction="both").to_numpy()
    g = np.where(np.isfinite(g), g, np.nanmedian(g)); g = np.where(np.isfinite(g), g, 0.0)
    z0 = _rz(g); s5, s21, s81 = _rm(z0, 5), _rm(z0, 21), _rm(z0, 81)
    d5 = np.gradient(s5) * 10.0; v21 = _rs(z0, 21); dt = _detr(z0, 301)
    dmd = np.maximum(np.gradient(md), 1e-3)
    dzdm = np.clip(np.gradient(z) / dmd, -0.2, 0.2) * 5.0
    X = np.stack([z0, s5, s21, s81, d5, v21, dt, dzdm], 1).astype(np.float32)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


def feats_t(gg):
    z0 = _rz(gg); s3, s9, s33 = _rm(z0, 3), _rm(z0, 9), _rm(z0, 33)
    d3 = np.gradient(s3) * 10.0; v9 = _rs(z0, 9); dt = _detr(z0, 301)
    X = np.stack([z0, s3, s9, s33, d3, v9, dt], 1).astype(np.float32)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


# ---- encoder(capaug: ch=128, dils 6段。凍結ckpt に一致) ----
class Enc1D(nn.Module):
    def __init__(self, cin, ch=128, dout=96, dils=(1, 2, 4, 8, 16, 32)):
        super().__init__()
        self.inp = nn.Conv1d(cin, ch, 5, padding=2)
        self.blocks = nn.ModuleList()
        for d in dils:
            self.blocks.append(nn.Sequential(
                nn.Conv1d(ch, ch, 5, padding=2 * d, dilation=d), nn.GELU(), nn.Conv1d(ch, ch, 1)))
        self.norm = nn.ModuleList([nn.GroupNorm(8, ch) for _ in dils])
        self.out = nn.Conv1d(ch, dout, 1)

    def forward(self, x):
        h = self.inp(x.t()[None])
        for blk, nm in zip(self.blocks, self.norm):
            h = nm(h + blk(h))
        return F.normalize(self.out(h)[0].t(), dim=1)


def _load_encoders():
    encs = []
    for f in range(5):
        ck = torch.load(NN50 / f"stageA_enccapaug_f{f}.pt", map_location=DEVICE)
        eA = Enc1D(8).to(DEVICE); eA.load_state_dict(ck["encA"]); eA.eval()
        eB = Enc1D(7).to(DEVICE); eB.load_state_dict(ck["encB"]); eB.eval()
        encs.append((eA, eB))
    return encs


def _well_gg(tw):
    """typewell を 0.5ft TVT格子へ(sim帯の座標系)。返り (gg, gmin) or None。"""
    tt = tw["TVT"].to_numpy(float); tg = tw["GR"].to_numpy(float)
    m = np.isfinite(tt) & np.isfinite(tg); tt, tg = tt[m], tg[m]
    if len(tt) < 8:
        return None
    o = np.argsort(tt); tt, tg = tt[o], tg[o]
    gmin = float(tt.min())
    gg = np.interp(np.arange(gmin, float(tt.max()) + SIM_STEP, SIM_STEP), tt, tg)
    return gg, gmin


def build_sims(verify_cache=True, split=None):
    """split井の sim を GRフリー錨帯中心で生成。
       train: OOF fold=wells[f::5](井index i の fold=i%5 の encoder f)。
       test : どの学習fold にも入らないため encoder 5本を平均(全平均)。錨=_ANCHOR(=test錨)。"""
    split = split or NN_SPLIT
    is_test = (split == "test")
    TR = DATA_DIR / split
    wells = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(TR / "*__horizontal_well.csv"))})
    encs = _load_encoders()
    offs = np.arange(-NBAND, NBAND + 1)
    # 検証用 cacheA(Fh/Ft の厳密一致確認, train のみ)。無ければskip
    cache = None
    if verify_cache and not is_test:
        cp = PROJ / "v50" / "artifacts_v50" / "cacheA_v50.pkl"
        if cp.exists():
            cache = pickle.loads(cp.read_bytes())
    out = {}; nver = 0; maxdev = 0.0
    print(f"[NN-emission] split={split} wells={len(wells)} 錨={_ANCHOR_PKL.name}({len(_ANCHOR)}井) "
          f"encoder={'全5平均' if is_test else 'OOF fold'} -> {SIM_FP.name}", flush=True)
    with torch.no_grad():
        for i, wn in enumerate(wells):
            enc_use = encs if is_test else [encs[i % 5]]   # test=全5, train=fold i%5 のみ
            sp = _ANCHOR.get(wn)
            hw = pd.read_csv(TR / f"{wn}__horizontal_well.csv")
            tw = pd.read_csv(TR / f"{wn}__typewell.csv")
            evm = hw["TVT_input"].isna().to_numpy()
            if sp is None or len(sp["tvt"]) != int(evm.sum()):
                continue                                # 錨欠損/行数不一致 → sim無し(注入OFF)
            gg_gmin = _well_gg(tw)
            if gg_gmin is None:
                continue
            gg, gmin = gg_gmin; K = len(gg)
            Fh = feats_h(hw["GR"].to_numpy(float), hw["MD"].to_numpy(float), hw["Z"].to_numpy(float))
            Ft = feats_t(gg)
            if cache is not None and wn in cache and nver < 30:
                cFh = np.asarray(cache[wn]["Fh"]); cFt = np.asarray(cache[wn]["Ft"])
                if cFh.shape == Fh.shape and cFt.shape == Ft.shape:
                    dev = max(float(np.abs(cFh - Fh).max()), float(np.abs(cFt - Ft).max()))
                    maxdev = max(maxdev, dev); nver += 1
            ev_idx = np.where(evm)[0]
            st = np.asarray(sp["tvt"], np.float64)      # ★帯中心=GRフリー錨tvt(struct不使用)
            ks = np.round((st - gmin) / SIM_STEP).astype(np.int64)
            cand = np.clip(ks[:, None] + offs[None, :], 0, K - 1)
            okc = (ks[:, None] + offs[None, :] >= 0) & (ks[:, None] + offs[None, :] <= K - 1)
            xh = torch.from_numpy(Fh).to(DEVICE); xt = torch.from_numpy(Ft).to(DEVICE)
            ct = torch.tensor(cand, device=DEVICE)
            sims_acc = None
            for (eA, eB) in enc_use:                     # test は 5本の類似度を平均
                a = eA(xh)[ev_idx]; b = eB(xt)
                s_e = (a[:, None, :] * b[ct]).sum(-1)
                sims_acc = s_e if sims_acc is None else (sims_acc + s_e)
            sims = (sims_acc / len(enc_use)).cpu().numpy()
            sims = np.where(okc, sims, -1.0)
            out[wn] = dict(sim=sims.astype(np.float16), st=st.astype(np.float32))
            if (i + 1) % 100 == 0:
                print(f"  sim {i+1}/{len(wells)} (ready {len(out)})", flush=True)
    if nver:
        print(f"[検証] Fh/Ft vs cacheA 最大乖離={maxdev:.2e} ({nver}井) -> 0付近ならencoder入力一致OK")
    SIM_FP.write_bytes(pickle.dumps(out, protocol=4))
    print(f"sim saved: {len(out)}/{len(wells)} wells -> {SIM_FP.name}")
    return out


if __name__ == "__main__":
    build_sims(verify_cache=(NN_SPLIT != "test"))


In [ ]:
%%writefile affine_v102.py
# -*- coding: utf-8 -*-
"""v102 affine 較正(v85/v87 verbatim移植)。事前区間(既知prefix)で well GR ≈ a·typewell + b を fit し、
照合GRを typewell スケールに再較正: gr_aligned = (gr − b)/a。gs も aligned 残差で再計算。
create_v95/v97/dec10 が AFFINE=1 のとき、builder 出力 x に対して apply_affine(x, hw, tw_gr, P) を挟む。"""
import numpy as np

AFF_CLIP = (0.3, 3.0)


def affine_cal(kgr, tw_at_k, min_pts=20):
    """kgr ≈ a·tw_at_k + b の polyfit(v52/v85 と同一)。"""
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1.0, (float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.0)
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1)
    return float(a), float(b)


def apply_affine(pf, x, hw, tw_gr, P):
    """x(builder出力)の gr を affine 較正。pf=pf_banks module(_smooth_radius_values 使用)。x2 を返す。
       kgr/ktvt=既知prefixの平滑GR/TVT_input, tw_at_k=参照grid gg を ktvt で内挿。"""
    kn = hw[hw["TVT_input"].notna()]
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = pf._smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    kgr = gr_sm[kpos].astype(float); ktvt = kn["TVT_input"].to_numpy(float)
    gg = np.asarray(x["gg"], float); tvt_grid = x["gmin"] + x["gst"] * np.arange(len(gg))
    tw_at_k = np.interp(ktvt, tvt_grid, gg)
    a, b = affine_cal(kgr, tw_at_k); a = float(np.clip(a, *AFF_CLIP))
    x2 = dict(x)
    x2["gr"] = (np.asarray(x["gr"], float) - b) / a
    if len(kgr) >= 20:
        resid = (kgr - b) / a - tw_at_k; gs = float(np.nanstd(resid))
        if not np.isfinite(gs) or gs <= 0:
            gs = float(P["gr_sig_def"])
    else:
        gs = float(P["gr_sig_def"])
    x2["gs"] = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    return x2


In [ ]:
%%writefile pf_banks_v95.py
# -*- coding: utf-8 -*-
"""v93 module1: 6バンク平滑PFエンジン(GPU固定ラグ smoother)。

v52 create の smooth-PF 部分をクリーン再構築。★v93 の唯一の実変更:
  - pfA バンクの錨を「過去struct(v38/v66)」から「GRフリー錨(v91 OOF, sig=9ft較正)」へ差替。
  - 錨強度 anchor_mult は global 定数でなく バンクparam P["anchor_mult"] から取る(pfA=20)。
過去struct は一切読まない([[no-struct-directive]])。錨源は v93/artifacts_v93 のみ。

構成(v52 と数値一致させる要素):
  - build_smoother_inputs / build_inputs_self / build_inputs_nbr : PF入力(tw/self/nbr疑似typewell)
  - attach_anchor : GRフリー錨(anc/ancs)を添付。非physicsバンクは ancs=1e9(錨OFF)
  - _smoother_core : 固定ラグ(L=smooth_lag)平滑PF。GR尤度×phys×錨×NN-emission
  - run_smoother_ext : seed尤度加重で smoothed平均/std を返す
NN-emission(sim/帯中心 st)は create 側が入力dictに _sim/_st を添付する形で受ける(module2が生成)。
"""
import os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
DATA_DIR = Path(os.environ.get("ROGII_DATA", str(PROJ / "rogii-wellbore-geology-prediction")))
ART = Path(os.environ.get("ROGII_ART95", str(PROJ / "v95" / "artifacts_v95")))  # ★v95: config/錨をv93からコピー済(PF param同一)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

# ---- config(6バンクparam) & GRフリー錨 ----
_CFG = json.loads((ART / "pf_banks_config.json").read_text(encoding="utf-8"))
BANK_ORDER = _CFG["bank_order"]                    # [pf_1,pf_2,pf_3,r0_seed32,r1_seed32,pfA]
PHYSICS_BANKS = set(_CFG["physics_banks"])         # {pfA}
BANK_PARAMS = _CFG["params"]                       # dict[bank] -> raw param dict
W_NN_BANK = _CFG["w_nn_bank"]                       # {pfA:0.01}
W_NN_DEFAULT = float(_CFG["w_nn_default"])          # 0.02
SMOOTH_LAG = int(_CFG["smooth_lag"])               # 16
SMOOTH_MODE_CFG = str(_CFG.get("smooth_mode", "fixedlag"))   # ★v99: full平滑切替
N_SEED = int(_CFG["n_seed"])                        # 32
# ★v102: ps_combo(seed集約を尤度×錨カーネルに=選択強化)。env で全体上書き可、off_banksは無効。
PS_COMBO_TAU = float(os.environ.get("PS_COMBO_TAU", _CFG.get("ps_combo_tau", 0.0)))
PS_COMBO_OFF = set(_CFG.get("ps_combo_off_banks", []))

_ANCHOR_PKL = ART / _CFG.get("anchor_pkl", "grfree_anchor_train.pkl")
if os.environ.get("V93_ANCHOR_PKL"):
    _ANCHOR_PKL = Path(os.environ["V93_ANCHOR_PKL"])
_ANCHOR = pickle.loads(_ANCHOR_PKL.read_bytes())   # {wid: {tvt, sig}}
print(f"[v93 pf_banks] 錨={_ANCHOR_PKL.name} {len(_ANCHOR)}井 / banks={BANK_ORDER} / smooth_lag={SMOOTH_LAG}")

# ---- NN-emission 帯(create/module2と共有する幾何定数) ----
SIM_STEP_NN = 0.5
NBAND_NN = 90                                       # ±45ft / 0.5ft = 90

# ---- self/nbr/Z傾斜 定数(v52同値) ----
SELF_MIX_W = 1.0
NBR_MIX_W = 1.0
ZGRAD_R = 25
K_MAX = 3
MAX_DIST = 1500.0
NEED_REFS = 2


def bank_param(bank):
    """バンク生paramに physics/w_nn を付けて返す(create が smoother に渡す P)。"""
    P = dict(BANK_PARAMS[bank])
    P["name"] = bank
    P.setdefault("smooth_lag", SMOOTH_LAG)
    P.setdefault("smooth_mode", SMOOTH_MODE_CFG)
    P["_physics"] = bank in PHYSICS_BANKS
    P["_w_nn"] = float(W_NN_BANK.get(bank, W_NN_DEFAULT))
    P["_ps_combo_tau"] = 0.0 if bank in PS_COMBO_OFF else PS_COMBO_TAU   # ★v102 選択強化(off_banksは無効)
    return P


def _ps_combo_reweight(ww, ps_jT, st, tau):
    """★v102 ps_combo: seed尤度重み ww を「錨に近いseedを重く」再加重(選択強化)。st無/tau0はそのまま。
       ps_jT=(S,T) その井の per-seed平滑軌跡, st=GRフリー錨tvt(=_st, emission帯中心)。"""
    if tau <= 0 or st is None:
        return ww
    T = ps_jT.shape[1]; stj = np.asarray(st, float)
    if len(stj) < T:
        return ww
    da = ((ps_jT - stj[:T][None, :]) ** 2).mean(1)                      # 各seedの錨距離²
    w2 = ww * np.exp(-(da - da.min()) / (2.0 * tau * tau)); s = w2.sum()
    return w2 / s if s > 1e-300 else np.full(len(ww), 1.0 / len(ww))


# ==================== 小物(v52 と同一) ====================
def _smooth_radius_values(vals, fb, r):
    r = int(r)
    s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    if r <= 0:
        return s.to_numpy(np.float32)
    return s.rolling(2 * r + 1, center=True, min_periods=1).mean().to_numpy(np.float32)


def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)


def _bin_grid(t, g, gmin, step, G):
    """(TVT, GR) を grid の TVT ビンに median 集約(疑似typewell)。無データビン=NaN。"""
    m = np.isfinite(t) & np.isfinite(g); t, g = t[m], g[m]
    idx = np.round((t - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, gg = idx[ok], g[ok]
    out = np.full(G, np.nan)
    if len(gg):
        s = pd.Series(gg).groupby(idx).median(); out[s.index.to_numpy()] = s.to_numpy()
    return out


# ==================== PF入力(tw / self / nbr) ====================
def build_smoother_inputs(hw, tw_tvt, tw_gr, P):
    """smooth PF 入力を hw/tw と生PFパラメータから作る。eval無しは None。"""
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    tw_gr_pf = _smooth_radius_values(tw_gr, tw_fb, P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(tw_tvt, tw_gr_pf)
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    ev_pos = hw.index.get_indexer(ev.index); kpos = hw.index.get_indexer(kn.index)
    kn_tvtin = kn["TVT_input"].to_numpy(float)
    if len(kpos) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[kpos] - np.interp(kn_tvtin, tw_tvt, tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    gs = gs * float(os.environ.get("GS_SCALE", "1.0"))     # ★post-clip の GR尤度σ 広げ(gs×1.30 検証。既定1.0=無効)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    return dict(md=ev["MD"].to_numpy(float), z=ev["Z"].to_numpy(float), gr=gr_sm[ev_pos].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=ls, ir=ir, N=int(P["n_particles"]))


def build_inputs_self(hw, tw_tvt, tw_gr, P):
    """tw入力の gg を、自分の既知prefix GR(TVT_input) 疑似typewell で上書き。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None:
        return None
    kn = hw[hw["TVT_input"].notna()]
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    gh = _bin_grid(kn["TVT_input"].to_numpy(float), gr_sm[kpos].astype(float), base["gmin"], base["gst"], len(base["gg"]))
    gg = base["gg"].copy(); have = np.isfinite(gh)
    gg[have] = SELF_MIX_W * gh[have] + (1 - SELF_MIX_W) * gg[have]
    base = dict(base); base["gg"] = gg
    return base


def build_inputs_nbr(hw, tw_tvt, tw_gr, P, refs):
    """tw入力の gg を、近傍train坑井の GR(TVT) 疑似typewell で上書き。refs無し=tw fallback。
       refs = [(gr_array, tvt_array), ...](module3 の近傍選択が供給)。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None:
        return None
    if refs:
        rt = np.concatenate([r[1] for r in refs])
        rg = np.concatenate([_smooth_radius_values(r[0], float(np.nanmean(r[0])), P["hgr_smooth_r"]).astype(float) for r in refs])
        gh = _bin_grid(rt, rg, base["gmin"], base["gst"], len(base["gg"]))
        gg = base["gg"].copy(); have = np.isfinite(gh)
        gg[have] = NBR_MIX_W * gh[have] + (1 - NBR_MIX_W) * gg[have]
        base = dict(base); base["gg"] = gg
    return base


def build_inputs_self_graft(hw, tw_tvt, tw_gr, P):
    """★v95新規: self(自分のprefix GR)を tw格子に substitute し、prefix範囲外(データ無しビン)を
       tw を self較正(self≈a·tw+b, a∈[0.2,5])で外挿。self被覆を広げる(v49 build_inputs_self_ext 由来)。
       較正係数 a,b・有効フラグ・外挿割合を base['_graft'] にメタ格納(provenance用)。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None:
        return None
    kn = hw[hw["TVT_input"].notna()]
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    gh = _bin_grid(kn["TVT_input"].to_numpy(float), gr_sm[kpos].astype(float), base["gmin"], base["gst"], len(base["gg"]))
    gg = base["gg"].copy(); have = np.isfinite(gh)
    a, b, valid = 1.0, 0.0, 0
    if have.sum() >= 8:
        gg[have] = gh[have]                                       # prefix範囲=self実測
        tw_ov = base["gg"][have]; self_ov = gh[have]
        A = np.c_[tw_ov, np.ones(len(tw_ov))]
        try:
            coef, *_ = np.linalg.lstsq(A, self_ov, rcond=None); a, b = float(coef[0]), float(coef[1])
        except Exception:
            a, b = 1.0, 0.0
        if 0.2 <= a <= 5.0 and np.isfinite(a) and np.isfinite(b):
            gg[~have] = a * base["gg"][~have] + b                 # 範囲外=self較正したtwで外挿
            valid = 1
    base = dict(base); base["gg"] = gg
    base["_graft"] = dict(a=float(a), b=float(b), valid=int(valid),
                          cover=float(have.mean()), extrap_frac=float((~have).mean()), nprefix=int(len(kpos)))
    return base


def z_gradient_eval(hw, r=ZGRAD_R):
    """Z傾斜 dZ/dMD を移動平均(断層影響回避)し eval 行のみ返す。"""
    ev = hw["TVT_input"].isna().to_numpy()
    md = hw["MD"].to_numpy(float); z = hw["Z"].to_numpy(float)
    with np.errstate(all="ignore"):
        g = np.gradient(z, md)
    g = pd.Series(g).rolling(2 * int(r) + 1, center=True, min_periods=1).mean().to_numpy()
    return g[ev].astype(np.float32)


# ==================== GRフリー錨 添付 ====================
def attach_anchor(x, wid, physics):
    """PF入力に GRフリー錨(anc/ancs)を添付。physics=False / 錨欠損 / 行数不一致 は錨OFF(ancs=1e9)。
       ★過去struct は使わない。錨源は _ANCHOR(v93 GRフリー, sig=9ft較正)のみ。"""
    n = len(x["md"]); sp = _ANCHOR.get(wid)
    x["_wid"] = wid
    if physics and sp is not None and len(sp["tvt"]) == n:
        x["anc"] = np.asarray(sp["tvt"], float); x["ancs"] = np.asarray(sp["sig"], float)
    else:
        x["anc"] = np.zeros(n); x["ancs"] = np.full(n, 1e9)
    return x


# ==================== GPU 固定ラグ 平滑PF 本体 ====================
def _smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir,
                   anc, ancs, amul, stq, sim, wmap, w_nn, P, N, device, gen, dtype=DTYPE):
    """v52 _v26_smoother_core と数値一致。★v93変更: 錨強度は amul(=P['anchor_mult'])。"""
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    NU = float(P.get("robust_nu", 0.0))                # ★v98: Student-t頑健尤度(0=Gaussian)。v97同様のPF強化をv95側へ移植
    BETA = float(P.get("temper_beta", 1.0))            # ★v98: tempering(1=無効, <1=軟化)

    def rn():
        return torch.randn((B, N), generator=gen, device=device, dtype=dtype)

    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + RN * rn()
        if RATE_CLIP > 0:
            rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        if NU > 0.0:                                   # ★v98: Student-t 頑健尤度(重い裾)
            lk = (1.0 + dp / NU) ** (-(NU + 1.0) * 0.5)
        else:
            lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero)
        if BETA != 1.0:                                # ★v98: tempering(尤度軟化)
            lk = lk ** BETA
        lk = lk.clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate - dipp) / PHYS_SIG; lkp = torch.exp(-0.5 * dphys * dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        # ★v93: GRフリー行錨(mult = amul = P['anchor_mult'])。非physicsは ancs=1e9 で無効化。
        asig = (ancs[:, i][:, None] * amul).clamp_min(1e-6)
        da = (v - anc[:, i][:, None]) / asig
        lka = torch.where(da * da < 1200.0, torch.exp(-0.5 * da * da), zero).clamp_min(1e-300)
        # NN-emission(学習類似度を温度 w_nn で乗算)。帯中心 stq は GRフリー錨tvt(module2)。
        if w_nn > 0:
            jj = torch.round((v - stq[:, i][:, None]) / SIM_STEP_NN).long() + NBAND_NN
            inb = (jj >= 0) & (jj <= 2 * NBAND_NN)
            sim_i = sim[:, i, :].to(dtype)[wmap]
            sv = torch.gather(sim_i, 1, jj.clamp(0, 2 * NBAND_NN))
            sv = torch.where(inb, sv, torch.full_like(sv, -1.0))
            lka = lka * torch.exp(w_nn * sv)
        w_new = torch.where(obs[:, None], w * lk * lkp * lka, w * lkp * lka)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0:
                rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w * (pos - zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i - L] = (w * old).sum(1) - z[:, i - L]
            buf[:, ptr, :] = pos; ptr = (ptr + 1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T - L), T):
            pts_s[:, j] = (w * buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik


def _smoother_core_full(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir,
                        anc, ancs, amul, stq, sim, wmap, w_nn, P, N, device, gen, dtype=DTYPE):
    """★v99 full平滑: forward は _smoother_core と数値一致。固定ラグbufの代わりに全履歴(pos fp32=錨delta / anc int16)を
       保存し、単一backward祖先sweep(最終重み)で全区間平滑 pts_s を返す。robust/temper/phys/anchor/NN-emission 全対応。"""
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    NU = float(P.get("robust_nu", 0.0)); BETA = float(P.get("temper_beta", 1.0))

    def rn():
        return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pos_hist = torch.empty((T, B, N), device=device, dtype=dtype)          # fp32(錨lsからのdelta)
    anc_hist = torch.empty((T, B, N), device=device, dtype=torch.int16)     # 祖先index
    lsr = ls[:, None]
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    arL = torch.arange(N, device=device, dtype=torch.long)[None, :].expand(B, N)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + RN * rn()
        if RATE_CLIP > 0:
            rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        if NU > 0.0:
            lk = (1.0 + dp / NU) ** (-(NU + 1.0) * 0.5)
        else:
            lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero)
        if BETA != 1.0:
            lk = lk ** BETA
        lk = lk.clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate - dipp) / PHYS_SIG; lkp = torch.exp(-0.5 * dphys * dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        asig = (ancs[:, i][:, None] * amul).clamp_min(1e-6)
        da = (v - anc[:, i][:, None]) / asig
        lka = torch.where(da * da < 1200.0, torch.exp(-0.5 * da * da), zero).clamp_min(1e-300)
        if w_nn > 0:
            jj = torch.round((v - stq[:, i][:, None]) / SIM_STEP_NN).long() + NBAND_NN
            inb = (jj >= 0) & (jj <= 2 * NBAND_NN)
            sim_i = sim[:, i, :].to(dtype)[wmap]
            sv = torch.gather(sim_i, 1, jj.clamp(0, 2 * NBAND_NN))
            sv = torch.where(inb, sv, torch.full_like(sv, -1.0))
            lka = lka * torch.exp(w_nn * sv)
        w_new = torch.where(obs[:, None], w * lk * lkp * lka, w * lkp * lka)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        anc_step = arL
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0:
                rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
            anc_step = torch.where(nm, idx, arL)
        pts_f[:, i] = (w * (pos - zi)).sum(1)
        pos_hist[i] = pos - lsr; anc_hist[i] = anc_step.to(torch.int16)
        pm = torch.where(act, cur, pm)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype); a = arL.clone(); wfin = w
    for i in range(T - 1, -1, -1):
        pts_s[:, i] = (wfin * (torch.gather(pos_hist[i], 1, a).to(dtype) + lsr)).sum(1) - z[:, i]
        a = torch.gather(anc_hist[i], 1, a).long()
    del pos_hist, anc_hist
    return pts_f, pts_s, log_lik


def _pad(inps, device, dtype=DTYPE):
    """可変長wellをバッチテンソルへ。anc/ancs/stq/sim(NN)も詰める。sim帯は入力dictの _sim/_st から。"""
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    anc = torch.zeros((W, Tmax), dtype=dtype); ancs = torch.full((W, Tmax), 1e9, dtype=dtype)
    stq = torch.zeros((W, Tmax), dtype=dtype)
    simt = torch.full((W, Tmax, 2 * NBAND_NN + 1), -1.0, dtype=torch.float16)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        _sim = x.get("_sim"); _st = x.get("_st")
        if _sim is not None and _st is not None:
            _Ts = min(Tn, len(_st))
            stq[b, :_Ts] = torch.from_numpy(np.asarray(_st[:_Ts], np.float32))
            if _Ts < Tmax:
                stq[b, _Ts:] = stq[b, _Ts - 1]
            simt[b, :_Ts] = torch.from_numpy(np.asarray(_sim[:_Ts], np.float16))
        if "anc" in x:
            anc[b, :Tn] = torch.from_numpy(x["anc"].astype("float32"))
            if Tn < Tmax:
                anc[b, Tn:] = anc[b, Tn - 1]
            ancs[b, :Tn] = torch.from_numpy(x["ancs"].astype("float32"))
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax:
            md[b, Tn:] = md[b, Tn - 1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax:
            grid[b, G:] = grid[b, G - 1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir), anc=to(anc), ancs=to(ancs),
                stq=to(stq), simt=to(simt))


def _pf_devices(chunk_env_default):
    """PF実行デバイス一覧を決定。
       PF_SIM_NGPU=N(>=2): 1GPU上でN論理分割=分割/結合/スレッド機構の検証用(全て同一物理GPU)。
       PF_NGPU=N(>=2): 実GPUをN枚使用(Kaggle 2×T4)。既定=[DEVICE]で現行完全不変。"""
    sim_ng = int(os.environ.get("PF_SIM_NGPU", "0"))
    if sim_ng >= 2:
        return [DEVICE] * sim_ng, True
    ng = int(os.environ.get("PF_NGPU", "1"))
    if ng >= 2 and torch.cuda.is_available() and torch.cuda.device_count() >= 2:
        return ["cuda:%d" % i for i in range(min(ng, torch.cuda.device_count()))], False
    return [DEVICE], False


def run_smoother_ext(inps, P, seed, n_seeds, chunk, w_nn=0.0, capture=False):
    """各 well: smoothed(seed尤度加重平均)と std(seed間加重std)を返す。
       ★v93: 錨強度 amul は P['anchor_mult'](pfA=20)。非physicsバンクは ancs=1e9 で錨無効。
       ★capture=True: v34モードゲート用に per-seed 平滑ps(S,T)/forward pfw/ll/ww も out に付与(RAM増)。
       ★マルチGPU: whole-chunkを各デバイスへラウンドロビン分配しスレッド並列。seedはchunk毎リセット・
         torch CUDA RNGはデバイス非依存なので、同一chunk境界なら単一GPUと(浮動小数の非決定性を除き)一致。"""
    import threading
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); out = [None] * W
    ls_scale = float(P["likelihood_scale"]); amul = float(P.get("anchor_mult", 1.0))
    ch_env = os.environ.get("PF_WELL_CHUNK")
    if ch_env:
        chunk = int(ch_env)

    # ★v99 full平滑モード: 固定ラグの代わりに全区間系譜平滑。可変chunk(VRAM予算)・長さソート。
    SMOOTH_MODE = str(P.get("smooth_mode", os.environ.get("SMOOTH_MODE", "fixedlag")))
    if SMOOTH_MODE == "full":
        budget = float(os.environ.get("FULL_VRAM_GB", "8.0")) * 1e9
        order = sorted(range(W), key=lambda i: len(inps[i]["md"]))
        chunks = []; p = 0                                       # ★可変chunkを全部先に構築(デバイス分配用)
        while p < W:
            Tmax = len(inps[order[p]]["md"]); nw = 0
            while p + nw < W and nw < 64:
                tmx = max(Tmax, len(inps[order[p + nw]]["md"]))
                if (nw + 1) * S * tmx * N * 6 > budget and nw >= 1:
                    break
                Tmax = tmx; nw += 1
            chunks.append(order[p:p + nw]); p += nw

        def _proc_full(sel, device):                            # ★1可変chunkを指定deviceで処理(seedはchunk毎リセット=device非依存)
            sub = [inps[j] for j in sel]; nw = len(sel); pad = _pad(sub, device)
            rep = lambda tt: tt.repeat_interleave(S, 0)
            gen = torch.Generator(device=device); gen.manual_seed(seed)
            _wmap = torch.arange(nw, device=device).repeat_interleave(S)
            pf, ps, ll = _smoother_core_full(
                rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
                pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
                pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
                rep(pad["anc"]), rep(pad["ancs"]), amul, rep(pad["stq"]), pad["simt"], _wmap, w_nn, P, N, device, gen)
            ps = ps.view(nw, S, -1).double().cpu().numpy(); pf = pf.view(nw, S, -1).double().cpu().numpy(); ll = ll.view(nw, S).cpu().numpy()
            for j in range(nw):
                gi = sel[j]; T = len(sub[j]["md"]); llw = ll[j]
                if np.isfinite(llw).any():
                    mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw - mx, -np.inf)
                    wwv = np.exp(lk / max(ls_scale, 1e-6)); s = wwv.sum(); wwv = wwv / s if s > 1e-300 else np.full(S, 1.0 / S)
                else:
                    wwv = np.full(S, 1.0 / S)
                wwv = _ps_combo_reweight(wwv, ps[j, :, :T], sub[j].get("_st"), float(P.get("_ps_combo_tau", 0.0)))
                smean = (wwv[:, None] * ps[j, :, :T]).sum(0)
                sstd = np.sqrt(np.maximum((wwv[:, None] * (ps[j, :, :T] - smean[None, :]) ** 2).sum(0), 0.0))
                nobs = int(np.isfinite(sub[j]["gr"][:T]).sum())
                llbest = float(np.nanmax(llw[np.isfinite(llw)])) if np.isfinite(llw).any() else np.nan
                out[gi] = dict(mean=smean.astype(np.float32), std=sstd.astype(np.float32), loglik=np.float32(llbest / max(nobs, 1)))
                if capture:
                    out[gi].update(ps=ps[j, :, :T].astype(np.float32), pfw=pf[j, :, :T].astype(np.float32),
                                   ll=llw.astype(np.float64), ww=wwv.astype(np.float64))

        fdevices, fsim = _pf_devices(chunk)                     # ★PF_NGPU>=2で実GPU複数(既定=[DEVICE]で単GPU=現行不変)
        if len(fdevices) <= 1:
            for sel in chunks:
                _proc_full(sel, fdevices[0])
        else:
            assign = {i: [] for i in range(len(fdevices))}      # 可変chunkをデバイスへラウンドロビン
            for i, sel in enumerate(chunks):
                assign[i % len(fdevices)].append(sel)
            errs = []
            def _wf(di):
                try:
                    for sel in assign[di]:
                        _proc_full(sel, fdevices[di])
                except Exception as e:
                    errs.append(e)
            ths = [threading.Thread(target=_wf, args=(di,)) for di in range(len(fdevices))]
            for t in ths: t.start()
            for t in ths: t.join()
            if errs:
                raise errs[0]
            print(f"[fullPF-multiGPU] devices={fdevices} sim={fsim} chunks={len(chunks)} wells={W}", flush=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return out

    devices, sim = _pf_devices(chunk)

    def _proc(c0, device):
        sub = inps[c0:c0 + chunk]; w = len(sub); pad = _pad(sub, device)
        rep = lambda tt: tt.repeat_interleave(S, 0)
        gen = torch.Generator(device=device); gen.manual_seed(seed)
        _wmap = torch.arange(w, device=device).repeat_interleave(S)
        pf, ps, ll = _smoother_core(
            rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
            pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
            pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
            rep(pad["anc"]), rep(pad["ancs"]), amul, rep(pad["stq"]), pad["simt"], _wmap, w_nn, P, N, device, gen)
        pf = pf.view(w, S, -1).double().cpu().numpy(); ps = ps.view(w, S, -1).double().cpu().numpy()
        ll = ll.view(w, S).cpu().numpy()
        for j in range(w):
            T = len(sub[j]["md"]); llw = ll[j]
            if np.isfinite(llw).any():
                mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw - mx, -np.inf)
                ww = np.exp(lk / max(ls_scale, 1e-6)); s = ww.sum()
                ww = ww / s if s > 1e-300 else np.full(S, 1.0 / S)
            else:
                ww = np.full(S, 1.0 / S)
            ww = _ps_combo_reweight(ww, ps[j, :, :T], sub[j].get("_st"), float(P.get("_ps_combo_tau", 0.0)))
            smean = (ww[:, None] * ps[j, :, :T]).sum(0)
            sstd = np.sqrt(np.maximum((ww[:, None] * (ps[j, :, :T] - smean[None, :]) ** 2).sum(0), 0.0))
            nobs = int(np.isfinite(sub[j]["gr"][:T]).sum())      # ★GR観測行数(loglik正規化)
            llbest = float(np.nanmax(llw[np.isfinite(llw)])) if np.isfinite(llw).any() else np.nan
            out[c0 + j] = dict(mean=smean.astype(np.float32), std=sstd.astype(np.float32),
                               loglik=np.float32(llbest / max(nobs, 1)))   # per-row平均対数尤度=その表現PFのGR適合
            if capture:                                                    # ★v34ゲート: per-seed 生軌跡
                out[c0 + j].update(ps=ps[j, :, :T].astype(np.float32), pfw=pf[j, :, :T].astype(np.float32),
                                   ll=llw.astype(np.float64), ww=ww.astype(np.float64))

    starts = list(range(0, W, chunk))
    if len(devices) <= 1:
        for c0 in starts:                                        # ★単一GPU経路=現行と完全同一(検証可能)
            _proc(c0, devices[0])
    else:
        assign = {i: [] for i in range(len(devices))}            # chunkをデバイスへラウンドロビン割当
        for i, c0 in enumerate(starts):
            assign[i % len(devices)].append(c0)
        errs = []
        def _worker(di):
            try:
                for c0 in assign[di]:
                    _proc(c0, devices[di])
            except Exception as e:
                errs.append(e)
        ths = [threading.Thread(target=_worker, args=(di,)) for di in range(len(devices))]
        for t in ths: t.start()
        for t in ths: t.join()
        if errs:
            raise errs[0]
        print(f"[PF-multiGPU] devices={devices} sim={sim} chunks={len(starts)} wells={W}", flush=True)
    if torch.cuda.is_available():
        for d in set(devices):
            if str(d).startswith("cuda"):
                torch.cuda.synchronize(d)
    return out


# ==================== 単体検証(python pf_banks_v93.py) ====================
if __name__ == "__main__":
    import glob
    TR = DATA_DIR / "train"
    wids = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(TR / "*__horizontal_well.csv"))})
    wids = wids[:12]
    for bank in ["r0_seed32", "pfA"]:
        P = bank_param(bank)
        inps, trues = [], []
        for w in wids:
            hw = pd.read_csv(TR / f"{w}__horizontal_well.csv")
            tw = pd.read_csv(TR / f"{w}__typewell.csv").sort_values("TVT")
            tt = tw["TVT"].to_numpy(float); tg = tw["GR"].to_numpy(float)
            m = np.isfinite(tt) & np.isfinite(tg)
            if m.sum() < 8:
                continue
            x = build_smoother_inputs(hw, tt[m], tg[m], P)
            if x is None:
                continue
            attach_anchor(x, w, P["_physics"])
            ev = hw[hw["TVT_input"].isna()]
            inps.append(x); trues.append(ev["TVT"].to_numpy(float))
        res = run_smoother_ext(inps, P, seed=99999, n_seeds=16, chunk=8, w_nn=0.0)
        errs = np.concatenate([res[k]["mean"] - trues[k] for k in range(len(inps))])
        rmse = float(np.sqrt(np.mean(errs[np.isfinite(errs)] ** 2)))
        print(f"  [{bank}] {len(inps)}井 smooth-PF(w_nn=0) overall RMSE = {rmse:.3f}"
              f"  (physics={P['_physics']}, anchor_mult={P.get('anchor_mult')})")


In [ ]:
%%writefile imputers_v95.py
# -*- coding: utf-8 -*-
"""v93 module3: 空間imputer(formation-plane / dense-ANCC)と近傍(train)プール選択。

v52 create の空間KNN部分をクリーン再構築。全て train CSV から自己完結で構築(cross-version無し)。
  - FormationPlaneKNN : 各train井のXY重心+層median → 局所平面fit(XY→層深)。self除外(leave-self-out)。
  - DenseANCCImputer  : train井のXY-ANCCを密サンプル → KNN加重。self除外。
  - 近傍プール(neighbors_of): XY重心距離<=MAX_DIST の最近傍<=K_MAX 井(self/nbr疑似typewell用)。
※ 層列(ANCC..BUDA)は train のみ存在するが、imputer は train から prior を作り query井の XY だけで引くため
  test でも安全(test井の層列は不要)。[[kaggle-submit-pitfalls]] の test既知列制約を満たす。
"""
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
DATA_DIR = Path(os.environ.get("ROGII_DATA", str(PROJ / "rogii-wellbore-geology-prediction")))

# ==================== ★train参照構造キャッシュ(全プロセスで1回だけ構築) ====================
# submit は create/build_forward を複数プロセスで起動し、各々が 773 train井から FI/DI/GR-TVT/重心を
# ゼロ再構築していた(setup が数プロセス分 重複=大きな固定コスト)。IMP_CACHE=<pkl> があれば1回構築を全プロセスで共有。
_CACHE = "unset"
def _cache():
    global _CACHE
    if _CACHE == "unset":
        _CACHE = None
        cp = os.environ.get("IMP_CACHE")
        if cp and os.path.exists(cp):
            try:
                import pickle
                _CACHE = pickle.loads(Path(cp).read_bytes())
            except Exception as e:
                print(f"[imp_cache] load 失敗({e})-> 再構築フォールバック", flush=True); _CACHE = None
    return _CACHE

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10
DENSE_SPW = 60
DENSE_K = 20
K_MAX = 3
MAX_DIST = 1500.0
NEED_REFS = 2


class FormationPlaneKNN:
    """XY→層深 の局所平面fit(近傍K井の逆距離加重)。self除外可。"""

    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f"{wid}__horizontal_well.csv"
            try:
                df = pd.read_csv(p, usecols=["X", "Y"] + FORMATIONS).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            row = {"wid": wid, "x": float(df["X"].median()), "y": float(df["Y"].median())}
            for c in FORMATIONS:
                row[f"{c}_m"] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df["wid"])}
        xy = self.df[["x", "y"]].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1.0, xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df["x"].to_numpy(); self.ya = self.df["y"].to_numpy()
        self.fa = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap:
            dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1.0 / (dk + 1e-3), 0.0).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1), (w[:, :, None] * fn).sum(1)], 1)
        try:
            coef = np.linalg.solve(A, rhs)
        except Exception:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try:
                    coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except Exception:
                    pass
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] + Yq[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)


class DenseANCCImputer:
    """train井のXY-ANCCを密サンプルしKNN加重推定。self除外可。"""

    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f"{wid}__horizontal_well.csv"
            try:
                df = pd.read_csv(p, usecols=["X", "Y", "ANCC"]).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s["X"].values); ys.append(s["Y"].values)
            anccs.append(s["ANCC"].values); wids.extend([wid] * len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32); self.wids = np.array(wids)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1.0, self.xy.std(0))
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q / self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid:
            dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1.0 / (dk + 1e-3), 0.0)
        sw = w.sum(1); safe = np.where(sw < 1e-9, 1.0, sw); an = self.ancc[ik]
        ap = (an * w).sum(1) / safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
        return (ap.astype(np.float32), np.sqrt(np.maximum(var, 0.0)).astype(np.float32),
                np.where(vk, dk, np.inf).min(1).astype(np.float32))


# ==================== 近傍(train)プール: XY重心 ====================
_TRAIN_CENT = None
_REFC = {}


def _train_centroids():
    global _TRAIN_CENT
    if _TRAIN_CENT is None:
        c = _cache()
        if c is not None:
            _TRAIN_CENT = c["cent"]; return _TRAIN_CENT
        tdir = DATA_DIR / "train"
        wids = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(tdir / "*__horizontal_well.csv"))})
        xy = []
        for w in wids:
            h = pd.read_csv(tdir / f"{w}__horizontal_well.csv", usecols=["X", "Y"])
            xy.append((float(h["X"].mean()), float(h["Y"].mean())))
        _TRAIN_CENT = (wids, np.array(xy))
    return _TRAIN_CENT


def ref_grtvt(wid):
    """近傍井の (GR, TVT)。疑似typewell の材料。"""
    if wid not in _REFC:
        h = pd.read_csv(DATA_DIR / "train" / f"{wid}__horizontal_well.csv", usecols=["GR", "TVT"])
        _REFC[wid] = (h["GR"].to_numpy(float), h["TVT"].to_numpy(float))
    return _REFC[wid]


def nearest_dist(cx, cy, exclude_wid=None):
    """最近傍train井までの実距離(孤立度)。"""
    wids, xy = _train_centroids(); d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy)
    if exclude_wid is not None:
        keep = np.array([w != exclude_wid for w in wids]); d = d[keep]
    return float(d.min()) if len(d) else float("nan")


def neighbors_of(cx, cy, exclude_wid=None):
    """XY重心距離<=MAX_DIST の最近傍<=K_MAX 井 [(wid, dist), ...]。"""
    wids, xy = _train_centroids(); d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy); out = []
    for j in np.argsort(d):
        w = wids[j]
        if w == exclude_wid:
            continue
        if d[j] > MAX_DIST:
            break
        out.append((w, float(d[j])))
        if len(out) >= K_MAX:
            break
    return out


# ==================== GR類似 近傍(v95新規): 全train井から層序が合う井をtop-K ====================
K_GR = 5                                    # nbr_gr5 の本数
_TRAIN_GRTVT = None                         # {wid: (tv_sorted, gr_sorted)}
_TRAIN_CENTZ = None                         # (wids, xyz)  Z重心込み


def _train_grtvt():
    """全train井の (TVT昇順, GR) をキャッシュ(GR類似の照合材料)。"""
    global _TRAIN_GRTVT
    if _TRAIN_GRTVT is None:
        c = _cache()
        if c is not None:
            _TRAIN_GRTVT = c["grtvt"]; return _TRAIN_GRTVT
        tdir = DATA_DIR / "train"; d = {}
        for q in glob.glob(str(tdir / "*__horizontal_well.csv")):
            w = os.path.basename(q).split("__")[0]
            h = pd.read_csv(q, usecols=["GR", "TVT"]); tv = h["TVT"].to_numpy(float); gr = h["GR"].to_numpy(float)
            m = np.isfinite(tv) & np.isfinite(gr)
            if m.sum() < 30:
                continue
            o = np.argsort(tv[m]); d[w] = (tv[m][o], gr[m][o])
        _TRAIN_GRTVT = d
    return _TRAIN_GRTVT


def _train_centroids_z():
    """train井のXYZ重心(ΔZ用にZも)。"""
    global _TRAIN_CENTZ
    if _TRAIN_CENTZ is None:
        c = _cache()
        if c is not None:
            _TRAIN_CENTZ = c["centz"]; return _TRAIN_CENTZ
        tdir = DATA_DIR / "train"
        wids = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(tdir / "*__horizontal_well.csv"))})
        xyz = []
        for w in wids:
            h = pd.read_csv(tdir / f"{w}__horizontal_well.csv", usecols=["X", "Y", "Z"])
            xyz.append((float(h["X"].mean()), float(h["Y"].mean()), float(h["Z"].mean())))
        _TRAIN_CENTZ = (wids, np.array(xyz, float))
    return _TRAIN_CENTZ


def _cc(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[m], b[m])[0, 1]) if m.sum() > 15 and np.std(a[m]) > 1e-9 and np.std(b[m]) > 1e-9 else np.nan


def neighbors_gr_of(pre_tvt, pre_gr, cx, cy, cz, exclude_wid=None, k=K_GR):
    """★全train井から「targetの既知prefix層序(GR-vs-TVT)にGR一致する」top-K井を返す。
       選択= tip(実prefix点)でのGR相関(v95で最脱相関だったraw類似)。
       返り値 [(wid, sim, dist_xy, dz), ...] sim降順。refs材料は ref_grtvt(wid) で取る。"""
    G = _train_grtvt(); cw, cxyz = _train_centroids_z(); cmap = {w: i for i, w in enumerate(cw)}
    m = np.isfinite(pre_tvt) & np.isfinite(pre_gr); pt = pre_tvt[m]; pg = pre_gr[m]
    if len(pt) < 30:
        return []
    o = np.argsort(pt); pt = pt[o]; pg = pg[o]
    if len(pt) > 80:                                  # prefix間引き(高速化)
        s = len(pt) // 80; pt = pt[::s]; pg = pg[::s]
    lo, hi = float(pt[0]), float(pt[-1]); cand = []
    for w, (tv, gr) in G.items():
        if w == exclude_wid:
            continue
        if tv[0] > hi - 5 or tv[-1] < lo + 5:         # prefix範囲を被覆しない井は除外
            continue
        sim = _cc(pg, np.interp(pt, tv, gr))
        if np.isfinite(sim):
            cand.append((w, sim))
    cand.sort(key=lambda t: -t[1]); out = []
    for w, sim in cand[:k]:
        if w in cmap:
            dx = cxyz[cmap[w], 0] - cx; dy = cxyz[cmap[w], 1] - cy; dz = abs(cxyz[cmap[w], 2] - cz)
            out.append((w, float(sim), float(np.hypot(dx, dy)), float(dz)))
        else:
            out.append((w, float(sim), np.nan, np.nan))
    return out


def build_imputers():
    """train全井から FI/DI を構築(self除外は impute 時に self_wid で行う)。IMP_CACHE 有れば流用。"""
    c = _cache()
    if c is not None:
        return c["FI"], c["DI"], c["train_wids"]
    TR = DATA_DIR / "train"
    hw_paths = sorted(TR.glob("*__horizontal_well.csv"))
    train_wids = [p.stem.replace("__horizontal_well", "") for p in hw_paths]
    FI = FormationPlaneKNN(train_wids, TR)
    DI = DenseANCCImputer(train_wids, TR)
    return FI, DI, train_wids


def save_cache(path):
    """★全train参照構造(FI/DI/GR-TVT/重心×2)を1回だけ構築して pickle。submit の最初に1回呼ぶ。
       以後 IMP_CACHE=path を全 subprocess に渡せば、773井の再読込が消える(setup 重複を排除)。"""
    import pickle
    global _CACHE, _TRAIN_CENT, _TRAIN_GRTVT, _TRAIN_CENTZ
    _CACHE = None; _TRAIN_CENT = _TRAIN_GRTVT = _TRAIN_CENTZ = None   # raw build を強制
    TR = DATA_DIR / "train"
    hw = sorted(TR.glob("*__horizontal_well.csv")); wids = [p.stem.replace("__horizontal_well", "") for p in hw]
    obj = {"FI": FormationPlaneKNN(wids, TR), "DI": DenseANCCImputer(wids, TR), "train_wids": wids,
           "cent": _train_centroids(), "grtvt": _train_grtvt(), "centz": _train_centroids_z()}
    Path(path).write_bytes(pickle.dumps(obj, protocol=4))
    print(f"[imp_cache] saved {path} (FI={len(obj['FI'].df)}井 / grtvt={len(obj['grtvt'])}井)", flush=True)
    return path


if __name__ == "__main__":
    import io, sys
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
    FI, DI, wids = build_imputers()
    print(f"[imputers] FI={len(FI.df)}井 / DI点={len(DI.ancc)} / train={len(wids)}井")
    w0 = wids[0]
    h = pd.read_csv(DATA_DIR / "train" / f"{w0}__horizontal_well.csv", usecols=["X", "Y"])
    xy = np.array([[float(h["X"].mean()), float(h["Y"].mean())]])
    pf_pred, pf_d = FI.impute(xy, self_wid=w0)
    ap, asd, ad = DI.impute(xy, self_wid=w0)
    nb = neighbors_of(xy[0, 0], xy[0, 1], exclude_wid=w0)
    print(f"  {w0}: formation pred={np.round(pf_pred[0],1)} knn_d={pf_d[0]:.1f}")
    print(f"       dense ANCC={ap[0]:.1f}±{asd[0]:.1f} d={ad[0]:.1f} / 近傍{len(nb)}={[(w,round(d,0)) for w,d in nb]}")


In [ ]:
%%writefile features_v95.py
# -*- coding: utf-8 -*-
"""v93 module4: 特徴エンジン(v52 の v9-INLINE build_well を忠実移植)。

このモジュールは v52 `create_train_parquet_v52.py` の「特徴生成部」を、struct を一切参照せず
クリーンに再構築したもの。数値は v52 と一致させることが最優先(feature engine は v52 verbatim)。

★v93 で変わるのは pfA バンクの錨源だけ(struct → GRフリー錨)。それは module1/module2 の中で
  完結しており、この module4 は struct/v38/v41 を一切 import/参照しない。

構成:
  - CPU PF numba kernels (_interp1/_resamp/_beam_jit/_pf_ancc/_pf_z) : v52 と数値一致(verbatim)。
  - helpers (_gr_sig/_grid/_nn/_smooth/beam_search/_smooth_radius_values)。
  - run_pf_ancc / run_pf_z : マルチシード CPU PF ラッパ(v52 verbatim)。
  - 統計/特徴ヘルパ + build_well(per-well 特徴エンジン, v52 verbatim)。

★smooth-PF 注入(v52 の monkeypatch をクリーン化):
  v52 では run_pf_ancc を _patched_run_pf_ancc に monkeypatch し、前計算した GPU smooth-PF を
  スレッドローカル wid 経由で返していた。ここでは monkeypatch せず、
    - module-global `FWD_ENS` (dict[(wid, bank_name)] -> {mean,std})
    - thread-local `_CUR.wid`(`set_current_well(wid)` で設定)
  を持ち、run_pf_ancc が最初に FWD_ENS を引く。無ければ本物の CPU マルチシード PF にフォールバック
  (=このファイル単体でも動く)。run_pf_z は常に live 計算(build_well 内で本物を回す)。

  self/nbr 列の追加は build_self_nbr_columns(...) が担う(v52 の one() 相当)。
"""
import os, threading, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from numba import njit

# module3(空間 imputer / 近傍プール)から共有定数と imputer 構築を借りる
import imputers_v95 as imp
from imputers_v95 import FORMATIONS, PLANE_K, DENSE_K

warnings.filterwarnings("ignore"); np.seterr(all="ignore")

DATA_DIR = imp.DATA_DIR

# ==================== v9 定数(v52 line 331-345 と同一) ====================
SEED = 0
DENSE_SPW = 60
N_SPLITS = 5
PF_SUFFIX_START = 1
KEEP_FIRST_PF_ALIAS = True

BEAMS = [
    (15, 7.885250829793783, 69.70049362431021, 3, "gr"),
    (20, 49.43859691674568, 148.592415555271, 3, "hard"),
    (12, 11.84508580886079, 58.844708544034425, 2, "loose"),
    (15, 12.769364767855363, 89.75744127131894, 3, "mid"),
    (15, 10.693883733824851, 118.15908775467301, 8, "smooth"),
    (40, 67.99404057104671, 101.2117946585323, 1, "best"),
]

# 候補パスまわりのオフセット配列(v52 line 851-854 と同一)
ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], np.float32)
SC_OFFS = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
PF_OFFS = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)


# ==================== PF パラメータ写像(v52 line 360-401 verbatim) ====================
def _make_pf_params(raw, idx):
    """Optuna 形式の PF パラメータ dict を、この notebook が使う内部名へ写す。"""
    r = dict(raw)
    p = {}
    p["name"] = str(r.get("name", f"pf_{idx + PF_SUFFIX_START}"))
    p["suffix"] = str(r.get("suffix", f"_{idx + PF_SUFFIX_START}"))
    p["PF_N"] = int(r.get("PF_N", r.get("n_particles", 600)))
    p["ANCC_N"] = int(r.get("ANCC_N", r.get("n_particles", p["PF_N"])))
    p["PF_N_SEEDS"] = int(r.get("PF_N_SEEDS", r.get("n_seeds", 8)))
    p["PF_SEED0"] = int(r.get("PF_SEED0", SEED + idx * 100000))
    p["PF_LIKELIHOOD_SCALE"] = float(r.get("PF_LIKELIHOOD_SCALE", r.get("likelihood_scale", 20.0)))
    p["PF_INIT_SPR"] = float(r.get("PF_INIT_SPR", r.get("init_pos_std", 0.5)))
    p["PF_INIT_V_STD"] = float(r.get("PF_INIT_V_STD", r.get("init_rate_std", 0.01)))
    p["PF_MOM"] = float(r.get("PF_MOM", r.get("mom", 0.999)))
    p["PF_VN"] = float(r.get("PF_VN", r.get("vn", 0.001)))
    p["PF_PN"] = float(r.get("PF_PN", r.get("pn", 0.005)))
    p["PF_ROUGH_P"] = float(r.get("PF_ROUGH_P", r.get("rp", 0.5)))
    p["PF_ROUGH_V"] = float(r.get("PF_ROUGH_V", r.get("rr", 0.001)))
    p["PF_RESAMP"] = float(r.get("PF_RESAMP", r.get("resamp", 0.5)))
    p["PF_GR_SIG_MIN"] = float(r.get("PF_GR_SIG_MIN", r.get("gr_sig_min", 5.0)))
    p["PF_GR_SIG_MAX"] = float(r.get("PF_GR_SIG_MAX", r.get("gr_sig_max", 50.0)))
    p["PF_GR_SIG_DEF"] = float(r.get("PF_GR_SIG_DEF", r.get("gr_sig_def", 100.0)))
    p["PF_GR_SIG_MULT"] = float(r.get("PF_GR_SIG_MULT", r.get("gr_sig_mult", 2.0)))
    p["PF_TVT_CLIP_MARGIN"] = float(r.get("PF_TVT_CLIP_MARGIN", r.get("tvt_clip_margin", 50.0)))
    p["PF_RATE_CLIP"] = float(r.get("PF_RATE_CLIP", r.get("rate_clip", 0.0)))
    p["PF_GR_POWER"] = float(r.get("PF_GR_POWER", r.get("gr_power", 2.0)))
    p["PF_HGR_SMOOTH_R"] = int(r.get("PF_HGR_SMOOTH_R", r.get("hgr_smooth_r", 0)))
    p["PF_TW_GR_SMOOTH_R"] = int(r.get("PF_TW_GR_SMOOTH_R", r.get("tw_gr_smooth_r", 0)))
    p["PF_JUMP_PROB"] = float(r.get("PF_JUMP_PROB", r.get("jump_prob", 0.0)))
    p["PF_JUMP_STD"] = float(r.get("PF_JUMP_STD", r.get("jump_std", 0.0)))
    p["PF_GR_WIN"] = max(1, 2 * int(p["PF_HGR_SMOOTH_R"]) + 1)
    p["PF_GR_WT"] = float(r.get("PF_GR_WT", r.get("gr_wt", 0.3)))

    # ANCC-PF は TVT+Z を追跡。デフォルトは調整済み PF の遷移/初期化/リサンプルを流用。
    p["ANCC_ALPHA"] = float(r.get("ANCC_ALPHA", p["PF_MOM"]))
    p["ANCC_RN"] = float(r.get("ANCC_RN", p["PF_VN"]))
    p["ANCC_PN"] = float(r.get("ANCC_PN", p["PF_PN"]))
    p["ANCC_IR"] = float(r.get("ANCC_IR", p["PF_INIT_V_STD"]))
    p["ANCC_IS"] = float(r.get("ANCC_IS", p["PF_INIT_SPR"]))
    p["ANCC_RP"] = float(r.get("ANCC_RP", p["PF_ROUGH_P"]))
    p["ANCC_RR"] = float(r.get("ANCC_RR", p["PF_ROUGH_V"]))
    return p


# ---- PF_PARAM_SETS を module1 の BANK_ORDER から構築 ----
# ★重要: name をバンク名(pf_1/pf_2/pf_3/r0_seed32/r1_seed32/pfA)にする。
#   これで FWD_ENS/SELF_ENS/NBR_ENS のキー (wid, bank) と build_well 内 p["name"] が一致する。
#   suffix は _1.._6(=特徴列名は v52 と同一)。
import pf_banks_v95 as pf

PF_BANK = list(pf.BANK_ORDER)
PF_PARAM_SETS_RAW = [{**pf.BANK_PARAMS[_b], "name": _b} for _b in PF_BANK]
PF_PARAM_SETS = [_make_pf_params(raw, i) for i, raw in enumerate(PF_PARAM_SETS_RAW)]
if len(PF_PARAM_SETS) == 0:
    raise ValueError("PF_PARAM_SETS_RAW must contain at least one parameter set.")


# ==================== smooth-PF 注入(monkeypatch レス) ====================
FWD_ENS = {}          # (wid, bank_name) -> dict(mean, std)  ← create が populate
_CUR = threading.local()


def set_current_well(wid):
    """build_well 呼び出し前に(スレッドごとに)現在の坑井 wid を設定。"""
    _CUR.wid = wid


# ==================== CPU PF numba kernels(v52 line 410-594 verbatim) ====================
@njit(cache=False)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=False)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    """Beam search ±2 delta, Numba JIT."""
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):            # ±2: TVT can go down
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=False)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IR,IS,RP,RR,RESAMP,
              RATE_CLIP,GR_POWER,JUMP_PROB,JUMP_STD,CLIP_MARGIN,SEED_VALUE):
    if SEED_VALUE >= 0:
        np.random.seed(SEED_VALUE)
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+IR*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    log_lik=0.
    tvt_lo=vmin-CLIP_MARGIN
    tvt_hi=vmin+(len(gg)-1)*step+CLIP_MARGIN
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            if RATE_CLIP>0.:
                if rate[j] > RATE_CLIP: rate[j] = RATE_CLIP
                elif rate[j] < -RATE_CLIP: rate[j] = -RATE_CLIP
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            # Rare jump particles for extreme TVT movement. jump_prob=0 disables this.
            if JUMP_PROB>0. and JUMP_STD>0. and np.random.random()<JUMP_PROB:
                pos[j]+=JUMP_STD*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,tvt_lo); tvt_j=min(tvt_j,tvt_hi)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.; avg_lk=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=abs((gr_v[i]-eg)/gs)
                dp=d**GR_POWER
                lk=max(np.exp(-0.5*dp) if dp<600. else 0.,1e-300)
                old_w=w[j]
                avg_lk+=old_w*lk
                w[j]=old_w*lk; ws+=w[j]
            log_lik+=np.log(max(avg_lk,1e-300))
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            if RATE_CLIP>0.:
                for j in range(N):
                    if rate[j] > RATE_CLIP: rate[j] = RATE_CLIP
                    elif rate[j] < -RATE_CLIP: rate[j] = -RATE_CLIP
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_,log_lik

@njit(cache=False)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP,IP_STD,IV_STD,
          RATE_CLIP,GR_POWER,JUMP_PROB,JUMP_STD,CLIP_MARGIN,SEED_VALUE):
    if SEED_VALUE >= 0:
        np.random.seed(SEED_VALUE)
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+IP_STD*np.random.randn()
        vel[j]=iv+IV_STD*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            if RATE_CLIP>0.:
                if vel[j] > RATE_CLIP: vel[j] = RATE_CLIP
                elif vel[j] < -RATE_CLIP: vel[j] = -RATE_CLIP
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            if JUMP_PROB>0. and JUMP_STD>0. and np.random.random()<JUMP_PROB:
                pos[j]+=JUMP_STD*np.random.randn()
            pos[j]=max(pos[j],vmin-CLIP_MARGIN); pos[j]=min(pos[j],vmin+(len(gg_p)-1)*step+CLIP_MARGIN)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=abs((gr_v[i]-ep)/gs)
                dpow=dp**GR_POWER
                lp=max(np.exp(-0.5*dpow) if dpow<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=abs((gr_sm_v[i]-es)/(gs*1.5))
                    dsp=ds**GR_POWER
                    ls=max(np.exp(-0.5*dsp) if dsp<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_


# ==================== helpers(v52 line 597-639 verbatim) ====================
def _smooth_radius_values(vals, fb, r):
    r = int(r)
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(float(fb))
    if r <= 0:
        return s.to_numpy(np.float32)
    return s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt,tw_gr,step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

def _gr_sig(hw,tw_tvt,tw_gr,p):
    # Match pf_tuning_4_early_resume: estimate sigma using the same smoothed
    # horizontal GR and typewell GR that PF likelihood will use.
    gr_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    hgr=_smooth_radius_values(hw['GR'].astype(float).to_numpy(), gr_fb, p["PF_HGR_SMOOTH_R"])
    kn=hw[hw['TVT_input'].notna()]
    if len(kn)<20:
        gs=float(p["PF_GR_SIG_DEF"])
    else:
        kpos=hw.index.get_indexer(kn.index)
        resid=hgr[kpos]-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)
        gs=float(np.nanstd(resid))
        if not np.isfinite(gs) or gs<=0:
            gs=float(p["PF_GR_SIG_DEF"])
    return float(np.clip(gs*p["PF_GR_SIG_MULT"],p["PF_GR_SIG_MIN"],p["PF_GR_SIG_MAX"]))

def _nn(arr,v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i

def _smooth(vals,fb,r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)


# ==================== マルチシード CPU PF ラッパ ====================
def run_pf_ancc(hw, tw_tvt, tw_gr, p):
    """★smooth-PF 注入つき run_pf_ancc。
       まず FWD_ENS[(現在wid, p['name'])] を引き、あればその smooth-PF 平均/std を返す。
       無ければ本物の CPU マルチシード PF(v52 line 641-677 verbatim)を計算(=単体実行可)。"""
    r = FWD_ENS.get((getattr(_CUR, "wid", None), p.get("name")))
    if r is not None:
        return r["mean"].astype(np.float32), r["std"].astype(np.float32)
    # ---- fallback: 本物の CPU マルチシード ANCC-PF(v52 verbatim) ----
    tw_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth_radius_values(tw_gr, tw_fb, p["PF_TW_GR_SMOOTH_R"]).astype(np.float64)
    gs=_gr_sig(hw,tw_tvt,tw_gr_pf,p)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr_pf)
    md_ev=ev['MD'].values.astype(np.float64)
    z_ev=ev['Z'].values.astype(np.float64)
    # Tuning notebook interpolates + optionally smooths horizontal GR before likelihood.
    gr_fb=tw_fb
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(gr_fb).to_numpy(dtype=float)
    gr_proc=_smooth_radius_values(gr_full, gr_fb, p["PF_HGR_SMOOTH_R"])
    ev_pos=hw.index.get_indexer(ev.index)
    gr_ev=gr_proc[ev_pos].astype(np.float64)

    preds=[]; stds=[]; liks=[]
    for s in range(int(p["PF_N_SEEDS"])):
        pts,std,ll=_pf_ancc(md_ev,z_ev,gr_ev,gg,gmin,gst,
                            gs,ls,ir,int(p["ANCC_N"]),
                            p["ANCC_ALPHA"],p["ANCC_RN"],p["ANCC_PN"],p["ANCC_IR"],p["ANCC_IS"],p["ANCC_RP"],p["ANCC_RR"],
                            p["PF_RESAMP"],p["PF_RATE_CLIP"],p["PF_GR_POWER"],p["PF_JUMP_PROB"],p["PF_JUMP_STD"],
                            p["PF_TVT_CLIP_MARGIN"],int(p["PF_SEED0"])+s)
        preds.append(pts); stds.append(std); liks.append(ll)
    pred_arr=np.stack(preds,0)
    std_arr=np.stack(stds,0)
    lik_arr=np.asarray(liks,dtype=np.float64)
    lik_arr=lik_arr-np.nanmax(lik_arr)
    ww=np.exp(lik_arr/max(float(p["PF_LIKELIHOOD_SCALE"]),1e-6))
    ww=ww/max(float(ww.sum()),1e-300)
    ens=(ww[:,None]*pred_arr).sum(0)
    ens_var=(ww[:,None]*(std_arr**2+(pred_arr-ens[None,:])**2)).sum(0)
    return ens.astype(np.float32),np.sqrt(np.maximum(ens_var,0.)).astype(np.float32)

def run_pf_z(hw,tw_tvt,tw_gr,p):
    """Z追跡 PF(常に live 計算, v52 line 679-710 verbatim)。"""
    tw_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth_radius_values(tw_gr, tw_fb, p["PF_TW_GR_SMOOTH_R"]).astype(np.float64)
    gs=_gr_sig(hw,tw_tvt,tw_gr_pf,p)
    tw_s=pd.Series(tw_gr_pf).rolling(p["PF_GR_WIN"],center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr_pf)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_fb=tw_fb
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(gr_fb).to_numpy(dtype=float)
    gr_proc=_smooth_radius_values(gr_full, gr_fb, p["PF_HGR_SMOOTH_R"])
    gr_sm=pd.Series(gr_proc).rolling(p["PF_GR_WIN"],center=True,min_periods=1).mean().to_numpy(dtype=float)
    ev_pos=hw.index.get_indexer(ev.index)
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   gr_proc[ev_pos].astype(np.float64),
                   gr_sm[ev_pos].astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,int(p["PF_N"]),
                   p["PF_MOM"],p["PF_VN"],p["PF_PN"],p["PF_GR_WT"],p["PF_ROUGH_P"],p["PF_ROUGH_V"],p["PF_RESAMP"],
                   p["PF_INIT_SPR"],p["PF_INIT_V_STD"],p["PF_RATE_CLIP"],p["PF_GR_POWER"],p["PF_JUMP_PROB"],p["PF_JUMP_STD"],
                   p["PF_TVT_CLIP_MARGIN"],int(p["PF_SEED0"]))
    return pts.astype(np.float32),std.astype(np.float32)


# ---- numba warmup(v52 line 713-722) ----
_md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
_gg=np.linspace(45,55,100,np.float64)
_p0=PF_PARAM_SETS[0]
_pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,
         _p0["ANCC_ALPHA"],_p0["ANCC_RN"],_p0["ANCC_PN"],_p0["ANCC_IR"],_p0["ANCC_IS"],_p0["ANCC_RP"],_p0["ANCC_RR"],_p0["PF_RESAMP"],
         _p0["PF_RATE_CLIP"],_p0["PF_GR_POWER"],_p0["PF_JUMP_PROB"],_p0["PF_JUMP_STD"],_p0["PF_TVT_CLIP_MARGIN"],123)
_pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,
      _p0["PF_MOM"],_p0["PF_VN"],_p0["PF_PN"],_p0["PF_GR_WT"],_p0["PF_ROUGH_P"],_p0["PF_ROUGH_V"],_p0["PF_RESAMP"],_p0["PF_INIT_SPR"],_p0["PF_INIT_V_STD"],
      _p0["PF_RATE_CLIP"],_p0["PF_GR_POWER"],_p0["PF_JUMP_PROB"],_p0["PF_JUMP_STD"],_p0["PF_TVT_CLIP_MARGIN"],123)
_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)


# ==================== 統計 / 特徴ヘルパ(v52 line 724-943 verbatim) ====================
def robust_slope(x,y,w=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

def seg_b_well(ktvt,kz,form_col):
    """Segment b_well: early/mid/late thirds + full prefix.
    Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    # WLS (tail-upweighted)
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    # Score-weighted ensemble (NEW: softmax-weighted combination)
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

def _rolling_mean_np(x, w):
    return pd.Series(np.asarray(x, np.float32)).rolling(w, center=True, min_periods=1).mean().to_numpy(np.float32)

def _rolling_std_np(x, w):
    return pd.Series(np.asarray(x, np.float32)).rolling(w, center=True, min_periods=1).std().fillna(0.).to_numpy(np.float32)

def _safe_grad(y, x=None):
    y = np.asarray(y, np.float32)
    n = len(y)
    if n <= 1:
        return np.zeros(n, np.float32)
    if x is None:
        g = np.gradient(y).astype(np.float32)
    else:
        x = np.asarray(x, np.float32)
        dx = np.gradient(x).astype(np.float32)
        dx = np.where(np.abs(dx) < 1e-6, np.nan, dx)
        g = np.gradient(y).astype(np.float32) / dx
        g = np.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return g.astype(np.float32)

def _add_path_shape_features(feats, prefix, path, md=None):
    """Local geometry of a candidate TVT path. No target is used."""
    path = np.asarray(path, np.float32)
    rate = _safe_grad(path, md)
    acc = _safe_grad(rate, md)
    feats[f'{prefix}_rate'] = rate.astype(np.float32)
    feats[f'{prefix}_acc'] = acc.astype(np.float32)
    feats[f'{prefix}_rate_abs'] = np.abs(rate).astype(np.float32)
    feats[f'{prefix}_acc_abs'] = np.abs(acc).astype(np.float32)
    feats[f'{prefix}_rate_rmean21'] = _rolling_mean_np(rate, 21)
    feats[f'{prefix}_rate_rstd21'] = _rolling_std_np(rate, 21)
    feats[f'{prefix}_acc_rmean21'] = _rolling_mean_np(acc, 21)

def _add_gr_match_features(feats, prefix, path, hgr, tw_tvt, tw_gr, offs):
    """How well a candidate TVT path explains the observed horizontal GR."""
    path = np.asarray(path, np.float32)
    hgr = np.asarray(hgr, np.float32)
    offs = np.asarray(offs, np.float32)
    gr_mat = np.stack([np.interp(path + float(o), tw_tvt, tw_gr) for o in offs], axis=1).astype(np.float32)
    resid = (hgr[:, None] - gr_mat).astype(np.float32)
    abs_resid = np.abs(resid)
    zero_idx = int(np.argmin(np.abs(offs)))
    best_idx = abs_resid.argmin(axis=1)
    row_idx = np.arange(len(path))
    r0 = resid[:, zero_idx].astype(np.float32)
    best_signed = resid[row_idx, best_idx].astype(np.float32)
    feats[f'{prefix}_gr_resid0'] = r0
    feats[f'{prefix}_gr_abs0'] = np.abs(r0).astype(np.float32)
    feats[f'{prefix}_gr_abs_min'] = abs_resid[row_idx, best_idx].astype(np.float32)
    feats[f'{prefix}_gr_best_off'] = offs[best_idx].astype(np.float32)
    feats[f'{prefix}_gr_best_resid'] = best_signed
    feats[f'{prefix}_gr_resid0_rmean21'] = _rolling_mean_np(r0, 21)
    feats[f'{prefix}_gr_resid0_rstd21'] = _rolling_std_np(r0, 21)
    feats[f'{prefix}_gr_abs0_rmean21'] = _rolling_mean_np(np.abs(r0), 21)

def _pairwise_abs_stats(mat):
    mat = np.asarray(mat, np.float32)
    k = mat.shape[1]
    if k < 2:
        z = np.zeros(mat.shape[0], np.float32)
        return z, z
    vals = []
    for i in range(k):
        for j in range(i + 1, k):
            vals.append(np.abs(mat[:, i] - mat[:, j]).astype(np.float32))
    diffs = np.stack(vals, axis=1)
    return diffs.mean(axis=1).astype(np.float32), diffs.max(axis=1).astype(np.float32)

def _corr_summary(mat):
    mat = np.asarray(mat, np.float32)
    k = mat.shape[1]
    if k < 2:
        return 0.0, 0.0, 0.0
    vals = []
    for i in range(k):
        xi = mat[:, i]
        xi_std = float(np.std(xi))
        for j in range(i + 1, k):
            xj = mat[:, j]
            xj_std = float(np.std(xj))
            if xi_std < 1e-6 or xj_std < 1e-6:
                vals.append(0.0)
            else:
                vals.append(float(np.corrcoef(xi, xj)[0, 1]))
    vals = np.nan_to_num(np.asarray(vals, np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return float(vals.mean()), float(vals.min()), float(vals.max())


def _add_pf_feature_columns(feats, pf_runs, last_tvt, tvt_f_ancc, tvt_dense, hgr, tw_tvt, tw_gr, sc,
                            hmd=None, beam_mean=None, sc_ens=None, hyb_ref=None):
    """Add PF-derived feature columns for all parameter sets.(v52 line 946-1051 verbatim)"""
    def add_one(r, suffix):
        pf_a = r["pf_ancc"].astype(np.float32)
        std_a = r["pf_ancc_std"].astype(np.float32)
        has_z = bool(r["has_z"])
        pf_z = r["pf_z"].astype(np.float32) if has_z else sc(last_tvt)
        feats[f'pf_ancc{suffix}'] = pf_a
        feats[f'pf_ancc_std{suffix}'] = std_a
        feats[f'pf_ancc_delta{suffix}'] = (pf_a - np.float32(last_tvt)).astype(np.float32)
        feats[f'pf_z{suffix}'] = pf_z
        feats[f'pf_z_delta{suffix}'] = (pf_z - np.float32(last_tvt)).astype(np.float32) if has_z else sc(0.)
        feats[f'pf_vs_z{suffix}'] = (pf_a - pf_z).astype(np.float32) if has_z else sc(0.)
        feats[f'pf_vs_spatial{suffix}'] = (pf_a - tvt_f_ancc).astype(np.float32)
        feats[f'pf_vs_dense{suffix}'] = (pf_a - tvt_dense).astype(np.float32)
        for o in PF_OFFS:
            feats[f'tdpf{int(o)}{suffix}'] = hgr - np.interp(pf_a + o, tw_tvt, tw_gr).astype(np.float32)

        prefix = 'pf' if suffix == '' else f'pf{suffix}'
        _add_gr_match_features(feats, prefix, pf_a, hgr, tw_tvt, tw_gr, PF_OFFS)
        if hmd is not None:
            _add_path_shape_features(feats, prefix, pf_a, hmd)

    if KEEP_FIRST_PF_ALIAS and len(pf_runs) > 0:
        add_one(pf_runs[0], "")
    for r in pf_runs:
        add_one(r, r["suffix"])

    if len(pf_runs) >= 2:
        pf_stack = np.stack([r["pf_ancc"].astype(np.float32) for r in pf_runs], axis=1)
        pf_mean = pf_stack.mean(1).astype(np.float32)
        pf_med = np.median(pf_stack, axis=1).astype(np.float32)
        pf_pair_mean, pf_pair_max = _pairwise_abs_stats(pf_stack)
        c_mean, c_min, c_max = _corr_summary(pf_stack)
        feats['pf_ancc_mean'] = pf_mean
        feats['pf_ancc_med'] = pf_med
        feats['pf_ancc_std_between'] = pf_stack.std(1).astype(np.float32)
        feats['pf_ancc_range_between'] = (pf_stack.max(1) - pf_stack.min(1)).astype(np.float32)
        feats['pf_ancc_iqr_between'] = (np.percentile(pf_stack, 75, axis=1) - np.percentile(pf_stack, 25, axis=1)).astype(np.float32)
        feats['pf_ancc_pair_absmean_between'] = pf_pair_mean
        feats['pf_ancc_pair_absmax_between'] = pf_pair_max
        feats['pf_ancc_mean_delta'] = (pf_mean - np.float32(last_tvt)).astype(np.float32)
        feats['pf_ancc_med_delta'] = (pf_med - np.float32(last_tvt)).astype(np.float32)
        feats['pf_ancc_corr_mean'] = sc(c_mean)
        feats['pf_ancc_corr_min'] = sc(c_min)
        feats['pf_ancc_corr_max'] = sc(c_max)

        for i in range(len(pf_runs)):
            si = pf_runs[i]['suffix']
            feats[f'pf_dev_mean{si}'] = (pf_stack[:, i] - pf_mean).astype(np.float32)
            feats[f'pf_dev_med{si}'] = (pf_stack[:, i] - pf_med).astype(np.float32)
        for i in range(len(pf_runs)):
            for j in range(i + 1, len(pf_runs)):
                si = pf_runs[i]['suffix'].replace('_', '')
                sj = pf_runs[j]['suffix'].replace('_', '')
                diff = (pf_stack[:, i] - pf_stack[:, j]).astype(np.float32)
                feats[f'pf_diff_{si}_{sj}'] = diff
                feats[f'pf_absdiff_{si}_{sj}'] = np.abs(diff).astype(np.float32)

        _add_gr_match_features(feats, 'pf_mean', pf_mean, hgr, tw_tvt, tw_gr, PF_OFFS)
        _add_gr_match_features(feats, 'pf_med', pf_med, hgr, tw_tvt, tw_gr, PF_OFFS)
        if hmd is not None:
            _add_path_shape_features(feats, 'pf_mean', pf_mean, hmd)
            _add_path_shape_features(feats, 'pf_med', pf_med, hmd)

        if beam_mean is not None:
            feats['pf_mean_vs_beam_mean'] = (pf_mean - np.asarray(beam_mean, np.float32)).astype(np.float32)
            feats['pf_med_vs_beam_mean'] = (pf_med - np.asarray(beam_mean, np.float32)).astype(np.float32)
        if sc_ens is not None:
            feats['pf_mean_vs_sc_ens'] = (pf_mean - np.asarray(sc_ens, np.float32)).astype(np.float32)
            feats['pf_med_vs_sc_ens'] = (pf_med - np.asarray(sc_ens, np.float32)).astype(np.float32)
        if hyb_ref is not None:
            feats['pf_mean_vs_hyb'] = (pf_mean - np.asarray(hyb_ref, np.float32)).astype(np.float32)
        feats['pf_mean_vs_spatial'] = (pf_mean - tvt_f_ancc).astype(np.float32)
        feats['pf_mean_vs_dense'] = (pf_mean - tvt_dense).astype(np.float32)

        z_list = [r["pf_z"].astype(np.float32) for r in pf_runs if bool(r["has_z"])]
        if len(z_list) >= 2:
            z_stack = np.stack(z_list, axis=1)
            z_mean = z_stack.mean(1).astype(np.float32)
            z_med = np.median(z_stack, axis=1).astype(np.float32)
            z_pair_mean, z_pair_max = _pairwise_abs_stats(z_stack)
            zc_mean, zc_min, zc_max = _corr_summary(z_stack)
            feats['pf_z_mean'] = z_mean
            feats['pf_z_med'] = z_med
            feats['pf_z_std_between'] = z_stack.std(1).astype(np.float32)
            feats['pf_z_range_between'] = (z_stack.max(1) - z_stack.min(1)).astype(np.float32)
            feats['pf_z_pair_absmean_between'] = z_pair_mean
            feats['pf_z_pair_absmax_between'] = z_pair_max
            feats['pf_z_corr_mean'] = sc(zc_mean)
            feats['pf_z_corr_min'] = sc(zc_min)
            feats['pf_z_corr_max'] = sc(zc_max)
            feats['pf_mean_vs_z_mean'] = (pf_mean - z_mean).astype(np.float32)
            feats['pf_med_vs_z_med'] = (pf_med - z_med).astype(np.float32)


# ==================== imputer(module3)を借りる ====================
_FI = None
_DI = None


def set_imputers(FI, DI):
    """create 側で構築した imputer を注入(二重構築回避)。"""
    global _FI, _DI
    _FI, _DI = FI, DI


def _ensure_imputers():
    """build_well が imputer を必要とするときに、無ければ自前で構築(単体実行可)。"""
    global _FI, _DI
    if _FI is None or _DI is None:
        FI, DI, _ = imp.build_imputers()
        _FI, _DI = FI, DI


# ==================== per-well 特徴エンジン(v52 line 1054-1329 verbatim) ====================
def build_well(hw_path, tw_path, is_train):
    _ensure_imputers()
    global _FI, _DI
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    try:
        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None

    pf_runs=[]
    for pconf in PF_PARAM_SETS:
        pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr,pconf)
        if len(pf_a)==0: return None
        pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr,pconf)
        has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))
        pf_runs.append({
            "name": pconf["name"],
            "suffix": pconf["suffix"],
            "pf_ancc": pf_a.astype(np.float32),
            "pf_ancc_std": std_a.astype(np.float32),
            "pf_z": pf_z.astype(np.float32) if len(pf_z)==len(pf_a) else np.full(len(pf_a), np.nan, np.float32),
            "pf_z_std": std_z.astype(np.float32) if len(std_z)==len(pf_a) else np.full(len(pf_a), np.nan, np.float32),
            "has_z": has_z,
        })
    pf_primary=pf_runs[0]
    pf_use=pf_primary["pf_ancc"].astype(np.float32)
    std_use=pf_primary["pf_ancc_std"].astype(np.float32)
    pf_z=pf_primary["pf_z"].astype(np.float32)
    has_z=bool(pf_primary["has_z"])

    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

    # 7 beams (Numba JIT ±2)
    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS:
        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    # Robust reference beam: older versions used tags 'cons'/'sm5', this notebook uses 'gr'/'smooth'.
    _beam_a=bpaths['cons'] if 'cons' in bpaths else bpaths.get('gr', next(iter(bpaths.values())))
    _beam_b=bpaths['sm5'] if 'sm5' in bpaths else bpaths.get('smooth', _beam_a)
    beam_ref=(_beam_a+_beam_b)/2.

    # Multi-scale NCC → score-weighted ensemble
    ktvt=kn['TVT_input'].to_numpy(np.float32)
    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc_cons=(sc8+sc15+sc25)/3.
    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single

    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
    slp_z=robust_slope(kz,ktvt)

    swid=wid if is_train else None
    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

    # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
        tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
        tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
        tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
        tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
        tvt_fs[f'tvtF50_{fn}']=tvt_f50
        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
        tvt_fs[f'bw50_{fn}']=np.float32(b_late)
        tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
        tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
        form_list.append(tvt_f)

    fs=np.stack(form_list,1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d =fs.std(1).astype(np.float32)
    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
    b_vd=ktvt+z_kn-d_kn
    _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
    b_d=float(np.median(b_vd))
    tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
    tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
    tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
    res_kn=ktvt+z_kn-d_kn
    d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

    candidate_names = [f"pf_ancc{r['suffix']}" for r in pf_runs] + [f"beam_{k}" for k in bpaths.keys()] + [
        'sc8', 'sc15', 'sc25', 'sc_ens', 'spatial_ancc', 'dense_ancc'
    ]
    all_sigs=[r['pf_ancc'] for r in pf_runs]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1).astype(np.float32)
    sig_std=sig_mat.std(1).astype(np.float32)
    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    sig_med=np.median(sig_mat,axis=1).astype(np.float32)
    sig_q10=np.percentile(sig_mat,10,axis=1).astype(np.float32)
    sig_q25=np.percentile(sig_mat,25,axis=1).astype(np.float32)
    sig_q75=np.percentile(sig_mat,75,axis=1).astype(np.float32)
    sig_q90=np.percentile(sig_mat,90,axis=1).astype(np.float32)
    sig_iqr=(sig_q75-sig_q25).astype(np.float32)
    sig_range=(sig_mat.max(1)-sig_mat.min(1)).astype(np.float32)
    sig_mad=np.mean(np.abs(sig_mat-sig_med[:,None]),axis=1).astype(np.float32)
    sig_centered=(sig_mat-sig_mat.mean(1,keepdims=True)).astype(np.float32)
    sig_skew=(np.mean(sig_centered**3,axis=1)/(np.maximum(sig_std,1e-6)**3)).astype(np.float32)

    sig_gr_pred_mat=np.stack([np.interp(sig_mat[:,j],tw_tvt,tw_gr) for j in range(sig_mat.shape[1])],axis=1).astype(np.float32)
    sig_gr_resid_mat=(hgr[:,None]-sig_gr_pred_mat).astype(np.float32)
    sig_gr_abs_mat=np.abs(sig_gr_resid_mat).astype(np.float32)
    sig_gr_best_idx=sig_gr_abs_mat.argmin(axis=1)
    _sig_rows=np.arange(len(hgr))
    sig_gr_best_path=sig_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_best_abs=sig_gr_abs_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_best_resid=sig_gr_resid_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_pred_std=sig_gr_pred_mat.std(1).astype(np.float32)
    sig_gr_resid_mean=sig_gr_resid_mat.mean(1).astype(np.float32)
    sig_gr_resid_std=sig_gr_resid_mat.std(1).astype(np.float32)

    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                   ).iloc[ev.index].values.astype(np.float32)

    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)

    feats={
        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':sc(last_tvt),
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        'sig_std':sig_std,'sig_mean_d':sig_mean,
        'sig_med_d':(sig_med-np.float32(last_tvt)).astype(np.float32),
        'sig_q10_d':(sig_q10-np.float32(last_tvt)).astype(np.float32),
        'sig_q25_d':(sig_q25-np.float32(last_tvt)).astype(np.float32),
        'sig_q75_d':(sig_q75-np.float32(last_tvt)).astype(np.float32),
        'sig_q90_d':(sig_q90-np.float32(last_tvt)).astype(np.float32),
        'sig_iqr':sig_iqr,'sig_range':sig_range,'sig_mad':sig_mad,'sig_skew':sig_skew,
        'sig_gr_pred_std':sig_gr_pred_std,
        'sig_gr_resid_mean':sig_gr_resid_mean,'sig_gr_resid_std':sig_gr_resid_std,
        'sig_gr_best_d':(sig_gr_best_path-np.float32(last_tvt)).astype(np.float32),
        'sig_gr_best_idx':sig_gr_best_idx.astype(np.float32),
        'sig_gr_best_abs':sig_gr_best_abs,'sig_gr_best_resid':sig_gr_best_resid,
        'sig_gr_best_vs_med':(sig_gr_best_path-sig_med).astype(np.float32),
        **tvt_fs,
        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
        'spatial_knn_dist':knn_d,
        'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_spatial':(beam_ref-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':(sc_ens-beam_ref).astype(np.float32),
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),
        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
        'z':z_ev,
        'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
        'dz':(z_ev-float(lk['Z'])).astype(np.float32),
        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
        'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
    }
    _add_pf_feature_columns(
        feats, pf_runs, last_tvt, tvt_fs['tvtF_ANCC'], tvt_dense, hgr, tw_tvt, tw_gr, sc,
        hmd=hmd,
        beam_mean=(np.float32(last_tvt) + feats['beam_mean_d']).astype(np.float32),
        sc_ens=sc_ens,
        hyb_ref=hyb_ref,
    )

    # Shape/GR-match features for non-PF ensemble references.
    beam_mean_path=(np.float32(last_tvt)+feats['beam_mean_d']).astype(np.float32)
    beam_med_path=(np.float32(last_tvt)+feats['beam_med_d']).astype(np.float32)
    _add_path_shape_features(feats,'beam_mean',beam_mean_path,hmd)
    _add_path_shape_features(feats,'beam_med',beam_med_path,hmd)
    _add_path_shape_features(feats,'sc_ens',sc_ens,hmd)
    _add_path_shape_features(feats,'hyb',hyb_ref,hmd)
    _add_path_shape_features(feats,'sig_med',sig_med,hmd)
    _add_path_shape_features(feats,'sig_gr_best',sig_gr_best_path,hmd)
    _add_gr_match_features(feats,'beam_mean',beam_mean_path,hgr,tw_tvt,tw_gr,BEAM_OFFS)
    _add_gr_match_features(feats,'beam_med',beam_med_path,hgr,tw_tvt,tw_gr,BEAM_OFFS)
    _add_gr_match_features(feats,'sc_ens',sc_ens,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'hyb',hyb_ref,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'sig_med',sig_med,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'sig_gr_best',sig_gr_best_path,hgr,tw_tvt,tw_gr,SC_OFFS)

    for k,v in rolls.items(): feats[k]=v

    # Additional GR morphology features. These use only observed horizontal/typewell GR.
    feats['gr_z21']=((hgr-rolls['grm21'])/(rolls['grs21']+1e-3)).astype(np.float32)
    feats['gr_z51']=((hgr-rolls['grm51'])/(rolls['grs51']+1e-3)).astype(np.float32)
    feats['gr_abs_d1']=np.abs(gr_d1).astype(np.float32)
    feats['gr_abs_d2']=np.abs(gr_d2).astype(np.float32)
    feats['gr_d1_rmean21']=_rolling_mean_np(gr_d1,21)
    feats['gr_abs_d1_rmean21']=_rolling_mean_np(np.abs(gr_d1),21)
    feats['gr_rough21']=_rolling_mean_np(np.abs(gr_d1),21)
    feats['gr_rough51']=_rolling_mean_np(np.abs(gr_d1),51)
    _tw_gr_sorted=np.sort(tw_gr.astype(np.float32))
    _kg_sorted=np.sort(kgr.astype(np.float32)) if len(kgr)>0 else _tw_gr_sorted
    feats['gr_pct_tw']=(np.searchsorted(_tw_gr_sorted,hgr,side='right')/max(len(_tw_gr_sorted),1)).astype(np.float32)
    feats['gr_pct_known']=(np.searchsorted(_kg_sorted,hgr,side='right')/max(len(_kg_sorted),1)).astype(np.float32)
    feats['gr_z_known']=((hgr-np.float32(np.mean(kgr)))/(np.float32(np.std(kgr))+1e-3)).astype(np.float32)
    feats['gr_z_tw']=((hgr-np.float32(np.mean(tw_gr)))/(np.float32(np.std(tw_gr))+1e-3)).astype(np.float32)
    feats['gr_vs_known_last']=(hgr-np.float32(kgr[-1])).astype(np.float32)
    feats['gr_vs_known_mean']=(hgr-np.float32(np.mean(kgr))).astype(np.float32)

    # Geometry roughness of the actual drilled path.
    step_xy=np.sqrt(dxdmd**2+dydmd**2).astype(np.float32)
    feats['xy_rate']=step_xy
    feats['xyz_rate']=np.sqrt(dxdmd**2+dydmd**2+dzdmd**2).astype(np.float32)
    feats['z_rate_abs']=np.abs(dzdmd).astype(np.float32)
    feats['dzdmd_rmean21']=_rolling_mean_np(dzdmd,21)
    feats['dzdmd_rstd21']=_rolling_std_np(dzdmd,21)
    feats['xy_rate_rmean21']=_rolling_mean_np(step_xy,21)
    feats['xy_rate_rstd21']=_rolling_std_np(step_xy,21)

    result=pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
    return result


# ==================== self/nbr 列追加(v52 one() line 1725-1763 verbatim) ====================
METHODS_V95 = ["self", "nbr", "nbr_gr5", "self_graft"]   # tw は build_well の pf_ancc(baseline)


def build_self_nbr_columns(df, wid, ENS, ZGRAD, NBR_META, PROV):
    """★v95: 5表現(tw baseline + self/nbr/nbr_gr5/self_graft)の smooth-PF 予測・std・delta・vs・loglik、
       表現間比較、クロスバンク集約、そして各表現の provenance(GR一致度/距離/ΔZ/類似度/較正/被覆)を列追加。
       ENS = {method: {(wid,bank)->dict(mean,std,loglik)}}。PROV = {wid: {method: {...provenance...}}}。"""
    n = len(df); lk = df["last_known_tvt"].to_numpy(float)
    def col(a):
        out = np.full(n, np.nan, np.float32)
        if a is not None:
            mm = min(len(a), n); out[:mm] = np.asarray(a[:mm], np.float32)
        return out
    def setp(name, v):
        df[name] = np.float32(v) if (v is not None and np.isfinite(v)) else np.float32(np.nan)
    mt = NBR_META.get(wid, {}); pv = PROV.get(wid, {})
    acc = {m: [] for m in METHODS_V95}; T_all = []
    for p in PF_PARAM_SETS:
        suf = p["suffix"]; nm = p["name"]
        twk = df[f"pf_ancc{suf}"].to_numpy(float) if f"pf_ancc{suf}" in df.columns else lk.copy()
        preds = {}
        for m in METHODS_V95:
            e = ENS[m].get((wid, nm))
            pm_ = col(e["mean"] if e else None); ps_ = col(e["std"] if e else None)
            df[f"pf_{m}{suf}"] = pm_; df[f"pf_{m}_std{suf}"] = ps_
            df[f"pf_{m}_delta{suf}"] = (pm_ - lk).astype(np.float32)
            df[f"pf_{m}_vs_tw{suf}"] = (pm_ - twk).astype(np.float32)      # ★baseline tw との不一致
            df[f"pf_{m}_loglik{suf}"] = np.full(n, np.float32(e["loglik"]) if (e and "loglik" in e) else np.float32(np.nan), np.float32)
            preds[m] = pm_.astype(float); acc[m].append(pm_.astype(float))
        # 表現間比較(相対的妥当性)
        df[f"pf_self_vs_nbr{suf}"] = (preds["self"] - preds["nbr"]).astype(np.float32)
        df[f"pf_nbr_gr5_vs_nbr{suf}"] = (preds["nbr_gr5"] - preds["nbr"]).astype(np.float32)
        df[f"pf_self_graft_vs_self{suf}"] = (preds["self_graft"] - preds["self"]).astype(np.float32)
        # 5表現 consensus(tw + 4手法)
        st = np.stack([twk] + [preds[m] for m in METHODS_V95], axis=1)
        df[f"pf_agree_std{suf}"] = np.nanstd(st, axis=1).astype(np.float32)
        df[f"pf_agree_range{suf}"] = (np.nanmax(st, axis=1) - np.nanmin(st, axis=1)).astype(np.float32)
        T_all.append(twk)
    # クロスバンク集約(各手法: 平均 / バンク間ばらつき / 平均のtw不一致)
    TM = np.stack(T_all, 1)
    for m in METHODS_V95:
        M = np.stack(acc[m], 1)
        df[f"pf_{m}_mean"] = np.nanmean(M, 1).astype(np.float32)
        df[f"pf_{m}_pstd"] = np.nanstd(M, 1).astype(np.float32)
        df[f"pf_{m}_mean_vs_tw"] = (np.nanmean(M, 1) - np.nanmean(TM, 1)).astype(np.float32)
    df["z_grad"] = col(ZGRAD.get(wid))
    # ==== provenance(その特徴がどう作られたか=妥当性の説明変数, well単位→全行broadcast) ====
    for m in ["tw", "self", "nbr", "nbr_gr5", "self_graft"]:
        d = pv.get(m, {})
        setp(f"prov_{m}_grfit", d.get("grfit")); setp(f"prov_{m}_grcorr", d.get("grcorr"))
    for m in ["nbr", "nbr_gr5"]:
        d = pv.get(m, {})
        setp(f"prov_{m}_dist_min", d.get("dist_min")); setp(f"prov_{m}_dist_mean", d.get("dist_mean"))
        setp(f"prov_{m}_dz_min", d.get("dz_min")); setp(f"prov_{m}_dz_mean", d.get("dz_mean"))
        setp(f"prov_{m}_nref", d.get("nref"))
    setp("prov_nbr_gr5_sim_top1", pv.get("nbr_gr5", {}).get("sim_top1"))
    setp("prov_nbr_gr5_sim_mean", pv.get("nbr_gr5", {}).get("sim_mean"))
    setp("prov_self_cover", pv.get("self", {}).get("cover")); setp("prov_self_nprefix", pv.get("self", {}).get("nprefix"))
    for k in ["a", "b", "valid", "cover", "extrap_frac"]:
        setp(f"prov_self_graft_{k}", pv.get("self_graft", {}).get(k))
    # grfit の相対(どの手法がこの井のGRに最も合うか)
    twf = pv.get("tw", {}).get("grfit")
    for m in ["self", "nbr", "nbr_gr5", "self_graft"]:
        mf = pv.get(m, {}).get("grfit")
        setp(f"prov_{m}_grfit_vs_tw", (mf - twf) if (mf is not None and twf is not None and np.isfinite(mf) and np.isfinite(twf)) else None)
    # legacy 近傍メタ
    df["pf_nbr_has"] = np.float32(mt.get("has", 0)); df["pf_nbr_nndist"] = np.float32(mt.get("nn", np.nan)); df["pf_nbr_nrefs"] = np.float32(mt.get("nref", 0))
    return df


In [ ]:
%%writefile create_v95.py
# -*- coding: utf-8 -*-
"""v95 orchestration: train/test.parquet ビルダー(v93 create を 5表現+provenance に拡張)。

v93 → v95 変更:
  - PF表現を tw/self/nbr の3 → tw/self/nbr/nbr_gr5/self_graft の5表現に拡張。
    * nbr_gr5 = 全train井から GR類似 top5(imputers.neighbors_gr_of) を build_inputs_nbr に渡す(refs差替のみ)。
    * self_graft = build_inputs_self_graft(prefix実測+tw較正外挿)。
  - ★NN-emission 無し(w_nn=0・sim非添付)。物理錨(attach_anchor)は温存(pfAのみ)。逆方向PFも無し。
  - ★provenance(各表現の妥当性説明変数)を well単位で計算し features へ: GR一致度(grfit/grcorr)・
    近傍距離/ΔZ/GR類似度・self被覆・self_graft較正係数。
  - PFパラメータは pf_banks_config.json(v93からコピー=同一)。

出力: v95/artifacts_v95/data/{train,test}.parquet
実行: rogii_claude/.venv/Scripts/python.exe v95/create_v95.py   スモーク: NWELLS=6 python v95/create_v95.py
"""
import os, sys, io, glob, time, pickle
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import pf_banks_v95 as pf
import imputers_v95 as imp
import features_v95 as feats

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
DATA_DIR = pf.DATA_DIR
ART = pf.ART
SPLIT = os.environ.get("SPLIT", "train")
IS_TRAIN = (SPLIT == "train")
OUT_PARQUET = (Path(os.environ["OUT_PARQUET"]) if os.environ.get("OUT_PARQUET")
               else ART / "data" / ("train.parquet" if IS_TRAIN else "test.parquet"))
SEED = 4423098
N_SEED = pf.N_SEED
WELL_CHUNK = 60
N_JOBS = 4
NWELLS = int(os.environ["NWELLS"]) if os.environ.get("NWELLS") else None
DEVICE = pf.DEVICE
BANK_ORDER = pf.BANK_ORDER
METHODS = ["tw", "self", "nbr", "nbr_gr5", "self_graft"]
# ★v102: NN-emission を全バンクに(v96/v100両バンク)。SIM_PKL は orchestrator が指定(v97 sim を流用)。
SIM_PKL = (Path(os.environ["SIM_PKL"]) if os.environ.get("SIM_PKL")
           else pf.ART / ("sim_grfree_v97.pkl" if os.environ.get("SPLIT", "train") == "train" else "sim_grfree_test_v97.pkl"))


def _attach_sim(x, wid, SIMD):
    sd = SIMD.get(wid)
    if sd is not None and len(sd.get("st", [])) == len(x["md"]):
        x["_sim"] = sd["sim"]; x["_st"] = sd["st"]
    return x


AFFINE = os.environ.get("AFFINE", "0") == "1"   # ★affine版特徴生成(照合GRをprefix較正)
if AFFINE:
    import affine_v102 as _aff


def _gr_prefix_sm(hw, P):
    """既知prefix の 平滑GR と TVT_input(provenance grfit 用)。"""
    kn = hw[hw["TVT_input"].notna()]
    fb = float(np.nanmean(hw["GR"])) if np.isfinite(np.nanmean(hw["GR"])) else 0.0
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(fb).to_numpy(float)
    gr_sm = pf._smooth_radius_values(gr_full, fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    return kn["TVT_input"].to_numpy(float), gr_sm[kpos].astype(float)


def _grfit(kn_tvtin, gr_pre, x):
    """参照 gg が prefix GR をどれだけ説明するか: 残差std(grfit)・相関(grcorr)。"""
    if x is None or len(kn_tvtin) < 8:
        return np.nan, np.nan
    gg = x["gg"]; grid = x["gmin"] + np.arange(len(gg)) * x["gst"]
    at = np.interp(kn_tvtin, grid, gg); r = gr_pre - at; m = np.isfinite(r)
    if m.sum() < 8:
        return np.nan, np.nan
    gf = float(np.std(r[m]))
    gc = (float(np.corrcoef(gr_pre[m], at[m])[0, 1])
          if np.std(gr_pre[m]) > 1e-9 and np.std(at[m]) > 1e-9 else np.nan)
    return gf, gc


def build_dataset():
    base = DATA_DIR / SPLIT
    wells = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(base / "*__horizontal_well.csv"))})
    if NWELLS is not None:
        wells = wells[:NWELLS]
    P0 = pf.bank_param(BANK_ORDER[0])
    names = []; paths = []
    for w in wells:
        hp = base / f"{w}__horizontal_well.csv"; tp = base / f"{w}__typewell.csv"
        try:
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
        except Exception:
            continue
        if IS_TRAIN and "TVT" not in hw.columns:
            continue
        if pf.build_smoother_inputs(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), P0) is None:
            continue
        names.append(w); paths.append((hp, tp))
    print(f"[{SPLIT}] 使用坑井: {len(names)}  banks={BANK_ORDER} x 5表現={METHODS}  lag={pf.SMOOTH_LAG} n_seed={N_SEED} (★v102: NN-emission ON 全バンク)")

    FI, DI, _tw = imp.build_imputers()
    feats.set_imputers(FI, DI)
    print(f"[imputers] FI={len(FI.df)}井 / DI点={len(DI.ancc)}")

    FWD_ENS = feats.FWD_ENS; FWD_ENS.clear()
    ENS = {m: {} for m in ["self", "nbr", "nbr_gr5", "self_graft"]}   # tw は FWD_ENS
    ZGRAD = {}; NBR_META = {}; NREFS = {}; NREFS_GR5 = {}; PROV = {}

    # ---- 近傍探索(距離 & GR類似) + provenance ----
    print("[nbr] 近傍探索(距離top3 & GR類似top5) + provenance ...")
    t0 = time.perf_counter(); n_nbr = 0; n_gr5 = 0
    _cw, _cxyz = imp._train_centroids_z(); _cmap = {w: i for i, w in enumerate(_cw)}
    for (hp, tp) in paths:
        wid = os.path.basename(str(hp)).split("__")[0]
        hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
        tvt_tw = tw["TVT"].to_numpy(float); gr_tw = tw["GR"].to_numpy(float)
        cx, cy = float(hw["X"].mean()), float(hw["Y"].mean()); cz = float(hw["Z"].mean())
        exc = wid if SPLIT == "train" else None
        # 距離近傍
        refw = imp.neighbors_of(cx, cy, exclude_wid=exc)
        NREFS[wid] = [imp.ref_grtvt(r) for r, _ in refw] if len(refw) >= imp.NEED_REFS else []
        nbr_dists = [d for _, d in refw]; nbr_dz = []
        for r, _ in refw:
            if r in _cmap:
                nbr_dz.append(abs(_cxyz[_cmap[r], 2] - cz))
        # GR類似近傍
        kn = hw[hw["TVT_input"].notna()]
        ng = imp.neighbors_gr_of(kn["TVT_input"].to_numpy(float), kn["GR"].to_numpy(float), cx, cy, cz, exclude_wid=exc)
        NREFS_GR5[wid] = [imp.ref_grtvt(w) for w, _, _, _ in ng]
        gr5_sim = [s for _, s, _, _ in ng]; gr5_dist = [d for _, _, d, _ in ng]; gr5_dz = [z for _, _, _, z in ng]
        NBR_META[wid] = dict(has=int(len(NREFS[wid]) > 0), nn=imp.nearest_dist(cx, cy, exclude_wid=exc), nref=len(refw))
        if NREFS[wid]:
            n_nbr += 1
        if NREFS_GR5[wid]:
            n_gr5 += 1
        # provenance: 5表現の gg を P0 で作り grfit
        kt, kg = _gr_prefix_sm(hw, P0)
        x_tw = pf.build_smoother_inputs(hw, tvt_tw, gr_tw, P0)
        x_self = pf.build_inputs_self(hw, tvt_tw, gr_tw, P0)
        x_nbr = pf.build_inputs_nbr(hw, tvt_tw, gr_tw, P0, NREFS[wid])
        x_gr5 = pf.build_inputs_nbr(hw, tvt_tw, gr_tw, P0, NREFS_GR5[wid])
        x_sg = pf.build_inputs_self_graft(hw, tvt_tw, gr_tw, P0)
        prov = {}
        for mm, xx in [("tw", x_tw), ("self", x_self), ("nbr", x_nbr), ("nbr_gr5", x_gr5), ("self_graft", x_sg)]:
            gf, gc = _grfit(kt, kg, xx); prov[mm] = dict(grfit=gf, grcorr=gc)
        _cov = float(x_sg["_graft"]["cover"]) if (x_sg is not None and "_graft" in x_sg) else np.nan
        prov["self"].update(cover=_cov, nprefix=int(len(kt)))     # self被覆=prefixがtw格子を覆う割合(self_graftのcoverと同一計算)
        prov["nbr"].update(nref=len(refw),
                           dist_min=(min(nbr_dists) if nbr_dists else np.nan), dist_mean=(float(np.mean(nbr_dists)) if nbr_dists else np.nan),
                           dz_min=(min(nbr_dz) if nbr_dz else np.nan), dz_mean=(float(np.mean(nbr_dz)) if nbr_dz else np.nan))
        prov["nbr_gr5"].update(nref=len(ng),
                               sim_top1=(gr5_sim[0] if gr5_sim else np.nan), sim_mean=(float(np.mean(gr5_sim)) if gr5_sim else np.nan),
                               dist_min=(min(gr5_dist) if gr5_dist else np.nan), dist_mean=(float(np.mean(gr5_dist)) if gr5_dist else np.nan),
                               dz_min=(min(gr5_dz) if gr5_dz else np.nan), dz_mean=(float(np.mean(gr5_dz)) if gr5_dz else np.nan))
        if x_sg is not None and "_graft" in x_sg:
            prov["self_graft"].update(**x_sg["_graft"])
        PROV[wid] = prov
    print(f"      近傍あり {n_nbr}/{len(names)} / GR類似あり {n_gr5}/{len(names)}  ({time.perf_counter()-t0:.1f}s)")

    # ---- 各バンク × 5表現 smooth-PF(★v102: 全バンク NN-emission ON) ----
    if SIM_PKL.exists():
        SIMD = pickle.loads(SIM_PKL.read_bytes())
        print(f"[v102 emission] sim load: {len(SIMD)}井 <- {SIM_PKL.name}(全バンクemission)")
    else:
        SIMD = {}; print(f"[v102 emission] WARN: {SIM_PKL} 無し -> emission OFF")
    PFCACHE = Path(os.environ.get("PF_CACHE_DIR", str(ART / "pfcache"))); PFCACHE.mkdir(parents=True, exist_ok=True)
    FORCE_PF = os.environ.get("FORCE_PF", "0") == "1"

    def run_method(bank, P, builder, store, m):
        # ★v99 PFキャッシュ: per-(split,表現,bank)。後でバンク/特徴を足しても既存PFは再実行しない
        cache = PFCACHE / f"{SPLIT}_{m}_{bank}.pkl"
        if cache.exists() and not FORCE_PF:
            d = pickle.loads(cache.read_bytes())
            if all(w in d for w in names):                     # 現在の井戸集合を完全にカバーする時のみ流用
                for w in names:
                    store[(w, bank)] = d[w]
                print(f"      [cache] {cache.name} 流用({len(names)}井)")
                return
            print(f"      [cache] {cache.name} 井戸不一致({len(d)}井<必要{len(names)}井) -> 再計算")
        inps = []
        for (hp, tp) in paths:
            wid = os.path.basename(str(hp)).split("__")[0]
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
            x = builder(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), wid)
            if AFFINE:
                x = _aff.apply_affine(pf, x, hw, tw["GR"].to_numpy(float), P)   # ★affine較正
            x = pf.attach_anchor(x, wid, P["_physics"])       # 物理錨
            x = _attach_sim(x, wid, SIMD)                      # ★v102: NN-emission 添付
            inps.append(x)
        outs = pf.run_smoother_ext(inps, P, SEED, N_SEED, WELL_CHUNK, w_nn=P["_w_nn"])   # ★v102: emission ON(バンク別w_nn)
        dd = {}
        for i, w in enumerate(names):
            store[(w, bank)] = outs[i]; dd[w] = outs[i]
        cache.write_bytes(pickle.dumps(dd, protocol=4))       # キャッシュ保存
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    for bank in BANK_ORDER:
        P = pf.bank_param(bank)
        builders = {
            "tw":         lambda hw, tt, tg, wid, _P=P: pf.build_smoother_inputs(hw, tt, tg, _P),
            "self":       lambda hw, tt, tg, wid, _P=P: pf.build_inputs_self(hw, tt, tg, _P),
            "nbr":        lambda hw, tt, tg, wid, _P=P: pf.build_inputs_nbr(hw, tt, tg, _P, NREFS[wid]),
            "nbr_gr5":    lambda hw, tt, tg, wid, _P=P: pf.build_inputs_nbr(hw, tt, tg, _P, NREFS_GR5[wid]),
            "self_graft": lambda hw, tt, tg, wid, _P=P: pf.build_inputs_self_graft(hw, tt, tg, _P),
        }
        for m in METHODS:
            t0 = time.perf_counter(); print(f"[smoothPF-{m}] {bank} ...")
            store = FWD_ENS if m == "tw" else ENS[m]
            run_method(bank, P, builders[m], store, m)
            print(f"      done {time.perf_counter()-t0:.1f}s")

    for (hp, tp) in paths:
        wid = os.path.basename(str(hp)).split("__")[0]
        ZGRAD[wid] = pf.z_gradient_eval(pd.read_csv(hp))

    ENS_ALL = {"self": ENS["self"], "nbr": ENS["nbr"], "nbr_gr5": ENS["nbr_gr5"], "self_graft": ENS["self_graft"]}

    def one(w, hp, tp):
        feats.set_current_well(w)
        df = feats.build_well(str(hp), str(tp), IS_TRAIN)
        if df is None or len(df) == 0:
            return None
        return feats.build_self_nbr_columns(df, w, ENS_ALL, ZGRAD, NBR_META, PROV)

    print("[build] build_well(tw注入) + 5表現/provenance 列 ...")
    t0 = time.perf_counter()
    res = Parallel(n_jobs=N_JOBS, prefer="threads", verbose=5)(
        delayed(one)(names[i], paths[i][0], paths[i][1]) for i in range(len(names)))
    parts = [r for r in res if r is not None]
    print(f"      done {time.perf_counter()-t0:.1f}s  parts={len(parts)}")
    if not parts:
        raise RuntimeError("有効な坑井がありません。")
    return pd.concat(parts, ignore_index=True)


def main():
    print(f"[v95] device={DEVICE} smooth_lag={pf.SMOOTH_LAG} n_seed={N_SEED} chunk={WELL_CHUNK} NWELLS={NWELLS} (NN-emission/逆方向 なし)")
    df = build_dataset()
    if SPLIT == "train" and NWELLS is None and df["well"].nunique() < 700:
        raise SystemExit(f"[GUARD] 出力坑井数 {df['well'].nunique()} 本は異常。")
    if os.environ.get("PF_ONLY_OUT", "0") == "1":         # ★dec10/affine: 共通の非PF特徴は base v102 と同一=出力不要。pf_候補のみ残す
        meta = [c for c in ["id", "well", "target", "last_known_tvt", "md_since"] if c in df.columns]
        keep = meta + [c for c in df.columns if c.startswith("pf_")]
        df = df[keep]
        print(f"[PF_ONLY_OUT] pf_候補のみ出力({len(keep)}列 = meta{len(meta)} + pf_{len(keep)-len(meta)})", flush=True)
    OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(OUT_PARQUET, index=False)
    cols = list(df.columns)
    print(f"\nsaved: {OUT_PARQUET}  rows={len(df)} cols={len(cols)} wells={df['well'].nunique()}")
    for pre in ["pf_self", "pf_nbr", "pf_nbr_gr5", "pf_self_graft", "prov_", "pf_"]:
        print(f"  {pre}* = {sum(c.startswith(pre) for c in cols)} 列")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile pf_banks_v97.py
# -*- coding: utf-8 -*-
"""v93 module1: 6バンク平滑PFエンジン(GPU固定ラグ smoother)。

v52 create の smooth-PF 部分をクリーン再構築。★v93 の唯一の実変更:
  - pfA バンクの錨を「過去struct(v38/v66)」から「GRフリー錨(v91 OOF, sig=9ft較正)」へ差替。
  - 錨強度 anchor_mult は global 定数でなく バンクparam P["anchor_mult"] から取る(pfA=20)。
過去struct は一切読まない([[no-struct-directive]])。錨源は v93/artifacts_v93 のみ。

構成(v52 と数値一致させる要素):
  - build_smoother_inputs / build_inputs_self / build_inputs_nbr : PF入力(tw/self/nbr疑似typewell)
  - attach_anchor : GRフリー錨(anc/ancs)を添付。非physicsバンクは ancs=1e9(錨OFF)
  - _smoother_core : 固定ラグ(L=smooth_lag)平滑PF。GR尤度×phys×錨×NN-emission
  - run_smoother_ext : seed尤度加重で smoothed平均/std を返す
NN-emission(sim/帯中心 st)は create 側が入力dictに _sim/_st を添付する形で受ける(module2が生成)。
"""
import os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
DATA_DIR = Path(os.environ.get("ROGII_DATA", str(PROJ / "rogii-wellbore-geology-prediction")))
ART = Path(os.environ.get("ROGII_ART97", str(PROJ / "v97" / "artifacts_v97")))  # ★v95: config/錨をv93からコピー済(PF param同一)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

# ---- config(6バンクparam) & GRフリー錨 ----
_CFG = json.loads((ART / "pf_banks_config.json").read_text(encoding="utf-8"))
BANK_ORDER = _CFG["bank_order"]                    # [pf_1,pf_2,pf_3,r0_seed32,r1_seed32,pfA]
PHYSICS_BANKS = set(_CFG["physics_banks"])         # {pfA}
BANK_PARAMS = _CFG["params"]                       # dict[bank] -> raw param dict
W_NN_BANK = _CFG["w_nn_bank"]                       # {pfA:0.01}
W_NN_DEFAULT = float(_CFG["w_nn_default"])          # 0.02
SMOOTH_LAG = int(_CFG["smooth_lag"])               # 16
SMOOTH_MODE_CFG = str(_CFG.get("smooth_mode", "fixedlag"))   # ★v99: full平滑切替
N_SEED = int(_CFG["n_seed"])                        # 32
# ★v102: ps_combo(seed集約を尤度×錨カーネルに=選択強化)。env で全体上書き可、off_banksは無効。
PS_COMBO_TAU = float(os.environ.get("PS_COMBO_TAU", _CFG.get("ps_combo_tau", 0.0)))
PS_COMBO_OFF = set(_CFG.get("ps_combo_off_banks", []))

_ANCHOR_PKL = ART / _CFG.get("anchor_pkl", "grfree_anchor_train.pkl")
if os.environ.get("V93_ANCHOR_PKL"):
    _ANCHOR_PKL = Path(os.environ["V93_ANCHOR_PKL"])
_ANCHOR = pickle.loads(_ANCHOR_PKL.read_bytes())   # {wid: {tvt, sig}}
print(f"[v93 pf_banks] 錨={_ANCHOR_PKL.name} {len(_ANCHOR)}井 / banks={BANK_ORDER} / smooth_lag={SMOOTH_LAG}")

# ---- NN-emission 帯(create/module2と共有する幾何定数) ----
SIM_STEP_NN = 0.5
NBAND_NN = 90                                       # ±45ft / 0.5ft = 90

# ---- self/nbr/Z傾斜 定数(v52同値) ----
SELF_MIX_W = 1.0
NBR_MIX_W = 1.0
ZGRAD_R = 25
K_MAX = 3
MAX_DIST = 1500.0
NEED_REFS = 2


def bank_param(bank):
    """バンク生paramに physics/w_nn を付けて返す(create が smoother に渡す P)。"""
    P = dict(BANK_PARAMS[bank])
    P["name"] = bank
    P.setdefault("smooth_lag", SMOOTH_LAG)
    P.setdefault("smooth_mode", SMOOTH_MODE_CFG)
    P["_physics"] = bank in PHYSICS_BANKS
    P["_w_nn"] = float(W_NN_BANK.get(bank, W_NN_DEFAULT))
    P["_ps_combo_tau"] = 0.0 if bank in PS_COMBO_OFF else PS_COMBO_TAU   # ★v102 選択強化(off_banksは無効)
    return P


def _ps_combo_reweight(ww, ps_jT, st, tau):
    """★v102 ps_combo: seed尤度重み ww を「錨に近いseedを重く」再加重(選択強化)。st無/tau0はそのまま。"""
    if tau <= 0 or st is None:
        return ww
    T = ps_jT.shape[1]; stj = np.asarray(st, float)
    if len(stj) < T:
        return ww
    da = ((ps_jT - stj[:T][None, :]) ** 2).mean(1)
    w2 = ww * np.exp(-(da - da.min()) / (2.0 * tau * tau)); s = w2.sum()
    return w2 / s if s > 1e-300 else np.full(len(ww), 1.0 / len(ww))


# ==================== 小物(v52 と同一) ====================
def _smooth_radius_values(vals, fb, r):
    r = int(r)
    s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    if r <= 0:
        return s.to_numpy(np.float32)
    return s.rolling(2 * r + 1, center=True, min_periods=1).mean().to_numpy(np.float32)


def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)


def _bin_grid(t, g, gmin, step, G):
    """(TVT, GR) を grid の TVT ビンに median 集約(疑似typewell)。無データビン=NaN。"""
    m = np.isfinite(t) & np.isfinite(g); t, g = t[m], g[m]
    idx = np.round((t - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, gg = idx[ok], g[ok]
    out = np.full(G, np.nan)
    if len(gg):
        s = pd.Series(gg).groupby(idx).median(); out[s.index.to_numpy()] = s.to_numpy()
    return out


# ==================== PF入力(tw / self / nbr) ====================
def build_smoother_inputs(hw, tw_tvt, tw_gr, P):
    """smooth PF 入力を hw/tw と生PFパラメータから作る。eval無しは None。"""
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    tw_gr_pf = _smooth_radius_values(tw_gr, tw_fb, P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(tw_tvt, tw_gr_pf)
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    ev_pos = hw.index.get_indexer(ev.index); kpos = hw.index.get_indexer(kn.index)
    kn_tvtin = kn["TVT_input"].to_numpy(float)
    if len(kpos) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[kpos] - np.interp(kn_tvtin, tw_tvt, tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    gs = gs * float(os.environ.get("GS_SCALE", "1.0"))     # ★post-clip の GR尤度σ 広げ(gs×1.30 検証。既定1.0=無効)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    return dict(md=ev["MD"].to_numpy(float), z=ev["Z"].to_numpy(float), gr=gr_sm[ev_pos].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=ls, ir=ir, N=int(P["n_particles"]))


def build_inputs_self(hw, tw_tvt, tw_gr, P):
    """tw入力の gg を、自分の既知prefix GR(TVT_input) 疑似typewell で上書き。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None:
        return None
    kn = hw[hw["TVT_input"].notna()]
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    gh = _bin_grid(kn["TVT_input"].to_numpy(float), gr_sm[kpos].astype(float), base["gmin"], base["gst"], len(base["gg"]))
    gg = base["gg"].copy(); have = np.isfinite(gh)
    gg[have] = SELF_MIX_W * gh[have] + (1 - SELF_MIX_W) * gg[have]
    base = dict(base); base["gg"] = gg
    return base


def build_inputs_nbr(hw, tw_tvt, tw_gr, P, refs):
    """tw入力の gg を、近傍train坑井の GR(TVT) 疑似typewell で上書き。refs無し=tw fallback。
       refs = [(gr_array, tvt_array), ...](module3 の近傍選択が供給)。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None:
        return None
    if refs:
        rt = np.concatenate([r[1] for r in refs])
        rg = np.concatenate([_smooth_radius_values(r[0], float(np.nanmean(r[0])), P["hgr_smooth_r"]).astype(float) for r in refs])
        gh = _bin_grid(rt, rg, base["gmin"], base["gst"], len(base["gg"]))
        gg = base["gg"].copy(); have = np.isfinite(gh)
        gg[have] = NBR_MIX_W * gh[have] + (1 - NBR_MIX_W) * gg[have]
        base = dict(base); base["gg"] = gg
    return base


def build_inputs_self_graft(hw, tw_tvt, tw_gr, P):
    """★v95新規: self(自分のprefix GR)を tw格子に substitute し、prefix範囲外(データ無しビン)を
       tw を self較正(self≈a·tw+b, a∈[0.2,5])で外挿。self被覆を広げる(v49 build_inputs_self_ext 由来)。
       較正係数 a,b・有効フラグ・外挿割合を base['_graft'] にメタ格納(provenance用)。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None:
        return None
    kn = hw[hw["TVT_input"].notna()]
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    gh = _bin_grid(kn["TVT_input"].to_numpy(float), gr_sm[kpos].astype(float), base["gmin"], base["gst"], len(base["gg"]))
    gg = base["gg"].copy(); have = np.isfinite(gh)
    a, b, valid = 1.0, 0.0, 0
    if have.sum() >= 8:
        gg[have] = gh[have]                                       # prefix範囲=self実測
        tw_ov = base["gg"][have]; self_ov = gh[have]
        A = np.c_[tw_ov, np.ones(len(tw_ov))]
        try:
            coef, *_ = np.linalg.lstsq(A, self_ov, rcond=None); a, b = float(coef[0]), float(coef[1])
        except Exception:
            a, b = 1.0, 0.0
        if 0.2 <= a <= 5.0 and np.isfinite(a) and np.isfinite(b):
            gg[~have] = a * base["gg"][~have] + b                 # 範囲外=self較正したtwで外挿
            valid = 1
    base = dict(base); base["gg"] = gg
    base["_graft"] = dict(a=float(a), b=float(b), valid=int(valid),
                          cover=float(have.mean()), extrap_frac=float((~have).mean()), nprefix=int(len(kpos)))
    return base


def z_gradient_eval(hw, r=ZGRAD_R):
    """Z傾斜 dZ/dMD を移動平均(断層影響回避)し eval 行のみ返す。"""
    ev = hw["TVT_input"].isna().to_numpy()
    md = hw["MD"].to_numpy(float); z = hw["Z"].to_numpy(float)
    with np.errstate(all="ignore"):
        g = np.gradient(z, md)
    g = pd.Series(g).rolling(2 * int(r) + 1, center=True, min_periods=1).mean().to_numpy()
    return g[ev].astype(np.float32)


# ==================== GRフリー錨 添付 ====================
def attach_anchor(x, wid, physics):
    """PF入力に GRフリー錨(anc/ancs)を添付。physics=False / 錨欠損 / 行数不一致 は錨OFF(ancs=1e9)。
       ★過去struct は使わない。錨源は _ANCHOR(v93 GRフリー, sig=9ft較正)のみ。"""
    n = len(x["md"]); sp = _ANCHOR.get(wid)
    x["_wid"] = wid
    if physics and sp is not None and len(sp["tvt"]) == n:
        x["anc"] = np.asarray(sp["tvt"], float); x["ancs"] = np.asarray(sp["sig"], float)
    else:
        x["anc"] = np.zeros(n); x["ancs"] = np.full(n, 1e9)
    return x


# ==================== GPU 固定ラグ 平滑PF 本体 ====================
def _smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir,
                   anc, ancs, amul, stq, sim, wmap, w_nn, P, N, device, gen, dtype=DTYPE):
    """v52 _v26_smoother_core と数値一致。★v93変更: 錨強度は amul(=P['anchor_mult'])。"""
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    NU = float(P.get("robust_nu", 0.0))                # ★v97: Student-t頑健尤度(0=Gaussian)
    BETA = float(P.get("temper_beta", 1.0))            # ★v97: tempering(1=無効, <1=軟化)

    def rn():
        return torch.randn((B, N), generator=gen, device=device, dtype=dtype)

    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + RN * rn()
        if RATE_CLIP > 0:
            rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        if NU > 0.0:                                   # ★v97: Student-t 頑健尤度
            lk = (1.0 + dp / NU) ** (-(NU + 1.0) * 0.5)
        else:
            lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero)
        if BETA != 1.0:                                # ★v97: tempering(尤度軟化)
            lk = lk ** BETA
        lk = lk.clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate - dipp) / PHYS_SIG; lkp = torch.exp(-0.5 * dphys * dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        # ★v93: GRフリー行錨(mult = amul = P['anchor_mult'])。非physicsは ancs=1e9 で無効化。
        asig = (ancs[:, i][:, None] * amul).clamp_min(1e-6)
        da = (v - anc[:, i][:, None]) / asig
        lka = torch.where(da * da < 1200.0, torch.exp(-0.5 * da * da), zero).clamp_min(1e-300)
        # NN-emission(学習類似度を温度 w_nn で乗算)。帯中心 stq は GRフリー錨tvt(module2)。
        if w_nn > 0:
            jj = torch.round((v - stq[:, i][:, None]) / SIM_STEP_NN).long() + NBAND_NN
            inb = (jj >= 0) & (jj <= 2 * NBAND_NN)
            sim_i = sim[:, i, :].to(dtype)[wmap]
            sv = torch.gather(sim_i, 1, jj.clamp(0, 2 * NBAND_NN))
            sv = torch.where(inb, sv, torch.full_like(sv, -1.0))
            lka = lka * torch.exp(w_nn * sv)
        w_new = torch.where(obs[:, None], w * lk * lkp * lka, w * lkp * lka)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0:
                rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w * (pos - zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i - L] = (w * old).sum(1) - z[:, i - L]
            buf[:, ptr, :] = pos; ptr = (ptr + 1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T - L), T):
            pts_s[:, j] = (w * buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik

def _smoother_core_full(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir,
                        anc, ancs, amul, stq, sim, wmap, w_nn, P, N, device, gen, dtype=DTYPE):
    """★v99 full平滑: forward は _smoother_core と数値一致。固定ラグbufの代わりに全履歴(pos fp32=錨delta / anc int16)を
       保存し、単一backward祖先sweep(最終重み)で全区間平滑 pts_s を返す。robust/temper/phys/anchor/NN-emission 全対応。"""
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    NU = float(P.get("robust_nu", 0.0)); BETA = float(P.get("temper_beta", 1.0))

    def rn():
        return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pos_hist = torch.empty((T, B, N), device=device, dtype=dtype)          # fp32(錨lsからのdelta)
    anc_hist = torch.empty((T, B, N), device=device, dtype=torch.int16)     # 祖先index
    lsr = ls[:, None]
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    arL = torch.arange(N, device=device, dtype=torch.long)[None, :].expand(B, N)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + RN * rn()
        if RATE_CLIP > 0:
            rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        if NU > 0.0:
            lk = (1.0 + dp / NU) ** (-(NU + 1.0) * 0.5)
        else:
            lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero)
        if BETA != 1.0:
            lk = lk ** BETA
        lk = lk.clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate - dipp) / PHYS_SIG; lkp = torch.exp(-0.5 * dphys * dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        asig = (ancs[:, i][:, None] * amul).clamp_min(1e-6)
        da = (v - anc[:, i][:, None]) / asig
        lka = torch.where(da * da < 1200.0, torch.exp(-0.5 * da * da), zero).clamp_min(1e-300)
        if w_nn > 0:
            jj = torch.round((v - stq[:, i][:, None]) / SIM_STEP_NN).long() + NBAND_NN
            inb = (jj >= 0) & (jj <= 2 * NBAND_NN)
            sim_i = sim[:, i, :].to(dtype)[wmap]
            sv = torch.gather(sim_i, 1, jj.clamp(0, 2 * NBAND_NN))
            sv = torch.where(inb, sv, torch.full_like(sv, -1.0))
            lka = lka * torch.exp(w_nn * sv)
        w_new = torch.where(obs[:, None], w * lk * lkp * lka, w * lkp * lka)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        anc_step = arL
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0:
                rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
            anc_step = torch.where(nm, idx, arL)
        pts_f[:, i] = (w * (pos - zi)).sum(1)
        pos_hist[i] = pos - lsr; anc_hist[i] = anc_step.to(torch.int16)
        pm = torch.where(act, cur, pm)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype); a = arL.clone(); wfin = w
    for i in range(T - 1, -1, -1):
        pts_s[:, i] = (wfin * (torch.gather(pos_hist[i], 1, a).to(dtype) + lsr)).sum(1) - z[:, i]
        a = torch.gather(anc_hist[i], 1, a).long()
    del pos_hist, anc_hist
    return pts_f, pts_s, log_lik


def _pad(inps, device, dtype=DTYPE):
    """可変長wellをバッチテンソルへ。anc/ancs/stq/sim(NN)も詰める。sim帯は入力dictの _sim/_st から。"""
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    anc = torch.zeros((W, Tmax), dtype=dtype); ancs = torch.full((W, Tmax), 1e9, dtype=dtype)
    stq = torch.zeros((W, Tmax), dtype=dtype)
    simt = torch.full((W, Tmax, 2 * NBAND_NN + 1), -1.0, dtype=torch.float16)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        _sim = x.get("_sim"); _st = x.get("_st")
        if _sim is not None and _st is not None:
            _Ts = min(Tn, len(_st))
            stq[b, :_Ts] = torch.from_numpy(np.asarray(_st[:_Ts], np.float32))
            if _Ts < Tmax:
                stq[b, _Ts:] = stq[b, _Ts - 1]
            simt[b, :_Ts] = torch.from_numpy(np.asarray(_sim[:_Ts], np.float16))
        if "anc" in x:
            anc[b, :Tn] = torch.from_numpy(x["anc"].astype("float32"))
            if Tn < Tmax:
                anc[b, Tn:] = anc[b, Tn - 1]
            ancs[b, :Tn] = torch.from_numpy(x["ancs"].astype("float32"))
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax:
            md[b, Tn:] = md[b, Tn - 1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax:
            grid[b, G:] = grid[b, G - 1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir), anc=to(anc), ancs=to(ancs),
                stq=to(stq), simt=to(simt))




def _pad(inps, device, dtype=DTYPE):
    """可変長wellをバッチテンソルへ。anc/ancs/stq/sim(NN)も詰める。sim帯は入力dictの _sim/_st から。"""
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    anc = torch.zeros((W, Tmax), dtype=dtype); ancs = torch.full((W, Tmax), 1e9, dtype=dtype)
    stq = torch.zeros((W, Tmax), dtype=dtype)
    simt = torch.full((W, Tmax, 2 * NBAND_NN + 1), -1.0, dtype=torch.float16)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        _sim = x.get("_sim"); _st = x.get("_st")
        if _sim is not None and _st is not None:
            _Ts = min(Tn, len(_st))
            stq[b, :_Ts] = torch.from_numpy(np.asarray(_st[:_Ts], np.float32))
            if _Ts < Tmax:
                stq[b, _Ts:] = stq[b, _Ts - 1]
            simt[b, :_Ts] = torch.from_numpy(np.asarray(_sim[:_Ts], np.float16))
        if "anc" in x:
            anc[b, :Tn] = torch.from_numpy(x["anc"].astype("float32"))
            if Tn < Tmax:
                anc[b, Tn:] = anc[b, Tn - 1]
            ancs[b, :Tn] = torch.from_numpy(x["ancs"].astype("float32"))
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax:
            md[b, Tn:] = md[b, Tn - 1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax:
            grid[b, G:] = grid[b, G - 1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir), anc=to(anc), ancs=to(ancs),
                stq=to(stq), simt=to(simt))


def _pf_devices(chunk_env_default):
    """PF実行デバイス一覧を決定。
       PF_SIM_NGPU=N(>=2): 1GPU上でN論理分割=分割/結合/スレッド機構の検証用(全て同一物理GPU)。
       PF_NGPU=N(>=2): 実GPUをN枚使用(Kaggle 2×T4)。既定=[DEVICE]で現行完全不変。"""
    sim_ng = int(os.environ.get("PF_SIM_NGPU", "0"))
    if sim_ng >= 2:
        return [DEVICE] * sim_ng, True
    ng = int(os.environ.get("PF_NGPU", "1"))
    if ng >= 2 and torch.cuda.is_available() and torch.cuda.device_count() >= 2:
        return ["cuda:%d" % i for i in range(min(ng, torch.cuda.device_count()))], False
    return [DEVICE], False


def run_smoother_ext(inps, P, seed, n_seeds, chunk, w_nn=0.0, capture=False):
    """各 well: smoothed(seed尤度加重平均)と std(seed間加重std)を返す。
       ★v93: 錨強度 amul は P['anchor_mult'](pfA=20)。非physicsバンクは ancs=1e9 で錨無効。
       ★capture=True: v34モードゲート用に per-seed 平滑ps(S,T)/forward pfw/ll/ww も out に付与(RAM増)。
       ★マルチGPU: whole-chunkを各デバイスへラウンドロビン分配しスレッド並列。seedはchunk毎リセット・
         torch CUDA RNGはデバイス非依存なので、同一chunk境界なら単一GPUと(浮動小数の非決定性を除き)一致。"""
    import threading
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); out = [None] * W
    ls_scale = float(P["likelihood_scale"]); amul = float(P.get("anchor_mult", 1.0))
    ch_env = os.environ.get("PF_WELL_CHUNK")
    if ch_env:
        chunk = int(ch_env)
    # ★v99 full平滑モード: 固定ラグの代わりに全区間系譜平滑。可変chunk(VRAM予算)・長さソート。
    SMOOTH_MODE = str(P.get("smooth_mode", os.environ.get("SMOOTH_MODE", "fixedlag")))
    if SMOOTH_MODE == "full":
        budget = float(os.environ.get("FULL_VRAM_GB", "8.0")) * 1e9
        order = sorted(range(W), key=lambda i: len(inps[i]["md"]))
        chunks = []; p = 0                                       # ★可変chunkを全部先に構築(デバイス分配用)
        while p < W:
            Tmax = len(inps[order[p]]["md"]); nw = 0
            while p + nw < W and nw < 64:
                tmx = max(Tmax, len(inps[order[p + nw]]["md"]))
                if (nw + 1) * S * tmx * N * 6 > budget and nw >= 1:
                    break
                Tmax = tmx; nw += 1
            chunks.append(order[p:p + nw]); p += nw

        def _proc_full(sel, device):                            # ★1可変chunkを指定deviceで処理(seedはchunk毎リセット=device非依存)
            sub = [inps[j] for j in sel]; nw = len(sel); pad = _pad(sub, device)
            rep = lambda tt: tt.repeat_interleave(S, 0)
            gen = torch.Generator(device=device); gen.manual_seed(seed)
            _wmap = torch.arange(nw, device=device).repeat_interleave(S)
            pf, ps, ll = _smoother_core_full(
                rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
                pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
                pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
                rep(pad["anc"]), rep(pad["ancs"]), amul, rep(pad["stq"]), pad["simt"], _wmap, w_nn, P, N, device, gen)
            ps = ps.view(nw, S, -1).double().cpu().numpy(); pf = pf.view(nw, S, -1).double().cpu().numpy(); ll = ll.view(nw, S).cpu().numpy()
            for j in range(nw):
                gi = sel[j]; T = len(sub[j]["md"]); llw = ll[j]
                if np.isfinite(llw).any():
                    mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw - mx, -np.inf)
                    wwv = np.exp(lk / max(ls_scale, 1e-6)); s = wwv.sum(); wwv = wwv / s if s > 1e-300 else np.full(S, 1.0 / S)
                else:
                    wwv = np.full(S, 1.0 / S)
                wwv = _ps_combo_reweight(wwv, ps[j, :, :T], sub[j].get("_st"), float(P.get("_ps_combo_tau", 0.0)))
                smean = (wwv[:, None] * ps[j, :, :T]).sum(0)
                sstd = np.sqrt(np.maximum((wwv[:, None] * (ps[j, :, :T] - smean[None, :]) ** 2).sum(0), 0.0))
                nobs = int(np.isfinite(sub[j]["gr"][:T]).sum())
                llbest = float(np.nanmax(llw[np.isfinite(llw)])) if np.isfinite(llw).any() else np.nan
                out[gi] = dict(mean=smean.astype(np.float32), std=sstd.astype(np.float32), loglik=np.float32(llbest / max(nobs, 1)))
                if capture:
                    out[gi].update(ps=ps[j, :, :T].astype(np.float32), pfw=pf[j, :, :T].astype(np.float32),
                                   ll=llw.astype(np.float64), ww=wwv.astype(np.float64))

        fdevices, fsim = _pf_devices(chunk)                     # ★PF_NGPU>=2で実GPU複数(既定=[DEVICE]で単GPU=現行不変)
        if len(fdevices) <= 1:
            for sel in chunks:
                _proc_full(sel, fdevices[0])
        else:
            assign = {i: [] for i in range(len(fdevices))}      # 可変chunkをデバイスへラウンドロビン
            for i, sel in enumerate(chunks):
                assign[i % len(fdevices)].append(sel)
            errs = []
            def _wf(di):
                try:
                    for sel in assign[di]:
                        _proc_full(sel, fdevices[di])
                except Exception as e:
                    errs.append(e)
            ths = [threading.Thread(target=_wf, args=(di,)) for di in range(len(fdevices))]
            for t in ths: t.start()
            for t in ths: t.join()
            if errs:
                raise errs[0]
            print(f"[fullPF-multiGPU] devices={fdevices} sim={fsim} chunks={len(chunks)} wells={W}", flush=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return out

    devices, sim = _pf_devices(chunk)

    def _proc(c0, device):
        sub = inps[c0:c0 + chunk]; w = len(sub); pad = _pad(sub, device)
        rep = lambda tt: tt.repeat_interleave(S, 0)
        gen = torch.Generator(device=device); gen.manual_seed(seed)
        _wmap = torch.arange(w, device=device).repeat_interleave(S)
        pf, ps, ll = _smoother_core(
            rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
            pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
            pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
            rep(pad["anc"]), rep(pad["ancs"]), amul, rep(pad["stq"]), pad["simt"], _wmap, w_nn, P, N, device, gen)
        pf = pf.view(w, S, -1).double().cpu().numpy(); ps = ps.view(w, S, -1).double().cpu().numpy()
        ll = ll.view(w, S).cpu().numpy()
        for j in range(w):
            T = len(sub[j]["md"]); llw = ll[j]
            if np.isfinite(llw).any():
                mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw - mx, -np.inf)
                ww = np.exp(lk / max(ls_scale, 1e-6)); s = ww.sum()
                ww = ww / s if s > 1e-300 else np.full(S, 1.0 / S)
            else:
                ww = np.full(S, 1.0 / S)
            ww = _ps_combo_reweight(ww, ps[j, :, :T], sub[j].get("_st"), float(P.get("_ps_combo_tau", 0.0)))
            smean = (ww[:, None] * ps[j, :, :T]).sum(0)
            sstd = np.sqrt(np.maximum((ww[:, None] * (ps[j, :, :T] - smean[None, :]) ** 2).sum(0), 0.0))
            nobs = int(np.isfinite(sub[j]["gr"][:T]).sum())      # ★GR観測行数(loglik正規化)
            llbest = float(np.nanmax(llw[np.isfinite(llw)])) if np.isfinite(llw).any() else np.nan
            out[c0 + j] = dict(mean=smean.astype(np.float32), std=sstd.astype(np.float32),
                               loglik=np.float32(llbest / max(nobs, 1)))   # per-row平均対数尤度=その表現PFのGR適合
            if capture:                                                    # ★v34ゲート: per-seed 生軌跡
                out[c0 + j].update(ps=ps[j, :, :T].astype(np.float32), pfw=pf[j, :, :T].astype(np.float32),
                                   ll=llw.astype(np.float64), ww=ww.astype(np.float64))

    starts = list(range(0, W, chunk))
    if len(devices) <= 1:
        for c0 in starts:                                        # ★単一GPU経路=現行と完全同一(検証可能)
            _proc(c0, devices[0])
    else:
        assign = {i: [] for i in range(len(devices))}            # chunkをデバイスへラウンドロビン割当
        for i, c0 in enumerate(starts):
            assign[i % len(devices)].append(c0)
        errs = []
        def _worker(di):
            try:
                for c0 in assign[di]:
                    _proc(c0, devices[di])
            except Exception as e:
                errs.append(e)
        ths = [threading.Thread(target=_worker, args=(di,)) for di in range(len(devices))]
        for t in ths: t.start()
        for t in ths: t.join()
        if errs:
            raise errs[0]
        print(f"[PF-multiGPU] devices={devices} sim={sim} chunks={len(starts)} wells={W}", flush=True)
    if torch.cuda.is_available():
        for d in set(devices):
            if str(d).startswith("cuda"):
                torch.cuda.synchronize(d)
    return out


# ==================== 単体検証(python pf_banks_v93.py) ====================
if __name__ == "__main__":
    import glob
    TR = DATA_DIR / "train"
    wids = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(TR / "*__horizontal_well.csv"))})
    wids = wids[:12]
    for bank in ["r0_seed32", "pfA"]:
        P = bank_param(bank)
        inps, trues = [], []
        for w in wids:
            hw = pd.read_csv(TR / f"{w}__horizontal_well.csv")
            tw = pd.read_csv(TR / f"{w}__typewell.csv").sort_values("TVT")
            tt = tw["TVT"].to_numpy(float); tg = tw["GR"].to_numpy(float)
            m = np.isfinite(tt) & np.isfinite(tg)
            if m.sum() < 8:
                continue
            x = build_smoother_inputs(hw, tt[m], tg[m], P)
            if x is None:
                continue
            attach_anchor(x, w, P["_physics"])
            ev = hw[hw["TVT_input"].isna()]
            inps.append(x); trues.append(ev["TVT"].to_numpy(float))
        res = run_smoother_ext(inps, P, seed=99999, n_seeds=16, chunk=8, w_nn=0.0)
        errs = np.concatenate([res[k]["mean"] - trues[k] for k in range(len(inps))])
        rmse = float(np.sqrt(np.mean(errs[np.isfinite(errs)] ** 2)))
        print(f"  [{bank}] {len(inps)}井 smooth-PF(w_nn=0) overall RMSE = {rmse:.3f}"
              f"  (physics={P['_physics']}, anchor_mult={P.get('anchor_mult')})")


In [ ]:
%%writefile imputers_v97.py
# -*- coding: utf-8 -*-
"""v93 module3: 空間imputer(formation-plane / dense-ANCC)と近傍(train)プール選択。

v52 create の空間KNN部分をクリーン再構築。全て train CSV から自己完結で構築(cross-version無し)。
  - FormationPlaneKNN : 各train井のXY重心+層median → 局所平面fit(XY→層深)。self除外(leave-self-out)。
  - DenseANCCImputer  : train井のXY-ANCCを密サンプル → KNN加重。self除外。
  - 近傍プール(neighbors_of): XY重心距離<=MAX_DIST の最近傍<=K_MAX 井(self/nbr疑似typewell用)。
※ 層列(ANCC..BUDA)は train のみ存在するが、imputer は train から prior を作り query井の XY だけで引くため
  test でも安全(test井の層列は不要)。[[kaggle-submit-pitfalls]] の test既知列制約を満たす。
"""
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
DATA_DIR = Path(os.environ.get("ROGII_DATA", str(PROJ / "rogii-wellbore-geology-prediction")))

# ==================== ★train参照構造キャッシュ(全プロセスで1回だけ構築) ====================
# submit は create/build_forward を複数プロセスで起動し、各々が 773 train井から FI/DI/GR-TVT/重心を
# ゼロ再構築していた(setup が数プロセス分 重複=大きな固定コスト)。IMP_CACHE=<pkl> があれば1回構築を全プロセスで共有。
_CACHE = "unset"
def _cache():
    global _CACHE
    if _CACHE == "unset":
        _CACHE = None
        cp = os.environ.get("IMP_CACHE")
        if cp and os.path.exists(cp):
            try:
                import pickle
                _CACHE = pickle.loads(Path(cp).read_bytes())
            except Exception as e:
                print(f"[imp_cache] load 失敗({e})-> 再構築フォールバック", flush=True); _CACHE = None
    return _CACHE

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10
DENSE_SPW = 60
DENSE_K = 20
K_MAX = 3
MAX_DIST = 1500.0
NEED_REFS = 2


class FormationPlaneKNN:
    """XY→層深 の局所平面fit(近傍K井の逆距離加重)。self除外可。"""

    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f"{wid}__horizontal_well.csv"
            try:
                df = pd.read_csv(p, usecols=["X", "Y"] + FORMATIONS).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            row = {"wid": wid, "x": float(df["X"].median()), "y": float(df["Y"].median())}
            for c in FORMATIONS:
                row[f"{c}_m"] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df["wid"])}
        xy = self.df[["x", "y"]].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1.0, xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df["x"].to_numpy(); self.ya = self.df["y"].to_numpy()
        self.fa = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap:
            dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1.0 / (dk + 1e-3), 0.0).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1), (w[:, :, None] * fn).sum(1)], 1)
        try:
            coef = np.linalg.solve(A, rhs)
        except Exception:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try:
                    coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except Exception:
                    pass
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] + Yq[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)


class DenseANCCImputer:
    """train井のXY-ANCCを密サンプルしKNN加重推定。self除外可。"""

    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f"{wid}__horizontal_well.csv"
            try:
                df = pd.read_csv(p, usecols=["X", "Y", "ANCC"]).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s["X"].values); ys.append(s["Y"].values)
            anccs.append(s["ANCC"].values); wids.extend([wid] * len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32); self.wids = np.array(wids)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1.0, self.xy.std(0))
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q / self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid:
            dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1.0 / (dk + 1e-3), 0.0)
        sw = w.sum(1); safe = np.where(sw < 1e-9, 1.0, sw); an = self.ancc[ik]
        ap = (an * w).sum(1) / safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
        return (ap.astype(np.float32), np.sqrt(np.maximum(var, 0.0)).astype(np.float32),
                np.where(vk, dk, np.inf).min(1).astype(np.float32))


# ==================== 近傍(train)プール: XY重心 ====================
_TRAIN_CENT = None
_REFC = {}


def _train_centroids():
    global _TRAIN_CENT
    if _TRAIN_CENT is None:
        c = _cache()
        if c is not None:
            _TRAIN_CENT = c["cent"]; return _TRAIN_CENT
        tdir = DATA_DIR / "train"
        wids = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(tdir / "*__horizontal_well.csv"))})
        xy = []
        for w in wids:
            h = pd.read_csv(tdir / f"{w}__horizontal_well.csv", usecols=["X", "Y"])
            xy.append((float(h["X"].mean()), float(h["Y"].mean())))
        _TRAIN_CENT = (wids, np.array(xy))
    return _TRAIN_CENT


def ref_grtvt(wid):
    """近傍井の (GR, TVT)。疑似typewell の材料。"""
    if wid not in _REFC:
        h = pd.read_csv(DATA_DIR / "train" / f"{wid}__horizontal_well.csv", usecols=["GR", "TVT"])
        _REFC[wid] = (h["GR"].to_numpy(float), h["TVT"].to_numpy(float))
    return _REFC[wid]


def nearest_dist(cx, cy, exclude_wid=None):
    """最近傍train井までの実距離(孤立度)。"""
    wids, xy = _train_centroids(); d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy)
    if exclude_wid is not None:
        keep = np.array([w != exclude_wid for w in wids]); d = d[keep]
    return float(d.min()) if len(d) else float("nan")


def neighbors_of(cx, cy, exclude_wid=None):
    """XY重心距離<=MAX_DIST の最近傍<=K_MAX 井 [(wid, dist), ...]。"""
    wids, xy = _train_centroids(); d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy); out = []
    for j in np.argsort(d):
        w = wids[j]
        if w == exclude_wid:
            continue
        if d[j] > MAX_DIST:
            break
        out.append((w, float(d[j])))
        if len(out) >= K_MAX:
            break
    return out


# ==================== GR類似 近傍(v95新規): 全train井から層序が合う井をtop-K ====================
K_GR = 5                                    # nbr_gr5 の本数
_TRAIN_GRTVT = None                         # {wid: (tv_sorted, gr_sorted)}
_TRAIN_CENTZ = None                         # (wids, xyz)  Z重心込み


def _train_grtvt():
    """全train井の (TVT昇順, GR) をキャッシュ(GR類似の照合材料)。"""
    global _TRAIN_GRTVT
    if _TRAIN_GRTVT is None:
        c = _cache()
        if c is not None:
            _TRAIN_GRTVT = c["grtvt"]; return _TRAIN_GRTVT
        tdir = DATA_DIR / "train"; d = {}
        for q in glob.glob(str(tdir / "*__horizontal_well.csv")):
            w = os.path.basename(q).split("__")[0]
            h = pd.read_csv(q, usecols=["GR", "TVT"]); tv = h["TVT"].to_numpy(float); gr = h["GR"].to_numpy(float)
            m = np.isfinite(tv) & np.isfinite(gr)
            if m.sum() < 30:
                continue
            o = np.argsort(tv[m]); d[w] = (tv[m][o], gr[m][o])
        _TRAIN_GRTVT = d
    return _TRAIN_GRTVT


def _train_centroids_z():
    """train井のXYZ重心(ΔZ用にZも)。"""
    global _TRAIN_CENTZ
    if _TRAIN_CENTZ is None:
        c = _cache()
        if c is not None:
            _TRAIN_CENTZ = c["centz"]; return _TRAIN_CENTZ
        tdir = DATA_DIR / "train"
        wids = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(tdir / "*__horizontal_well.csv"))})
        xyz = []
        for w in wids:
            h = pd.read_csv(tdir / f"{w}__horizontal_well.csv", usecols=["X", "Y", "Z"])
            xyz.append((float(h["X"].mean()), float(h["Y"].mean()), float(h["Z"].mean())))
        _TRAIN_CENTZ = (wids, np.array(xyz, float))
    return _TRAIN_CENTZ


def _cc(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[m], b[m])[0, 1]) if m.sum() > 15 and np.std(a[m]) > 1e-9 and np.std(b[m]) > 1e-9 else np.nan


def neighbors_gr_of(pre_tvt, pre_gr, cx, cy, cz, exclude_wid=None, k=K_GR):
    """★全train井から「targetの既知prefix層序(GR-vs-TVT)にGR一致する」top-K井を返す。
       選択= tip(実prefix点)でのGR相関(v95で最脱相関だったraw類似)。
       返り値 [(wid, sim, dist_xy, dz), ...] sim降順。refs材料は ref_grtvt(wid) で取る。"""
    G = _train_grtvt(); cw, cxyz = _train_centroids_z(); cmap = {w: i for i, w in enumerate(cw)}
    m = np.isfinite(pre_tvt) & np.isfinite(pre_gr); pt = pre_tvt[m]; pg = pre_gr[m]
    if len(pt) < 30:
        return []
    o = np.argsort(pt); pt = pt[o]; pg = pg[o]
    if len(pt) > 80:                                  # prefix間引き(高速化)
        s = len(pt) // 80; pt = pt[::s]; pg = pg[::s]
    lo, hi = float(pt[0]), float(pt[-1]); cand = []
    for w, (tv, gr) in G.items():
        if w == exclude_wid:
            continue
        if tv[0] > hi - 5 or tv[-1] < lo + 5:         # prefix範囲を被覆しない井は除外
            continue
        sim = _cc(pg, np.interp(pt, tv, gr))
        if np.isfinite(sim):
            cand.append((w, sim))
    cand.sort(key=lambda t: -t[1]); out = []
    for w, sim in cand[:k]:
        if w in cmap:
            dx = cxyz[cmap[w], 0] - cx; dy = cxyz[cmap[w], 1] - cy; dz = abs(cxyz[cmap[w], 2] - cz)
            out.append((w, float(sim), float(np.hypot(dx, dy)), float(dz)))
        else:
            out.append((w, float(sim), np.nan, np.nan))
    return out


def build_imputers():
    """train全井から FI/DI を構築(self除外は impute 時に self_wid で行う)。IMP_CACHE 有れば流用。"""
    c = _cache()
    if c is not None:
        return c["FI"], c["DI"], c["train_wids"]
    TR = DATA_DIR / "train"
    hw_paths = sorted(TR.glob("*__horizontal_well.csv"))
    train_wids = [p.stem.replace("__horizontal_well", "") for p in hw_paths]
    FI = FormationPlaneKNN(train_wids, TR)
    DI = DenseANCCImputer(train_wids, TR)
    return FI, DI, train_wids


def save_cache(path):
    """★全train参照構造(FI/DI/GR-TVT/重心×2)を1回だけ構築して pickle。submit の最初に1回呼ぶ。
       以後 IMP_CACHE=path を全 subprocess に渡せば、773井の再読込が消える(setup 重複を排除)。"""
    import pickle
    global _CACHE, _TRAIN_CENT, _TRAIN_GRTVT, _TRAIN_CENTZ
    _CACHE = None; _TRAIN_CENT = _TRAIN_GRTVT = _TRAIN_CENTZ = None   # raw build を強制
    TR = DATA_DIR / "train"
    hw = sorted(TR.glob("*__horizontal_well.csv")); wids = [p.stem.replace("__horizontal_well", "") for p in hw]
    obj = {"FI": FormationPlaneKNN(wids, TR), "DI": DenseANCCImputer(wids, TR), "train_wids": wids,
           "cent": _train_centroids(), "grtvt": _train_grtvt(), "centz": _train_centroids_z()}
    Path(path).write_bytes(pickle.dumps(obj, protocol=4))
    print(f"[imp_cache] saved {path} (FI={len(obj['FI'].df)}井 / grtvt={len(obj['grtvt'])}井)", flush=True)
    return path


if __name__ == "__main__":
    import io, sys
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
    FI, DI, wids = build_imputers()
    print(f"[imputers] FI={len(FI.df)}井 / DI点={len(DI.ancc)} / train={len(wids)}井")
    w0 = wids[0]
    h = pd.read_csv(DATA_DIR / "train" / f"{w0}__horizontal_well.csv", usecols=["X", "Y"])
    xy = np.array([[float(h["X"].mean()), float(h["Y"].mean())]])
    pf_pred, pf_d = FI.impute(xy, self_wid=w0)
    ap, asd, ad = DI.impute(xy, self_wid=w0)
    nb = neighbors_of(xy[0, 0], xy[0, 1], exclude_wid=w0)
    print(f"  {w0}: formation pred={np.round(pf_pred[0],1)} knn_d={pf_d[0]:.1f}")
    print(f"       dense ANCC={ap[0]:.1f}±{asd[0]:.1f} d={ad[0]:.1f} / 近傍{len(nb)}={[(w,round(d,0)) for w,d in nb]}")


In [ ]:
%%writefile features_v97.py
# -*- coding: utf-8 -*-
"""v93 module4: 特徴エンジン(v52 の v9-INLINE build_well を忠実移植)。

このモジュールは v52 `create_train_parquet_v52.py` の「特徴生成部」を、struct を一切参照せず
クリーンに再構築したもの。数値は v52 と一致させることが最優先(feature engine は v52 verbatim)。

★v93 で変わるのは pfA バンクの錨源だけ(struct → GRフリー錨)。それは module1/module2 の中で
  完結しており、この module4 は struct/v38/v41 を一切 import/参照しない。

構成:
  - CPU PF numba kernels (_interp1/_resamp/_beam_jit/_pf_ancc/_pf_z) : v52 と数値一致(verbatim)。
  - helpers (_gr_sig/_grid/_nn/_smooth/beam_search/_smooth_radius_values)。
  - run_pf_ancc / run_pf_z : マルチシード CPU PF ラッパ(v52 verbatim)。
  - 統計/特徴ヘルパ + build_well(per-well 特徴エンジン, v52 verbatim)。

★smooth-PF 注入(v52 の monkeypatch をクリーン化):
  v52 では run_pf_ancc を _patched_run_pf_ancc に monkeypatch し、前計算した GPU smooth-PF を
  スレッドローカル wid 経由で返していた。ここでは monkeypatch せず、
    - module-global `FWD_ENS` (dict[(wid, bank_name)] -> {mean,std})
    - thread-local `_CUR.wid`(`set_current_well(wid)` で設定)
  を持ち、run_pf_ancc が最初に FWD_ENS を引く。無ければ本物の CPU マルチシード PF にフォールバック
  (=このファイル単体でも動く)。run_pf_z は常に live 計算(build_well 内で本物を回す)。

  self/nbr 列の追加は build_self_nbr_columns(...) が担う(v52 の one() 相当)。
"""
import os, threading, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from numba import njit

# module3(空間 imputer / 近傍プール)から共有定数と imputer 構築を借りる
import imputers_v97 as imp
from imputers_v97 import FORMATIONS, PLANE_K, DENSE_K

warnings.filterwarnings("ignore"); np.seterr(all="ignore")

DATA_DIR = imp.DATA_DIR

# ==================== v9 定数(v52 line 331-345 と同一) ====================
SEED = 0
DENSE_SPW = 60
N_SPLITS = 5
PF_SUFFIX_START = 1
KEEP_FIRST_PF_ALIAS = True

BEAMS = [
    (15, 7.885250829793783, 69.70049362431021, 3, "gr"),
    (20, 49.43859691674568, 148.592415555271, 3, "hard"),
    (12, 11.84508580886079, 58.844708544034425, 2, "loose"),
    (15, 12.769364767855363, 89.75744127131894, 3, "mid"),
    (15, 10.693883733824851, 118.15908775467301, 8, "smooth"),
    (40, 67.99404057104671, 101.2117946585323, 1, "best"),
]

# 候補パスまわりのオフセット配列(v52 line 851-854 と同一)
ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], np.float32)
SC_OFFS = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
PF_OFFS = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)


# ==================== PF パラメータ写像(v52 line 360-401 verbatim) ====================
def _make_pf_params(raw, idx):
    """Optuna 形式の PF パラメータ dict を、この notebook が使う内部名へ写す。"""
    r = dict(raw)
    p = {}
    p["name"] = str(r.get("name", f"pf_{idx + PF_SUFFIX_START}"))
    p["suffix"] = str(r.get("suffix", f"_{idx + PF_SUFFIX_START}"))
    p["PF_N"] = int(r.get("PF_N", r.get("n_particles", 600)))
    p["ANCC_N"] = int(r.get("ANCC_N", r.get("n_particles", p["PF_N"])))
    p["PF_N_SEEDS"] = int(r.get("PF_N_SEEDS", r.get("n_seeds", 8)))
    p["PF_SEED0"] = int(r.get("PF_SEED0", SEED + idx * 100000))
    p["PF_LIKELIHOOD_SCALE"] = float(r.get("PF_LIKELIHOOD_SCALE", r.get("likelihood_scale", 20.0)))
    p["PF_INIT_SPR"] = float(r.get("PF_INIT_SPR", r.get("init_pos_std", 0.5)))
    p["PF_INIT_V_STD"] = float(r.get("PF_INIT_V_STD", r.get("init_rate_std", 0.01)))
    p["PF_MOM"] = float(r.get("PF_MOM", r.get("mom", 0.999)))
    p["PF_VN"] = float(r.get("PF_VN", r.get("vn", 0.001)))
    p["PF_PN"] = float(r.get("PF_PN", r.get("pn", 0.005)))
    p["PF_ROUGH_P"] = float(r.get("PF_ROUGH_P", r.get("rp", 0.5)))
    p["PF_ROUGH_V"] = float(r.get("PF_ROUGH_V", r.get("rr", 0.001)))
    p["PF_RESAMP"] = float(r.get("PF_RESAMP", r.get("resamp", 0.5)))
    p["PF_GR_SIG_MIN"] = float(r.get("PF_GR_SIG_MIN", r.get("gr_sig_min", 5.0)))
    p["PF_GR_SIG_MAX"] = float(r.get("PF_GR_SIG_MAX", r.get("gr_sig_max", 50.0)))
    p["PF_GR_SIG_DEF"] = float(r.get("PF_GR_SIG_DEF", r.get("gr_sig_def", 100.0)))
    p["PF_GR_SIG_MULT"] = float(r.get("PF_GR_SIG_MULT", r.get("gr_sig_mult", 2.0)))
    p["PF_TVT_CLIP_MARGIN"] = float(r.get("PF_TVT_CLIP_MARGIN", r.get("tvt_clip_margin", 50.0)))
    p["PF_RATE_CLIP"] = float(r.get("PF_RATE_CLIP", r.get("rate_clip", 0.0)))
    p["PF_GR_POWER"] = float(r.get("PF_GR_POWER", r.get("gr_power", 2.0)))
    p["PF_HGR_SMOOTH_R"] = int(r.get("PF_HGR_SMOOTH_R", r.get("hgr_smooth_r", 0)))
    p["PF_TW_GR_SMOOTH_R"] = int(r.get("PF_TW_GR_SMOOTH_R", r.get("tw_gr_smooth_r", 0)))
    p["PF_JUMP_PROB"] = float(r.get("PF_JUMP_PROB", r.get("jump_prob", 0.0)))
    p["PF_JUMP_STD"] = float(r.get("PF_JUMP_STD", r.get("jump_std", 0.0)))
    p["PF_GR_WIN"] = max(1, 2 * int(p["PF_HGR_SMOOTH_R"]) + 1)
    p["PF_GR_WT"] = float(r.get("PF_GR_WT", r.get("gr_wt", 0.3)))

    # ANCC-PF は TVT+Z を追跡。デフォルトは調整済み PF の遷移/初期化/リサンプルを流用。
    p["ANCC_ALPHA"] = float(r.get("ANCC_ALPHA", p["PF_MOM"]))
    p["ANCC_RN"] = float(r.get("ANCC_RN", p["PF_VN"]))
    p["ANCC_PN"] = float(r.get("ANCC_PN", p["PF_PN"]))
    p["ANCC_IR"] = float(r.get("ANCC_IR", p["PF_INIT_V_STD"]))
    p["ANCC_IS"] = float(r.get("ANCC_IS", p["PF_INIT_SPR"]))
    p["ANCC_RP"] = float(r.get("ANCC_RP", p["PF_ROUGH_P"]))
    p["ANCC_RR"] = float(r.get("ANCC_RR", p["PF_ROUGH_V"]))
    return p


# ---- PF_PARAM_SETS を module1 の BANK_ORDER から構築 ----
# ★重要: name をバンク名(pf_1/pf_2/pf_3/r0_seed32/r1_seed32/pfA)にする。
#   これで FWD_ENS/SELF_ENS/NBR_ENS のキー (wid, bank) と build_well 内 p["name"] が一致する。
#   suffix は _1.._6(=特徴列名は v52 と同一)。
import pf_banks_v97 as pf

PF_BANK = list(pf.BANK_ORDER)
PF_PARAM_SETS_RAW = [{**pf.BANK_PARAMS[_b], "name": _b} for _b in PF_BANK]
PF_PARAM_SETS = [_make_pf_params(raw, i) for i, raw in enumerate(PF_PARAM_SETS_RAW)]
if len(PF_PARAM_SETS) == 0:
    raise ValueError("PF_PARAM_SETS_RAW must contain at least one parameter set.")


# ==================== smooth-PF 注入(monkeypatch レス) ====================
FWD_ENS = {}          # (wid, bank_name) -> dict(mean, std)  ← create が populate
_CUR = threading.local()


def set_current_well(wid):
    """build_well 呼び出し前に(スレッドごとに)現在の坑井 wid を設定。"""
    _CUR.wid = wid


# ==================== CPU PF numba kernels(v52 line 410-594 verbatim) ====================
@njit(cache=False)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=False)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    """Beam search ±2 delta, Numba JIT."""
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):            # ±2: TVT can go down
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=False)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IR,IS,RP,RR,RESAMP,
              RATE_CLIP,GR_POWER,JUMP_PROB,JUMP_STD,CLIP_MARGIN,SEED_VALUE):
    if SEED_VALUE >= 0:
        np.random.seed(SEED_VALUE)
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+IR*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    log_lik=0.
    tvt_lo=vmin-CLIP_MARGIN
    tvt_hi=vmin+(len(gg)-1)*step+CLIP_MARGIN
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            if RATE_CLIP>0.:
                if rate[j] > RATE_CLIP: rate[j] = RATE_CLIP
                elif rate[j] < -RATE_CLIP: rate[j] = -RATE_CLIP
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            # Rare jump particles for extreme TVT movement. jump_prob=0 disables this.
            if JUMP_PROB>0. and JUMP_STD>0. and np.random.random()<JUMP_PROB:
                pos[j]+=JUMP_STD*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,tvt_lo); tvt_j=min(tvt_j,tvt_hi)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.; avg_lk=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=abs((gr_v[i]-eg)/gs)
                dp=d**GR_POWER
                lk=max(np.exp(-0.5*dp) if dp<600. else 0.,1e-300)
                old_w=w[j]
                avg_lk+=old_w*lk
                w[j]=old_w*lk; ws+=w[j]
            log_lik+=np.log(max(avg_lk,1e-300))
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            if RATE_CLIP>0.:
                for j in range(N):
                    if rate[j] > RATE_CLIP: rate[j] = RATE_CLIP
                    elif rate[j] < -RATE_CLIP: rate[j] = -RATE_CLIP
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_,log_lik

@njit(cache=False)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP,IP_STD,IV_STD,
          RATE_CLIP,GR_POWER,JUMP_PROB,JUMP_STD,CLIP_MARGIN,SEED_VALUE):
    if SEED_VALUE >= 0:
        np.random.seed(SEED_VALUE)
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+IP_STD*np.random.randn()
        vel[j]=iv+IV_STD*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            if RATE_CLIP>0.:
                if vel[j] > RATE_CLIP: vel[j] = RATE_CLIP
                elif vel[j] < -RATE_CLIP: vel[j] = -RATE_CLIP
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            if JUMP_PROB>0. and JUMP_STD>0. and np.random.random()<JUMP_PROB:
                pos[j]+=JUMP_STD*np.random.randn()
            pos[j]=max(pos[j],vmin-CLIP_MARGIN); pos[j]=min(pos[j],vmin+(len(gg_p)-1)*step+CLIP_MARGIN)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=abs((gr_v[i]-ep)/gs)
                dpow=dp**GR_POWER
                lp=max(np.exp(-0.5*dpow) if dpow<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=abs((gr_sm_v[i]-es)/(gs*1.5))
                    dsp=ds**GR_POWER
                    ls=max(np.exp(-0.5*dsp) if dsp<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_


# ==================== helpers(v52 line 597-639 verbatim) ====================
def _smooth_radius_values(vals, fb, r):
    r = int(r)
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(float(fb))
    if r <= 0:
        return s.to_numpy(np.float32)
    return s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt,tw_gr,step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

def _gr_sig(hw,tw_tvt,tw_gr,p):
    # Match pf_tuning_4_early_resume: estimate sigma using the same smoothed
    # horizontal GR and typewell GR that PF likelihood will use.
    gr_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    hgr=_smooth_radius_values(hw['GR'].astype(float).to_numpy(), gr_fb, p["PF_HGR_SMOOTH_R"])
    kn=hw[hw['TVT_input'].notna()]
    if len(kn)<20:
        gs=float(p["PF_GR_SIG_DEF"])
    else:
        kpos=hw.index.get_indexer(kn.index)
        resid=hgr[kpos]-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)
        gs=float(np.nanstd(resid))
        if not np.isfinite(gs) or gs<=0:
            gs=float(p["PF_GR_SIG_DEF"])
    return float(np.clip(gs*p["PF_GR_SIG_MULT"],p["PF_GR_SIG_MIN"],p["PF_GR_SIG_MAX"]))

def _nn(arr,v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i

def _smooth(vals,fb,r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)


# ==================== マルチシード CPU PF ラッパ ====================
def run_pf_ancc(hw, tw_tvt, tw_gr, p):
    """★smooth-PF 注入つき run_pf_ancc。
       まず FWD_ENS[(現在wid, p['name'])] を引き、あればその smooth-PF 平均/std を返す。
       無ければ本物の CPU マルチシード PF(v52 line 641-677 verbatim)を計算(=単体実行可)。"""
    r = FWD_ENS.get((getattr(_CUR, "wid", None), p.get("name")))
    if r is not None:
        return r["mean"].astype(np.float32), r["std"].astype(np.float32)
    # ---- fallback: 本物の CPU マルチシード ANCC-PF(v52 verbatim) ----
    tw_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth_radius_values(tw_gr, tw_fb, p["PF_TW_GR_SMOOTH_R"]).astype(np.float64)
    gs=_gr_sig(hw,tw_tvt,tw_gr_pf,p)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr_pf)
    md_ev=ev['MD'].values.astype(np.float64)
    z_ev=ev['Z'].values.astype(np.float64)
    # Tuning notebook interpolates + optionally smooths horizontal GR before likelihood.
    gr_fb=tw_fb
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(gr_fb).to_numpy(dtype=float)
    gr_proc=_smooth_radius_values(gr_full, gr_fb, p["PF_HGR_SMOOTH_R"])
    ev_pos=hw.index.get_indexer(ev.index)
    gr_ev=gr_proc[ev_pos].astype(np.float64)

    preds=[]; stds=[]; liks=[]
    for s in range(int(p["PF_N_SEEDS"])):
        pts,std,ll=_pf_ancc(md_ev,z_ev,gr_ev,gg,gmin,gst,
                            gs,ls,ir,int(p["ANCC_N"]),
                            p["ANCC_ALPHA"],p["ANCC_RN"],p["ANCC_PN"],p["ANCC_IR"],p["ANCC_IS"],p["ANCC_RP"],p["ANCC_RR"],
                            p["PF_RESAMP"],p["PF_RATE_CLIP"],p["PF_GR_POWER"],p["PF_JUMP_PROB"],p["PF_JUMP_STD"],
                            p["PF_TVT_CLIP_MARGIN"],int(p["PF_SEED0"])+s)
        preds.append(pts); stds.append(std); liks.append(ll)
    pred_arr=np.stack(preds,0)
    std_arr=np.stack(stds,0)
    lik_arr=np.asarray(liks,dtype=np.float64)
    lik_arr=lik_arr-np.nanmax(lik_arr)
    ww=np.exp(lik_arr/max(float(p["PF_LIKELIHOOD_SCALE"]),1e-6))
    ww=ww/max(float(ww.sum()),1e-300)
    ens=(ww[:,None]*pred_arr).sum(0)
    ens_var=(ww[:,None]*(std_arr**2+(pred_arr-ens[None,:])**2)).sum(0)
    return ens.astype(np.float32),np.sqrt(np.maximum(ens_var,0.)).astype(np.float32)

def run_pf_z(hw,tw_tvt,tw_gr,p):
    """Z追跡 PF(常に live 計算, v52 line 679-710 verbatim)。"""
    tw_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth_radius_values(tw_gr, tw_fb, p["PF_TW_GR_SMOOTH_R"]).astype(np.float64)
    gs=_gr_sig(hw,tw_tvt,tw_gr_pf,p)
    tw_s=pd.Series(tw_gr_pf).rolling(p["PF_GR_WIN"],center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr_pf)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_fb=tw_fb
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(gr_fb).to_numpy(dtype=float)
    gr_proc=_smooth_radius_values(gr_full, gr_fb, p["PF_HGR_SMOOTH_R"])
    gr_sm=pd.Series(gr_proc).rolling(p["PF_GR_WIN"],center=True,min_periods=1).mean().to_numpy(dtype=float)
    ev_pos=hw.index.get_indexer(ev.index)
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   gr_proc[ev_pos].astype(np.float64),
                   gr_sm[ev_pos].astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,int(p["PF_N"]),
                   p["PF_MOM"],p["PF_VN"],p["PF_PN"],p["PF_GR_WT"],p["PF_ROUGH_P"],p["PF_ROUGH_V"],p["PF_RESAMP"],
                   p["PF_INIT_SPR"],p["PF_INIT_V_STD"],p["PF_RATE_CLIP"],p["PF_GR_POWER"],p["PF_JUMP_PROB"],p["PF_JUMP_STD"],
                   p["PF_TVT_CLIP_MARGIN"],int(p["PF_SEED0"]))
    return pts.astype(np.float32),std.astype(np.float32)


# ---- numba warmup(v52 line 713-722) ----
_md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
_gg=np.linspace(45,55,100,np.float64)
_p0=PF_PARAM_SETS[0]
_pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,
         _p0["ANCC_ALPHA"],_p0["ANCC_RN"],_p0["ANCC_PN"],_p0["ANCC_IR"],_p0["ANCC_IS"],_p0["ANCC_RP"],_p0["ANCC_RR"],_p0["PF_RESAMP"],
         _p0["PF_RATE_CLIP"],_p0["PF_GR_POWER"],_p0["PF_JUMP_PROB"],_p0["PF_JUMP_STD"],_p0["PF_TVT_CLIP_MARGIN"],123)
_pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,
      _p0["PF_MOM"],_p0["PF_VN"],_p0["PF_PN"],_p0["PF_GR_WT"],_p0["PF_ROUGH_P"],_p0["PF_ROUGH_V"],_p0["PF_RESAMP"],_p0["PF_INIT_SPR"],_p0["PF_INIT_V_STD"],
      _p0["PF_RATE_CLIP"],_p0["PF_GR_POWER"],_p0["PF_JUMP_PROB"],_p0["PF_JUMP_STD"],_p0["PF_TVT_CLIP_MARGIN"],123)
_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)


# ==================== 統計 / 特徴ヘルパ(v52 line 724-943 verbatim) ====================
def robust_slope(x,y,w=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

def seg_b_well(ktvt,kz,form_col):
    """Segment b_well: early/mid/late thirds + full prefix.
    Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    # WLS (tail-upweighted)
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    # Score-weighted ensemble (NEW: softmax-weighted combination)
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

def _rolling_mean_np(x, w):
    return pd.Series(np.asarray(x, np.float32)).rolling(w, center=True, min_periods=1).mean().to_numpy(np.float32)

def _rolling_std_np(x, w):
    return pd.Series(np.asarray(x, np.float32)).rolling(w, center=True, min_periods=1).std().fillna(0.).to_numpy(np.float32)

def _safe_grad(y, x=None):
    y = np.asarray(y, np.float32)
    n = len(y)
    if n <= 1:
        return np.zeros(n, np.float32)
    if x is None:
        g = np.gradient(y).astype(np.float32)
    else:
        x = np.asarray(x, np.float32)
        dx = np.gradient(x).astype(np.float32)
        dx = np.where(np.abs(dx) < 1e-6, np.nan, dx)
        g = np.gradient(y).astype(np.float32) / dx
        g = np.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return g.astype(np.float32)

def _add_path_shape_features(feats, prefix, path, md=None):
    """Local geometry of a candidate TVT path. No target is used."""
    path = np.asarray(path, np.float32)
    rate = _safe_grad(path, md)
    acc = _safe_grad(rate, md)
    feats[f'{prefix}_rate'] = rate.astype(np.float32)
    feats[f'{prefix}_acc'] = acc.astype(np.float32)
    feats[f'{prefix}_rate_abs'] = np.abs(rate).astype(np.float32)
    feats[f'{prefix}_acc_abs'] = np.abs(acc).astype(np.float32)
    feats[f'{prefix}_rate_rmean21'] = _rolling_mean_np(rate, 21)
    feats[f'{prefix}_rate_rstd21'] = _rolling_std_np(rate, 21)
    feats[f'{prefix}_acc_rmean21'] = _rolling_mean_np(acc, 21)

def _add_gr_match_features(feats, prefix, path, hgr, tw_tvt, tw_gr, offs):
    """How well a candidate TVT path explains the observed horizontal GR."""
    path = np.asarray(path, np.float32)
    hgr = np.asarray(hgr, np.float32)
    offs = np.asarray(offs, np.float32)
    gr_mat = np.stack([np.interp(path + float(o), tw_tvt, tw_gr) for o in offs], axis=1).astype(np.float32)
    resid = (hgr[:, None] - gr_mat).astype(np.float32)
    abs_resid = np.abs(resid)
    zero_idx = int(np.argmin(np.abs(offs)))
    best_idx = abs_resid.argmin(axis=1)
    row_idx = np.arange(len(path))
    r0 = resid[:, zero_idx].astype(np.float32)
    best_signed = resid[row_idx, best_idx].astype(np.float32)
    feats[f'{prefix}_gr_resid0'] = r0
    feats[f'{prefix}_gr_abs0'] = np.abs(r0).astype(np.float32)
    feats[f'{prefix}_gr_abs_min'] = abs_resid[row_idx, best_idx].astype(np.float32)
    feats[f'{prefix}_gr_best_off'] = offs[best_idx].astype(np.float32)
    feats[f'{prefix}_gr_best_resid'] = best_signed
    feats[f'{prefix}_gr_resid0_rmean21'] = _rolling_mean_np(r0, 21)
    feats[f'{prefix}_gr_resid0_rstd21'] = _rolling_std_np(r0, 21)
    feats[f'{prefix}_gr_abs0_rmean21'] = _rolling_mean_np(np.abs(r0), 21)

def _pairwise_abs_stats(mat):
    mat = np.asarray(mat, np.float32)
    k = mat.shape[1]
    if k < 2:
        z = np.zeros(mat.shape[0], np.float32)
        return z, z
    vals = []
    for i in range(k):
        for j in range(i + 1, k):
            vals.append(np.abs(mat[:, i] - mat[:, j]).astype(np.float32))
    diffs = np.stack(vals, axis=1)
    return diffs.mean(axis=1).astype(np.float32), diffs.max(axis=1).astype(np.float32)

def _corr_summary(mat):
    mat = np.asarray(mat, np.float32)
    k = mat.shape[1]
    if k < 2:
        return 0.0, 0.0, 0.0
    vals = []
    for i in range(k):
        xi = mat[:, i]
        xi_std = float(np.std(xi))
        for j in range(i + 1, k):
            xj = mat[:, j]
            xj_std = float(np.std(xj))
            if xi_std < 1e-6 or xj_std < 1e-6:
                vals.append(0.0)
            else:
                vals.append(float(np.corrcoef(xi, xj)[0, 1]))
    vals = np.nan_to_num(np.asarray(vals, np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return float(vals.mean()), float(vals.min()), float(vals.max())


def _add_pf_feature_columns(feats, pf_runs, last_tvt, tvt_f_ancc, tvt_dense, hgr, tw_tvt, tw_gr, sc,
                            hmd=None, beam_mean=None, sc_ens=None, hyb_ref=None):
    """Add PF-derived feature columns for all parameter sets.(v52 line 946-1051 verbatim)"""
    def add_one(r, suffix):
        pf_a = r["pf_ancc"].astype(np.float32)
        std_a = r["pf_ancc_std"].astype(np.float32)
        has_z = bool(r["has_z"])
        pf_z = r["pf_z"].astype(np.float32) if has_z else sc(last_tvt)
        feats[f'pf_ancc{suffix}'] = pf_a
        feats[f'pf_ancc_std{suffix}'] = std_a
        feats[f'pf_ancc_delta{suffix}'] = (pf_a - np.float32(last_tvt)).astype(np.float32)
        feats[f'pf_z{suffix}'] = pf_z
        feats[f'pf_z_delta{suffix}'] = (pf_z - np.float32(last_tvt)).astype(np.float32) if has_z else sc(0.)
        feats[f'pf_vs_z{suffix}'] = (pf_a - pf_z).astype(np.float32) if has_z else sc(0.)
        feats[f'pf_vs_spatial{suffix}'] = (pf_a - tvt_f_ancc).astype(np.float32)
        feats[f'pf_vs_dense{suffix}'] = (pf_a - tvt_dense).astype(np.float32)
        for o in PF_OFFS:
            feats[f'tdpf{int(o)}{suffix}'] = hgr - np.interp(pf_a + o, tw_tvt, tw_gr).astype(np.float32)

        prefix = 'pf' if suffix == '' else f'pf{suffix}'
        _add_gr_match_features(feats, prefix, pf_a, hgr, tw_tvt, tw_gr, PF_OFFS)
        if hmd is not None:
            _add_path_shape_features(feats, prefix, pf_a, hmd)

    if KEEP_FIRST_PF_ALIAS and len(pf_runs) > 0:
        add_one(pf_runs[0], "")
    for r in pf_runs:
        add_one(r, r["suffix"])

    if len(pf_runs) >= 2:
        pf_stack = np.stack([r["pf_ancc"].astype(np.float32) for r in pf_runs], axis=1)
        pf_mean = pf_stack.mean(1).astype(np.float32)
        pf_med = np.median(pf_stack, axis=1).astype(np.float32)
        pf_pair_mean, pf_pair_max = _pairwise_abs_stats(pf_stack)
        c_mean, c_min, c_max = _corr_summary(pf_stack)
        feats['pf_ancc_mean'] = pf_mean
        feats['pf_ancc_med'] = pf_med
        feats['pf_ancc_std_between'] = pf_stack.std(1).astype(np.float32)
        feats['pf_ancc_range_between'] = (pf_stack.max(1) - pf_stack.min(1)).astype(np.float32)
        feats['pf_ancc_iqr_between'] = (np.percentile(pf_stack, 75, axis=1) - np.percentile(pf_stack, 25, axis=1)).astype(np.float32)
        feats['pf_ancc_pair_absmean_between'] = pf_pair_mean
        feats['pf_ancc_pair_absmax_between'] = pf_pair_max
        feats['pf_ancc_mean_delta'] = (pf_mean - np.float32(last_tvt)).astype(np.float32)
        feats['pf_ancc_med_delta'] = (pf_med - np.float32(last_tvt)).astype(np.float32)
        feats['pf_ancc_corr_mean'] = sc(c_mean)
        feats['pf_ancc_corr_min'] = sc(c_min)
        feats['pf_ancc_corr_max'] = sc(c_max)

        for i in range(len(pf_runs)):
            si = pf_runs[i]['suffix']
            feats[f'pf_dev_mean{si}'] = (pf_stack[:, i] - pf_mean).astype(np.float32)
            feats[f'pf_dev_med{si}'] = (pf_stack[:, i] - pf_med).astype(np.float32)
        for i in range(len(pf_runs)):
            for j in range(i + 1, len(pf_runs)):
                si = pf_runs[i]['suffix'].replace('_', '')
                sj = pf_runs[j]['suffix'].replace('_', '')
                diff = (pf_stack[:, i] - pf_stack[:, j]).astype(np.float32)
                feats[f'pf_diff_{si}_{sj}'] = diff
                feats[f'pf_absdiff_{si}_{sj}'] = np.abs(diff).astype(np.float32)

        _add_gr_match_features(feats, 'pf_mean', pf_mean, hgr, tw_tvt, tw_gr, PF_OFFS)
        _add_gr_match_features(feats, 'pf_med', pf_med, hgr, tw_tvt, tw_gr, PF_OFFS)
        if hmd is not None:
            _add_path_shape_features(feats, 'pf_mean', pf_mean, hmd)
            _add_path_shape_features(feats, 'pf_med', pf_med, hmd)

        if beam_mean is not None:
            feats['pf_mean_vs_beam_mean'] = (pf_mean - np.asarray(beam_mean, np.float32)).astype(np.float32)
            feats['pf_med_vs_beam_mean'] = (pf_med - np.asarray(beam_mean, np.float32)).astype(np.float32)
        if sc_ens is not None:
            feats['pf_mean_vs_sc_ens'] = (pf_mean - np.asarray(sc_ens, np.float32)).astype(np.float32)
            feats['pf_med_vs_sc_ens'] = (pf_med - np.asarray(sc_ens, np.float32)).astype(np.float32)
        if hyb_ref is not None:
            feats['pf_mean_vs_hyb'] = (pf_mean - np.asarray(hyb_ref, np.float32)).astype(np.float32)
        feats['pf_mean_vs_spatial'] = (pf_mean - tvt_f_ancc).astype(np.float32)
        feats['pf_mean_vs_dense'] = (pf_mean - tvt_dense).astype(np.float32)

        z_list = [r["pf_z"].astype(np.float32) for r in pf_runs if bool(r["has_z"])]
        if len(z_list) >= 2:
            z_stack = np.stack(z_list, axis=1)
            z_mean = z_stack.mean(1).astype(np.float32)
            z_med = np.median(z_stack, axis=1).astype(np.float32)
            z_pair_mean, z_pair_max = _pairwise_abs_stats(z_stack)
            zc_mean, zc_min, zc_max = _corr_summary(z_stack)
            feats['pf_z_mean'] = z_mean
            feats['pf_z_med'] = z_med
            feats['pf_z_std_between'] = z_stack.std(1).astype(np.float32)
            feats['pf_z_range_between'] = (z_stack.max(1) - z_stack.min(1)).astype(np.float32)
            feats['pf_z_pair_absmean_between'] = z_pair_mean
            feats['pf_z_pair_absmax_between'] = z_pair_max
            feats['pf_z_corr_mean'] = sc(zc_mean)
            feats['pf_z_corr_min'] = sc(zc_min)
            feats['pf_z_corr_max'] = sc(zc_max)
            feats['pf_mean_vs_z_mean'] = (pf_mean - z_mean).astype(np.float32)
            feats['pf_med_vs_z_med'] = (pf_med - z_med).astype(np.float32)


# ==================== imputer(module3)を借りる ====================
_FI = None
_DI = None


def set_imputers(FI, DI):
    """create 側で構築した imputer を注入(二重構築回避)。"""
    global _FI, _DI
    _FI, _DI = FI, DI


def _ensure_imputers():
    """build_well が imputer を必要とするときに、無ければ自前で構築(単体実行可)。"""
    global _FI, _DI
    if _FI is None or _DI is None:
        FI, DI, _ = imp.build_imputers()
        _FI, _DI = FI, DI


# ==================== per-well 特徴エンジン(v52 line 1054-1329 verbatim) ====================
def build_well(hw_path, tw_path, is_train):
    _ensure_imputers()
    global _FI, _DI
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    try:
        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None

    pf_runs=[]
    for pconf in PF_PARAM_SETS:
        pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr,pconf)
        if len(pf_a)==0: return None
        pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr,pconf)
        has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))
        pf_runs.append({
            "name": pconf["name"],
            "suffix": pconf["suffix"],
            "pf_ancc": pf_a.astype(np.float32),
            "pf_ancc_std": std_a.astype(np.float32),
            "pf_z": pf_z.astype(np.float32) if len(pf_z)==len(pf_a) else np.full(len(pf_a), np.nan, np.float32),
            "pf_z_std": std_z.astype(np.float32) if len(std_z)==len(pf_a) else np.full(len(pf_a), np.nan, np.float32),
            "has_z": has_z,
        })
    pf_primary=pf_runs[0]
    pf_use=pf_primary["pf_ancc"].astype(np.float32)
    std_use=pf_primary["pf_ancc_std"].astype(np.float32)
    pf_z=pf_primary["pf_z"].astype(np.float32)
    has_z=bool(pf_primary["has_z"])

    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

    # 7 beams (Numba JIT ±2)
    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS:
        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    # Robust reference beam: older versions used tags 'cons'/'sm5', this notebook uses 'gr'/'smooth'.
    _beam_a=bpaths['cons'] if 'cons' in bpaths else bpaths.get('gr', next(iter(bpaths.values())))
    _beam_b=bpaths['sm5'] if 'sm5' in bpaths else bpaths.get('smooth', _beam_a)
    beam_ref=(_beam_a+_beam_b)/2.

    # Multi-scale NCC → score-weighted ensemble
    ktvt=kn['TVT_input'].to_numpy(np.float32)
    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc_cons=(sc8+sc15+sc25)/3.
    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single

    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
    slp_z=robust_slope(kz,ktvt)

    swid=wid if is_train else None
    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

    # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
        tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
        tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
        tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
        tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
        tvt_fs[f'tvtF50_{fn}']=tvt_f50
        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
        tvt_fs[f'bw50_{fn}']=np.float32(b_late)
        tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
        tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
        form_list.append(tvt_f)

    fs=np.stack(form_list,1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d =fs.std(1).astype(np.float32)
    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
    b_vd=ktvt+z_kn-d_kn
    _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
    b_d=float(np.median(b_vd))
    tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
    tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
    tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
    res_kn=ktvt+z_kn-d_kn
    d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

    candidate_names = [f"pf_ancc{r['suffix']}" for r in pf_runs] + [f"beam_{k}" for k in bpaths.keys()] + [
        'sc8', 'sc15', 'sc25', 'sc_ens', 'spatial_ancc', 'dense_ancc'
    ]
    all_sigs=[r['pf_ancc'] for r in pf_runs]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1).astype(np.float32)
    sig_std=sig_mat.std(1).astype(np.float32)
    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    sig_med=np.median(sig_mat,axis=1).astype(np.float32)
    sig_q10=np.percentile(sig_mat,10,axis=1).astype(np.float32)
    sig_q25=np.percentile(sig_mat,25,axis=1).astype(np.float32)
    sig_q75=np.percentile(sig_mat,75,axis=1).astype(np.float32)
    sig_q90=np.percentile(sig_mat,90,axis=1).astype(np.float32)
    sig_iqr=(sig_q75-sig_q25).astype(np.float32)
    sig_range=(sig_mat.max(1)-sig_mat.min(1)).astype(np.float32)
    sig_mad=np.mean(np.abs(sig_mat-sig_med[:,None]),axis=1).astype(np.float32)
    sig_centered=(sig_mat-sig_mat.mean(1,keepdims=True)).astype(np.float32)
    sig_skew=(np.mean(sig_centered**3,axis=1)/(np.maximum(sig_std,1e-6)**3)).astype(np.float32)

    sig_gr_pred_mat=np.stack([np.interp(sig_mat[:,j],tw_tvt,tw_gr) for j in range(sig_mat.shape[1])],axis=1).astype(np.float32)
    sig_gr_resid_mat=(hgr[:,None]-sig_gr_pred_mat).astype(np.float32)
    sig_gr_abs_mat=np.abs(sig_gr_resid_mat).astype(np.float32)
    sig_gr_best_idx=sig_gr_abs_mat.argmin(axis=1)
    _sig_rows=np.arange(len(hgr))
    sig_gr_best_path=sig_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_best_abs=sig_gr_abs_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_best_resid=sig_gr_resid_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_pred_std=sig_gr_pred_mat.std(1).astype(np.float32)
    sig_gr_resid_mean=sig_gr_resid_mat.mean(1).astype(np.float32)
    sig_gr_resid_std=sig_gr_resid_mat.std(1).astype(np.float32)

    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                   ).iloc[ev.index].values.astype(np.float32)

    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)

    feats={
        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':sc(last_tvt),
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        'sig_std':sig_std,'sig_mean_d':sig_mean,
        'sig_med_d':(sig_med-np.float32(last_tvt)).astype(np.float32),
        'sig_q10_d':(sig_q10-np.float32(last_tvt)).astype(np.float32),
        'sig_q25_d':(sig_q25-np.float32(last_tvt)).astype(np.float32),
        'sig_q75_d':(sig_q75-np.float32(last_tvt)).astype(np.float32),
        'sig_q90_d':(sig_q90-np.float32(last_tvt)).astype(np.float32),
        'sig_iqr':sig_iqr,'sig_range':sig_range,'sig_mad':sig_mad,'sig_skew':sig_skew,
        'sig_gr_pred_std':sig_gr_pred_std,
        'sig_gr_resid_mean':sig_gr_resid_mean,'sig_gr_resid_std':sig_gr_resid_std,
        'sig_gr_best_d':(sig_gr_best_path-np.float32(last_tvt)).astype(np.float32),
        'sig_gr_best_idx':sig_gr_best_idx.astype(np.float32),
        'sig_gr_best_abs':sig_gr_best_abs,'sig_gr_best_resid':sig_gr_best_resid,
        'sig_gr_best_vs_med':(sig_gr_best_path-sig_med).astype(np.float32),
        **tvt_fs,
        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
        'spatial_knn_dist':knn_d,
        'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_spatial':(beam_ref-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':(sc_ens-beam_ref).astype(np.float32),
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),
        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
        'z':z_ev,
        'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
        'dz':(z_ev-float(lk['Z'])).astype(np.float32),
        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
        'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
    }
    _add_pf_feature_columns(
        feats, pf_runs, last_tvt, tvt_fs['tvtF_ANCC'], tvt_dense, hgr, tw_tvt, tw_gr, sc,
        hmd=hmd,
        beam_mean=(np.float32(last_tvt) + feats['beam_mean_d']).astype(np.float32),
        sc_ens=sc_ens,
        hyb_ref=hyb_ref,
    )

    # Shape/GR-match features for non-PF ensemble references.
    beam_mean_path=(np.float32(last_tvt)+feats['beam_mean_d']).astype(np.float32)
    beam_med_path=(np.float32(last_tvt)+feats['beam_med_d']).astype(np.float32)
    _add_path_shape_features(feats,'beam_mean',beam_mean_path,hmd)
    _add_path_shape_features(feats,'beam_med',beam_med_path,hmd)
    _add_path_shape_features(feats,'sc_ens',sc_ens,hmd)
    _add_path_shape_features(feats,'hyb',hyb_ref,hmd)
    _add_path_shape_features(feats,'sig_med',sig_med,hmd)
    _add_path_shape_features(feats,'sig_gr_best',sig_gr_best_path,hmd)
    _add_gr_match_features(feats,'beam_mean',beam_mean_path,hgr,tw_tvt,tw_gr,BEAM_OFFS)
    _add_gr_match_features(feats,'beam_med',beam_med_path,hgr,tw_tvt,tw_gr,BEAM_OFFS)
    _add_gr_match_features(feats,'sc_ens',sc_ens,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'hyb',hyb_ref,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'sig_med',sig_med,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'sig_gr_best',sig_gr_best_path,hgr,tw_tvt,tw_gr,SC_OFFS)

    for k,v in rolls.items(): feats[k]=v

    # Additional GR morphology features. These use only observed horizontal/typewell GR.
    feats['gr_z21']=((hgr-rolls['grm21'])/(rolls['grs21']+1e-3)).astype(np.float32)
    feats['gr_z51']=((hgr-rolls['grm51'])/(rolls['grs51']+1e-3)).astype(np.float32)
    feats['gr_abs_d1']=np.abs(gr_d1).astype(np.float32)
    feats['gr_abs_d2']=np.abs(gr_d2).astype(np.float32)
    feats['gr_d1_rmean21']=_rolling_mean_np(gr_d1,21)
    feats['gr_abs_d1_rmean21']=_rolling_mean_np(np.abs(gr_d1),21)
    feats['gr_rough21']=_rolling_mean_np(np.abs(gr_d1),21)
    feats['gr_rough51']=_rolling_mean_np(np.abs(gr_d1),51)
    _tw_gr_sorted=np.sort(tw_gr.astype(np.float32))
    _kg_sorted=np.sort(kgr.astype(np.float32)) if len(kgr)>0 else _tw_gr_sorted
    feats['gr_pct_tw']=(np.searchsorted(_tw_gr_sorted,hgr,side='right')/max(len(_tw_gr_sorted),1)).astype(np.float32)
    feats['gr_pct_known']=(np.searchsorted(_kg_sorted,hgr,side='right')/max(len(_kg_sorted),1)).astype(np.float32)
    feats['gr_z_known']=((hgr-np.float32(np.mean(kgr)))/(np.float32(np.std(kgr))+1e-3)).astype(np.float32)
    feats['gr_z_tw']=((hgr-np.float32(np.mean(tw_gr)))/(np.float32(np.std(tw_gr))+1e-3)).astype(np.float32)
    feats['gr_vs_known_last']=(hgr-np.float32(kgr[-1])).astype(np.float32)
    feats['gr_vs_known_mean']=(hgr-np.float32(np.mean(kgr))).astype(np.float32)

    # Geometry roughness of the actual drilled path.
    step_xy=np.sqrt(dxdmd**2+dydmd**2).astype(np.float32)
    feats['xy_rate']=step_xy
    feats['xyz_rate']=np.sqrt(dxdmd**2+dydmd**2+dzdmd**2).astype(np.float32)
    feats['z_rate_abs']=np.abs(dzdmd).astype(np.float32)
    feats['dzdmd_rmean21']=_rolling_mean_np(dzdmd,21)
    feats['dzdmd_rstd21']=_rolling_std_np(dzdmd,21)
    feats['xy_rate_rmean21']=_rolling_mean_np(step_xy,21)
    feats['xy_rate_rstd21']=_rolling_std_np(step_xy,21)

    result=pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
    return result


# ==================== self/nbr 列追加(v52 one() line 1725-1763 verbatim) ====================
METHODS_V95 = ["self", "nbr", "nbr_gr5", "self_graft"]   # tw は build_well の pf_ancc(baseline)


def build_self_nbr_columns(df, wid, ENS, ZGRAD, NBR_META, PROV):
    """★v95: 5表現(tw baseline + self/nbr/nbr_gr5/self_graft)の smooth-PF 予測・std・delta・vs・loglik、
       表現間比較、クロスバンク集約、そして各表現の provenance(GR一致度/距離/ΔZ/類似度/較正/被覆)を列追加。
       ENS = {method: {(wid,bank)->dict(mean,std,loglik)}}。PROV = {wid: {method: {...provenance...}}}。"""
    n = len(df); lk = df["last_known_tvt"].to_numpy(float)
    def col(a):
        out = np.full(n, np.nan, np.float32)
        if a is not None:
            mm = min(len(a), n); out[:mm] = np.asarray(a[:mm], np.float32)
        return out
    def setp(name, v):
        df[name] = np.float32(v) if (v is not None and np.isfinite(v)) else np.float32(np.nan)
    mt = NBR_META.get(wid, {}); pv = PROV.get(wid, {})
    acc = {m: [] for m in METHODS_V95}; T_all = []
    for p in PF_PARAM_SETS:
        suf = p["suffix"]; nm = p["name"]
        twk = df[f"pf_ancc{suf}"].to_numpy(float) if f"pf_ancc{suf}" in df.columns else lk.copy()
        preds = {}
        for m in METHODS_V95:
            e = ENS[m].get((wid, nm))
            pm_ = col(e["mean"] if e else None); ps_ = col(e["std"] if e else None)
            df[f"pf_{m}{suf}"] = pm_; df[f"pf_{m}_std{suf}"] = ps_
            df[f"pf_{m}_delta{suf}"] = (pm_ - lk).astype(np.float32)
            df[f"pf_{m}_vs_tw{suf}"] = (pm_ - twk).astype(np.float32)      # ★baseline tw との不一致
            df[f"pf_{m}_loglik{suf}"] = np.full(n, np.float32(e["loglik"]) if (e and "loglik" in e) else np.float32(np.nan), np.float32)
            preds[m] = pm_.astype(float); acc[m].append(pm_.astype(float))
        # 表現間比較(相対的妥当性)
        df[f"pf_self_vs_nbr{suf}"] = (preds["self"] - preds["nbr"]).astype(np.float32)
        df[f"pf_nbr_gr5_vs_nbr{suf}"] = (preds["nbr_gr5"] - preds["nbr"]).astype(np.float32)
        df[f"pf_self_graft_vs_self{suf}"] = (preds["self_graft"] - preds["self"]).astype(np.float32)
        # 5表現 consensus(tw + 4手法)
        st = np.stack([twk] + [preds[m] for m in METHODS_V95], axis=1)
        df[f"pf_agree_std{suf}"] = np.nanstd(st, axis=1).astype(np.float32)
        df[f"pf_agree_range{suf}"] = (np.nanmax(st, axis=1) - np.nanmin(st, axis=1)).astype(np.float32)
        T_all.append(twk)
    # クロスバンク集約(各手法: 平均 / バンク間ばらつき / 平均のtw不一致)
    TM = np.stack(T_all, 1)
    for m in METHODS_V95:
        M = np.stack(acc[m], 1)
        df[f"pf_{m}_mean"] = np.nanmean(M, 1).astype(np.float32)
        df[f"pf_{m}_pstd"] = np.nanstd(M, 1).astype(np.float32)
        df[f"pf_{m}_mean_vs_tw"] = (np.nanmean(M, 1) - np.nanmean(TM, 1)).astype(np.float32)
    df["z_grad"] = col(ZGRAD.get(wid))
    # ==== provenance(その特徴がどう作られたか=妥当性の説明変数, well単位→全行broadcast) ====
    for m in ["tw", "self", "nbr", "nbr_gr5", "self_graft"]:
        d = pv.get(m, {})
        setp(f"prov_{m}_grfit", d.get("grfit")); setp(f"prov_{m}_grcorr", d.get("grcorr"))
    for m in ["nbr", "nbr_gr5"]:
        d = pv.get(m, {})
        setp(f"prov_{m}_dist_min", d.get("dist_min")); setp(f"prov_{m}_dist_mean", d.get("dist_mean"))
        setp(f"prov_{m}_dz_min", d.get("dz_min")); setp(f"prov_{m}_dz_mean", d.get("dz_mean"))
        setp(f"prov_{m}_nref", d.get("nref"))
    setp("prov_nbr_gr5_sim_top1", pv.get("nbr_gr5", {}).get("sim_top1"))
    setp("prov_nbr_gr5_sim_mean", pv.get("nbr_gr5", {}).get("sim_mean"))
    setp("prov_self_cover", pv.get("self", {}).get("cover")); setp("prov_self_nprefix", pv.get("self", {}).get("nprefix"))
    for k in ["a", "b", "valid", "cover", "extrap_frac"]:
        setp(f"prov_self_graft_{k}", pv.get("self_graft", {}).get(k))
    # grfit の相対(どの手法がこの井のGRに最も合うか)
    twf = pv.get("tw", {}).get("grfit")
    for m in ["self", "nbr", "nbr_gr5", "self_graft"]:
        mf = pv.get(m, {}).get("grfit")
        setp(f"prov_{m}_grfit_vs_tw", (mf - twf) if (mf is not None and twf is not None and np.isfinite(mf) and np.isfinite(twf)) else None)
    # legacy 近傍メタ
    df["pf_nbr_has"] = np.float32(mt.get("has", 0)); df["pf_nbr_nndist"] = np.float32(mt.get("nn", np.nan)); df["pf_nbr_nrefs"] = np.float32(mt.get("nref", 0))
    return df


In [ ]:
%%writefile create_v97.py
# -*- coding: utf-8 -*-
"""v95 orchestration: train/test.parquet ビルダー(v93 create を 5表現+provenance に拡張)。

v93 → v95 変更:
  - PF表現を tw/self/nbr の3 → tw/self/nbr/nbr_gr5/self_graft の5表現に拡張。
    * nbr_gr5 = 全train井から GR類似 top5(imputers.neighbors_gr_of) を build_inputs_nbr に渡す(refs差替のみ)。
    * self_graft = build_inputs_self_graft(prefix実測+tw較正外挿)。
  - ★NN-emission 無し(w_nn=0・sim非添付)。物理錨(attach_anchor)は温存(pfAのみ)。逆方向PFも無し。
  - ★provenance(各表現の妥当性説明変数)を well単位で計算し features へ: GR一致度(grfit/grcorr)・
    近傍距離/ΔZ/GR類似度・self被覆・self_graft較正係数。
  - PFパラメータは pf_banks_config.json(v93からコピー=同一)。

出力: v95/artifacts_v95/data/{train,test}.parquet
実行: rogii_claude/.venv/Scripts/python.exe v95/create_v95.py   スモーク: NWELLS=6 python v95/create_v95.py
"""
import os, sys, io, glob, time, pickle
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import pf_banks_v97 as pf
import imputers_v97 as imp
import features_v97 as feats

PROJ = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
DATA_DIR = pf.DATA_DIR
ART = pf.ART
SPLIT = os.environ.get("SPLIT", "train")
IS_TRAIN = (SPLIT == "train")
OUT_PARQUET = (Path(os.environ["OUT_PARQUET"]) if os.environ.get("OUT_PARQUET")
               else ART / "data" / ("train.parquet" if IS_TRAIN else "test.parquet"))
SEED = int(os.environ.get("V97_SEED", "7715023"))   # ★v97: 生成seed=調整時と別値(n_seeds=32)
N_SEED = pf.N_SEED
# ---- NN-emission sim(module2 出力。帯中心=GRフリー錨tvt)。train/testで別pkl ----
SIM_PKL = (Path(os.environ["SIM_PKL"]) if os.environ.get("SIM_PKL")
           else pf.ART / ("sim_grfree_v97.pkl" if os.environ.get("SPLIT", "train") == "train" else "sim_grfree_test_v97.pkl"))


def _attach_sim(x, wid, SIMD):
    """PF入力dict に NN-emission(sim/帯中心 st)を添付。無い井は素通し(注入OFF)。"""
    sd = SIMD.get(wid)
    if sd is not None and len(sd.get("st", [])) == len(x["md"]):
        x["_sim"] = sd["sim"]; x["_st"] = sd["st"]
    return x


AFFINE = os.environ.get("AFFINE", "0") == "1"   # ★affine版特徴生成
if AFFINE:
    import affine_v102 as _aff
WELL_CHUNK = 60
N_JOBS = 4
NWELLS = int(os.environ["NWELLS"]) if os.environ.get("NWELLS") else None
DEVICE = pf.DEVICE
BANK_ORDER = pf.BANK_ORDER
METHODS = ["tw", "self", "nbr", "nbr_gr5", "self_graft"]


def _gr_prefix_sm(hw, P):
    """既知prefix の 平滑GR と TVT_input(provenance grfit 用)。"""
    kn = hw[hw["TVT_input"].notna()]
    fb = float(np.nanmean(hw["GR"])) if np.isfinite(np.nanmean(hw["GR"])) else 0.0
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(fb).to_numpy(float)
    gr_sm = pf._smooth_radius_values(gr_full, fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    return kn["TVT_input"].to_numpy(float), gr_sm[kpos].astype(float)


def _grfit(kn_tvtin, gr_pre, x):
    """参照 gg が prefix GR をどれだけ説明するか: 残差std(grfit)・相関(grcorr)。"""
    if x is None or len(kn_tvtin) < 8:
        return np.nan, np.nan
    gg = x["gg"]; grid = x["gmin"] + np.arange(len(gg)) * x["gst"]
    at = np.interp(kn_tvtin, grid, gg); r = gr_pre - at; m = np.isfinite(r)
    if m.sum() < 8:
        return np.nan, np.nan
    gf = float(np.std(r[m]))
    gc = (float(np.corrcoef(gr_pre[m], at[m])[0, 1])
          if np.std(gr_pre[m]) > 1e-9 and np.std(at[m]) > 1e-9 else np.nan)
    return gf, gc


def build_dataset():
    base = DATA_DIR / SPLIT
    wells = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(base / "*__horizontal_well.csv"))})
    if NWELLS is not None:
        wells = wells[:NWELLS]
    P0 = pf.bank_param(BANK_ORDER[0])
    names = []; paths = []
    for w in wells:
        hp = base / f"{w}__horizontal_well.csv"; tp = base / f"{w}__typewell.csv"
        try:
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
        except Exception:
            continue
        if IS_TRAIN and "TVT" not in hw.columns:
            continue
        if pf.build_smoother_inputs(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), P0) is None:
            continue
        names.append(w); paths.append((hp, tp))
    print(f"[{SPLIT}] 使用坑井: {len(names)}  banks={BANK_ORDER} x 5表現={METHODS}  lag={pf.SMOOTH_LAG} n_seed={N_SEED} (NN-emission ON)")

    FI, DI, _tw = imp.build_imputers()
    feats.set_imputers(FI, DI)
    print(f"[imputers] FI={len(FI.df)}井 / DI点={len(DI.ancc)}")

    FWD_ENS = feats.FWD_ENS; FWD_ENS.clear()
    ENS = {m: {} for m in ["self", "nbr", "nbr_gr5", "self_graft"]}   # tw は FWD_ENS
    ZGRAD = {}; NBR_META = {}; NREFS = {}; NREFS_GR5 = {}; PROV = {}

    # ---- 近傍探索(距離 & GR類似) + provenance ----
    print("[nbr] 近傍探索(距離top3 & GR類似top5) + provenance ...")
    t0 = time.perf_counter(); n_nbr = 0; n_gr5 = 0
    _cw, _cxyz = imp._train_centroids_z(); _cmap = {w: i for i, w in enumerate(_cw)}
    for (hp, tp) in paths:
        wid = os.path.basename(str(hp)).split("__")[0]
        hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
        tvt_tw = tw["TVT"].to_numpy(float); gr_tw = tw["GR"].to_numpy(float)
        cx, cy = float(hw["X"].mean()), float(hw["Y"].mean()); cz = float(hw["Z"].mean())
        exc = wid if SPLIT == "train" else None
        # 距離近傍
        refw = imp.neighbors_of(cx, cy, exclude_wid=exc)
        NREFS[wid] = [imp.ref_grtvt(r) for r, _ in refw] if len(refw) >= imp.NEED_REFS else []
        nbr_dists = [d for _, d in refw]; nbr_dz = []
        for r, _ in refw:
            if r in _cmap:
                nbr_dz.append(abs(_cxyz[_cmap[r], 2] - cz))
        # GR類似近傍
        kn = hw[hw["TVT_input"].notna()]
        ng = imp.neighbors_gr_of(kn["TVT_input"].to_numpy(float), kn["GR"].to_numpy(float), cx, cy, cz, exclude_wid=exc)
        NREFS_GR5[wid] = [imp.ref_grtvt(w) for w, _, _, _ in ng]
        gr5_sim = [s for _, s, _, _ in ng]; gr5_dist = [d for _, _, d, _ in ng]; gr5_dz = [z for _, _, _, z in ng]
        NBR_META[wid] = dict(has=int(len(NREFS[wid]) > 0), nn=imp.nearest_dist(cx, cy, exclude_wid=exc), nref=len(refw))
        if NREFS[wid]:
            n_nbr += 1
        if NREFS_GR5[wid]:
            n_gr5 += 1
        # provenance: 5表現の gg を P0 で作り grfit
        kt, kg = _gr_prefix_sm(hw, P0)
        x_tw = pf.build_smoother_inputs(hw, tvt_tw, gr_tw, P0)
        x_self = pf.build_inputs_self(hw, tvt_tw, gr_tw, P0)
        x_nbr = pf.build_inputs_nbr(hw, tvt_tw, gr_tw, P0, NREFS[wid])
        x_gr5 = pf.build_inputs_nbr(hw, tvt_tw, gr_tw, P0, NREFS_GR5[wid])
        x_sg = pf.build_inputs_self_graft(hw, tvt_tw, gr_tw, P0)
        prov = {}
        for mm, xx in [("tw", x_tw), ("self", x_self), ("nbr", x_nbr), ("nbr_gr5", x_gr5), ("self_graft", x_sg)]:
            gf, gc = _grfit(kt, kg, xx); prov[mm] = dict(grfit=gf, grcorr=gc)
        _cov = float(x_sg["_graft"]["cover"]) if (x_sg is not None and "_graft" in x_sg) else np.nan
        prov["self"].update(cover=_cov, nprefix=int(len(kt)))     # self被覆=prefixがtw格子を覆う割合(self_graftのcoverと同一計算)
        prov["nbr"].update(nref=len(refw),
                           dist_min=(min(nbr_dists) if nbr_dists else np.nan), dist_mean=(float(np.mean(nbr_dists)) if nbr_dists else np.nan),
                           dz_min=(min(nbr_dz) if nbr_dz else np.nan), dz_mean=(float(np.mean(nbr_dz)) if nbr_dz else np.nan))
        prov["nbr_gr5"].update(nref=len(ng),
                               sim_top1=(gr5_sim[0] if gr5_sim else np.nan), sim_mean=(float(np.mean(gr5_sim)) if gr5_sim else np.nan),
                               dist_min=(min(gr5_dist) if gr5_dist else np.nan), dist_mean=(float(np.mean(gr5_dist)) if gr5_dist else np.nan),
                               dz_min=(min(gr5_dz) if gr5_dz else np.nan), dz_mean=(float(np.mean(gr5_dz)) if gr5_dz else np.nan))
        if x_sg is not None and "_graft" in x_sg:
            prov["self_graft"].update(**x_sg["_graft"])
        PROV[wid] = prov
    print(f"      近傍あり {n_nbr}/{len(names)} / GR類似あり {n_gr5}/{len(names)}  ({time.perf_counter()-t0:.1f}s)")

    # ---- NN-emission sim 読込(★v97: ON) ----
    SIMD = {}
    if SIM_PKL.exists():
        SIMD = pickle.loads(SIM_PKL.read_bytes())
        print(f"[v97 NN-emission] sim load: {len(SIMD)} wells  <- {SIM_PKL.name}")
    else:
        print(f"[v97 NN-emission] WARN: {SIM_PKL.name} 無し -> NN-emission OFF(sim空)")

    # ---- 各バンク × 5表現 smooth-PF(★w_nn=P['_w_nn'] = NN-emission ON) ----
    PFCACHE = Path(os.environ.get("PF_CACHE_DIR", str(ART / "pfcache"))); PFCACHE.mkdir(parents=True, exist_ok=True)
    FORCE_PF = os.environ.get("FORCE_PF", "0") == "1"

    def run_method(bank, P, builder, store, m):
        # ★v99 PFキャッシュ: per-(split,表現,bank)。後でバンク/特徴追加時に既存PF再実行不要
        cache = PFCACHE / f"{SPLIT}_{m}_{bank}.pkl"
        if cache.exists() and not FORCE_PF:
            d = pickle.loads(cache.read_bytes())
            if all(w in d for w in names):                     # 現在の井戸集合を完全にカバーする時のみ流用
                for w in names:
                    store[(w, bank)] = d[w]
                print(f"      [cache] {cache.name} 流用({len(names)}井)")
                return
            print(f"      [cache] {cache.name} 井戸不一致({len(d)}井<必要{len(names)}井) -> 再計算")
        inps = []
        for (hp, tp) in paths:
            wid = os.path.basename(str(hp)).split("__")[0]
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
            x = builder(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), wid)
            if AFFINE:
                x = _aff.apply_affine(pf, x, hw, tw["GR"].to_numpy(float), P)   # ★affine較正
            x = pf.attach_anchor(x, wid, P["_physics"])       # GRフリー錨(use_anchorバンクのみ)
            x = _attach_sim(x, wid, SIMD)                      # ★NN-emission 添付
            inps.append(x)
        outs = pf.run_smoother_ext(inps, P, SEED, N_SEED, WELL_CHUNK, w_nn=P["_w_nn"])   # ★NN-emission ON
        dd = {}
        for i, w in enumerate(names):
            store[(w, bank)] = outs[i]; dd[w] = outs[i]
        cache.write_bytes(pickle.dumps(dd, protocol=4))
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    for bank in BANK_ORDER:
        P = pf.bank_param(bank)
        builders = {
            "tw":         lambda hw, tt, tg, wid, _P=P: pf.build_smoother_inputs(hw, tt, tg, _P),
            "self":       lambda hw, tt, tg, wid, _P=P: pf.build_inputs_self(hw, tt, tg, _P),
            "nbr":        lambda hw, tt, tg, wid, _P=P: pf.build_inputs_nbr(hw, tt, tg, _P, NREFS[wid]),
            "nbr_gr5":    lambda hw, tt, tg, wid, _P=P: pf.build_inputs_nbr(hw, tt, tg, _P, NREFS_GR5[wid]),
            "self_graft": lambda hw, tt, tg, wid, _P=P: pf.build_inputs_self_graft(hw, tt, tg, _P),
        }
        for m in METHODS:
            t0 = time.perf_counter(); print(f"[smoothPF-{m}] {bank} ...")
            store = FWD_ENS if m == "tw" else ENS[m]
            run_method(bank, P, builders[m], store, m)
            print(f"      done {time.perf_counter()-t0:.1f}s")

    for (hp, tp) in paths:
        wid = os.path.basename(str(hp)).split("__")[0]
        ZGRAD[wid] = pf.z_gradient_eval(pd.read_csv(hp))

    ENS_ALL = {"self": ENS["self"], "nbr": ENS["nbr"], "nbr_gr5": ENS["nbr_gr5"], "self_graft": ENS["self_graft"]}

    def one(w, hp, tp):
        feats.set_current_well(w)
        df = feats.build_well(str(hp), str(tp), IS_TRAIN)
        if df is None or len(df) == 0:
            return None
        return feats.build_self_nbr_columns(df, w, ENS_ALL, ZGRAD, NBR_META, PROV)

    print("[build] build_well(tw注入) + 5表現/provenance 列 ...")
    t0 = time.perf_counter()
    res = Parallel(n_jobs=N_JOBS, prefer="threads", verbose=5)(
        delayed(one)(names[i], paths[i][0], paths[i][1]) for i in range(len(names)))
    parts = [r for r in res if r is not None]
    print(f"      done {time.perf_counter()-t0:.1f}s  parts={len(parts)}")
    if not parts:
        raise RuntimeError("有効な坑井がありません。")
    return pd.concat(parts, ignore_index=True)


def main():
    print(f"[v97] device={DEVICE} smooth_lag={pf.SMOOTH_LAG} n_seed={N_SEED} chunk={WELL_CHUNK} NWELLS={NWELLS} (NN-emission ON / 逆方向なし)")
    df = build_dataset()
    if SPLIT == "train" and NWELLS is None and df["well"].nunique() < 700:
        raise SystemExit(f"[GUARD] 出力坑井数 {df['well'].nunique()} 本は異常。")
    if os.environ.get("PF_ONLY_OUT", "0") == "1":         # ★共通の非PF特徴は base と同一=不要。pf_候補のみ
        meta = [c for c in ["id", "well", "target", "last_known_tvt", "md_since"] if c in df.columns]
        keep = meta + [c for c in df.columns if c.startswith("pf_")]
        df = df[keep]; print(f"[PF_ONLY_OUT] pf_候補のみ({len(keep)}列)", flush=True)
    OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(OUT_PARQUET, index=False)
    cols = list(df.columns)
    print(f"\nsaved: {OUT_PARQUET}  rows={len(df)} cols={len(cols)} wells={df['well'].nunique()}")
    for pre in ["pf_self", "pf_nbr", "pf_nbr_gr5", "pf_self_graft", "prov_", "pf_"]:
        print(f"  {pre}* = {sum(c.startswith(pre) for c in cols)} 列")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile build_seqfeat_v97.py
# -*- coding: utf-8 -*-
"""v93 L5(v62 曲線トークン脚)用 seqfeat ビルダー -- STRUCT-FREE 版。

v62 の build_seqfeat_v62.py を忠実に再現するが、以下が違う:
  - 軌跡は 14 本(v62 の 16 本から gf / v50b を DROP)。★struct禁止方針。
    14本 = pf_ancc_1..6 / pf_self_1..3 / pf_nbr_1..3 / beam_med / sc_ens。
  - 基準線(anchor)を v62 の struct `st`(= v38 struct_prior 由来。struct混入!)から
    v93 の struct-free `last_known_tvt`(hold線)へ張り替え。
      * v93 parquet の `*_delta` 列 = (絶対値 - last_known_tvt) なので、そのまま δ として使える。
      * beam_med_d / sc_ens_d は元から last_known からの δ。
      * per-bin ターゲット dtrue = median(target)。ここで target = TVT - last_known_tvt(= parquet 'target')。
        v62 の dtrue = median(y_true - st) の st を last_known に置換したものに一致。
  - bin 構造(nb, bid, sel)は v62/v61 と完全一致(STR=16, sel=clip(arange(nb)*STR+STR//2,0,T-1))。
    v93 parquet の eval 行数は imagesR_v61 と一致(例: well0 = 3836 行 -> nb=240)を確認済み。

出力: v93/artifacts_v93/seqfeat_v93.pkl
  = {well: {D:(nb,14)δ, S:(nb,14)幅, C:(nb,3)文脈, dtrue:(nb,), wgt:(nb,),
            bid:(T,)行->bin, rows:(T,)parquet全体行index, T:int}}
  ※ D/S/C/dtrue/wgt は v62 と同じ役割。bid/rows/T は train 側で per-row OOF を復元するための追加。

struct混入なし: gf_feature(v48)/v50b/struct/v38/v41 は一切読まない。v93 parquet 単体。
"""
import io, sys
sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
import os, pickle, time
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

P0 = Path(os.environ.get("ROGII_PROJ", r"C:\Users\kosaka256\Documents\rogii_claude"))
ART = Path(os.environ.get("ROGII_ART97B", str(P0 / "v97" / "artifacts_v97")))
# --- 入力切替(env V93_INPUT): fwd / fwdbwd / fwdbwd_anchor ---
V93_INPUT = os.environ.get("V93_INPUT", "fwd").lower()
_SMAP = {"fwd": ("train.parquet", ""),
         "fwdbwd": ("train_fwdbwd.parquet", "_fwdbwd"),
         "fwdbwd_anchor": ("train_fwdbwd_anchor.parquet", "_fwdbwd_anchor")}
if V93_INPUT not in _SMAP:
    raise SystemExit(f"[seqfeat] 未知の V93_INPUT={V93_INPUT}")
_fn, _osuf = _SMAP[V93_INPUT]
# 提出(test): SF_IN=test_fwdbwd.parquet / SF_OUT=seqfeat_test... / SF_NO_TARGET=1(target列なし)
NO_TARGET = os.environ.get("SF_NO_TARGET", "0") == "1"
PQ = Path(os.environ["SF_IN"]) if os.environ.get("SF_IN") else ART / "data" / _fn
OUT = Path(os.environ["SF_OUT"]) if os.environ.get("SF_OUT") else ART / (f"seqfeat_v95b{_osuf}.pkl")
STR = 16

# --- ★v97: 4バンク構造(pf_1..4)。ancc/self/nbr × 4バンク + beam/sc_ens。schema に無い列は main で自動除外 ---
NB = int(os.environ.get("V97_NBANK", "4"))            # v97 = 4バンク
BASE_DELTA = ([f"pf_ancc_delta_{k}" for k in range(1, NB + 1)] +
              [f"pf_self_delta_{k}" for k in range(1, NB + 1)] +
              [f"pf_nbr_delta_{k}" for k in range(1, NB + 1)] +
              ["beam_med_d", "sc_ens_d"])
BASE_STD = ([f"pf_ancc_std_{k}" for k in range(1, NB + 1)] +
            [f"pf_self_std_{k}" for k in range(1, NB + 1)] +
            [f"pf_nbr_std_{k}" for k in range(1, NB + 1)] +
            ["beam_std_d", None])
assert len(BASE_DELTA) == len(BASE_STD)
# --- ★backward(bpf)軌跡ch(fwdbwd* のみ, parquet に在れば追加): ancc6/self3/nbr3 = 12本 ---
BPF_DELTA = ([f"bpf_ancc_delta_{k}" for k in range(1, 7)] +
             [f"bpf_self_delta_{k}" for k in range(1, 4)] +
             [f"bpf_nbr_delta_{k}" for k in range(1, 4)])
BPF_STD = ([f"bpf_ancc_std_{k}" for k in range(1, 7)] +
           [f"bpf_self_std_{k}" for k in range(1, 4)] +
           [f"bpf_nbr_std_{k}" for k in range(1, 4)])


def main():
    t0 = time.time()
    sch = set(pq.read_schema(PQ).names)
    _EXTRA_PQ = [p for p in os.environ.get("SF_EXTRA", "").split(",") if p]   # ★追加parqu(id整列)の列も候補に使える
    _extra_map = {}
    for _p in _EXTRA_PQ:
        for _c in pq.read_schema(_p).names:
            if _c not in sch and _c not in _extra_map:
                _extra_map[_c] = _p
    def _has(c):                                          # base schema か SF_EXTRA のどちらかに在れば真
        return (c in sch) or (c in _extra_map)
    # ★v97: base ch も schema に在るものだけ(beam/sc_ens やバンク数の差を自動吸収)
    DELTA = []; STD = []
    for d, s in zip(BASE_DELTA, BASE_STD):
        if d in sch and (s is None or s in sch):
            DELTA.append(d); STD.append(s)
    _ndrop = len(BASE_DELTA) - len(DELTA)
    if _ndrop:
        print(f"[seqfeat] base ch のうち schema 不在 {_ndrop} 本を除外(K={len(DELTA)})", flush=True)
    if V93_INPUT != "fwd":                          # bpf 軌跡chを(在るものだけ)追加
        for d, s in zip(BPF_DELTA, BPF_STD):
            if d in sch and s in sch:
                DELTA.append(d); STD.append(s)
    # ★実験: 追加チャネル(env EXTRA_DELTA/EXTRA_STD, カンマ区切り。新PFをL5に足す等)
    _ed = [x for x in os.environ.get("EXTRA_DELTA", "").split(",") if x]
    _es = [x for x in os.environ.get("EXTRA_STD", "").split(",") if x]
    n_extra = 0
    for d, s in zip(_ed, _es):
        if _has(d) and _has(s):                          # ★SF_EXTRA 側の列も許可(dec/aff 候補を落とさない)
            DELTA.append(d); STD.append(s); n_extra += 1
    print(f"[seqfeat] V93_INPUT={V93_INPUT} 入力={PQ.name} 出力={OUT.name} K={len(DELTA)} "
          f"(base14 + bpf{len(DELTA)-14-n_extra} + extra{n_extra})", flush=True)
    # --- ★struct禁止アサート ---
    for c in DELTA + [s for s in STD if s]:
        low = c.lower()
        assert ("gf" not in low) and ("v50b" not in low) and ("struct" not in low), f"banned col: {c}"
    # ★L5強化: v95既存特徴を C(query context)に追加(env CEXTRA, カンマ区切り。z正規化して付与)
    _cx = [x for x in os.environ.get("CEXTRA", "").split(",") if x and x in sch]
    if _cx:
        print(f"[seqfeat] CEXTRA: +{len(_cx)}ch {_cx}", flush=True)
    # ★候補別特徴(loglik=GR適合/vs_tw=tw乖離)を各候補トークンに付与(env CANDFEAT)。推論値だけでなく信頼性を入れる
    CANDFEAT = os.environ.get("CANDFEAT", "0") == "1"; CFMAP = []; cfcols = []
    _SIB = {"self": "pf_self_vs_nbr", "nbr_gr5": "pf_nbr_gr5_vs_nbr", "self_graft": "pf_self_graft_vs_self"}
    def _ok(c): return c if (c and c in sch) else None
    if CANDFEAT:
        # ★10役割/候補(役割スロット分離=意味混在なし, 0埋め):
        # [0 loglik, 1 vs_tw, 2 兄弟乖離, 3 agree_std, 4 agree_range,
        #  5 ancc gr_abs0, 6 ancc gr_best_resid, 7 ancc rate, 8 ancc vs_spatial, 9 ancc vs_z]
        for dc in DELTA:
            fs = [None] * 10
            px = "v97_" if dc.startswith("v97_") else ""    # ★combo対応: v97_接頭辞を剥がして同prefixで参照
            body = dc[len(px):]
            for rep in ("self", "nbr", "nbr_gr5", "self_graft"):
                if body.startswith(f"pf_{rep}_delta_"):
                    k = body.rsplit("_", 1)[1]
                    fs[0] = _ok(f"{px}pf_{rep}_loglik_{k}"); fs[1] = _ok(f"{px}pf_{rep}_vs_tw_{k}")
                    fs[2] = _ok(f"{px}{_SIB[rep]}_{k}") if rep in _SIB else None
                    fs[3] = _ok(f"{px}pf_agree_std_{k}"); fs[4] = _ok(f"{px}pf_agree_range_{k}")
            if body.startswith("pf_ancc_delta_"):
                k = body.rsplit("_", 1)[1]
                fs[3] = _ok(f"{px}pf_agree_std_{k}"); fs[4] = _ok(f"{px}pf_agree_range_{k}")
                fs[5] = _ok(f"{px}pf_{k}_gr_abs0"); fs[6] = _ok(f"{px}pf_{k}_gr_best_resid")
                fs[7] = _ok(f"{px}pf_{k}_rate"); fs[8] = _ok(f"{px}pf_{k}_vs_spatial"); fs[9] = _ok(f"{px}pf_{k}_vs_dense")
            CFMAP.append(fs)
        _nfm = int(os.environ.get("NF_MAX", "10")); CFMAP = [fs[:_nfm] for fs in CFMAP]   # ★役割数を絞る(sweet spot探索)
        cfcols = sorted({c for fs in CFMAP for c in fs if c})
        print(f"[seqfeat] CANDFEAT: 候補別特徴 {len(cfcols)}列 → 各トークンに{_nfm}役割(nF={_nfm})付与", flush=True)
    cols = (["id", "well", "last_known_tvt", "md_since"] + (["target"] if not NO_TARGET else [])
            + DELTA + [s for s in STD if s] + _cx + cfcols)
    cols = list(dict.fromkeys(cols))                  # ★重複列を排除(std共有等。読込時のKeyError回避)
    sch = set(pq.read_schema(PQ).names)
    _EXTRA_PQ = [p for p in os.environ.get("SF_EXTRA", "").split(",") if p]   # ★追加parquet(id整列済)から候補列を結合
    _extra_map = {}
    for _p in _EXTRA_PQ:
        for _c in pq.read_schema(_p).names:
            if _c not in sch and _c not in _extra_map:
                _extra_map[_c] = _p
    miss = [c for c in cols if c not in sch and c not in _extra_map]
    assert not miss, f"missing cols: {miss}"
    _main_cols = [c for c in cols if c in sch]
    tb = pq.read_table(PQ, columns=_main_cols)
    if _EXTRA_PQ:
        _base_id = tb.column("id").to_pandas().astype(str).to_numpy()
        _nadd = 0
        for _p in _EXTRA_PQ:
            _ec = [c for c in cols if _extra_map.get(c) == _p]
            if not _ec:
                continue
            _etb = pq.read_table(_p, columns=list(dict.fromkeys(["id"] + _ec)))
            _eid = _etb.column("id").to_pandas().astype(str).to_numpy()
            assert len(_eid) == len(_base_id) and bool((_eid == _base_id).all()), f"SF_EXTRA id不一致: {_p}"
            for c in _ec:
                tb = tb.append_column(c, _etb.column(c)); _nadd += 1
        print(f"[seqfeat] SF_EXTRA 結合: +{_nadd}列 from {len(_EXTRA_PQ)}parquet(id整列検証済)", flush=True)
    def get(c):
        return tb.column(c).to_numpy(zero_copy_only=False)
    well = tb.column("well").to_pandas().astype(str).to_numpy()
    ids = tb.column("id").to_pandas().astype(str)
    rn = ids.str.rsplit("_", n=1, expand=True)[1].astype(int).to_numpy()
    tgt = get("target").astype(np.float64) if not NO_TARGET else np.zeros(len(well))   # test は dtrue/wgt ダミー
    mds = np.maximum(get("md_since").astype(np.float64), 0.0)
    DEL = {c: get(c).astype(np.float64) for c in DELTA}
    STw = {c: get(c).astype(np.float64) for c in STD if c}
    CEX = {}                                              # ★CEXTRA: z正規化して保持
    for c in _cx:
        a = get(c).astype(np.float64); mu = np.nanmean(a); sd = np.nanstd(a) + 1e-6
        CEX[c] = np.nan_to_num((a - mu) / sd, nan=0.0)
    CF = {}                                               # ★CANDFEAT: z正規化して保持
    for c in cfcols:
        a = get(c).astype(np.float64); mu = np.nanmean(a); sd = np.nanstd(a) + 1e-6
        CF[c] = np.nan_to_num((a - mu) / sd, nan=0.0)
    del tb

    # ★実験: E-UNet/SDF 出力を外部npz(parquet行順アライメント済 δ)からチャネル注入(env EUNET_NPZ)
    #   npz の各キー = 追加チャネル名, 値 = (N,) の δ(= pred_TVT - last_known)。std は None(既定2.5)。
    if os.environ.get("EUNET_NPZ"):
        _ez = np.load(os.environ["EUNET_NPZ"])
        for _ch in _ez.files:
            _a = _ez[_ch].astype(np.float64)
            assert len(_a) == len(well), f"EUNET ch {_ch} 長さ不一致 {len(_a)}!={len(well)}"
            DEL[_ch] = _a
            DELTA.append(_ch); STD.append(None)
            if CANDFEAT:
                CFMAP.append([None] * int(os.environ.get("NF_MAX", "10")))   # ★注入chはCANDFEAT役割なし=CFMAP長をDELTAに揃える
        print(f"[seqfeat] EUNET_NPZ 注入: +{len(_ez.files)}ch {list(_ez.files)} -> K={len(DELTA)}", flush=True)

    # well ごとの連続ブロック境界(parquet 上で連続なのを利用。ビュー参照でコピー回避)
    change = np.where(well[1:] != well[:-1])[0] + 1
    starts = np.concatenate([[0], change])
    ends = np.concatenate([change, [len(well)]])
    N = len(well)

    OUTD = {}
    for wi, (s0, e0) in enumerate(zip(starts, ends)):
        wn = well[s0]
        T = e0 - s0
        # bin 順 = rn 昇順(v62 の rn ソートに合わせる。通常は既に昇順)
        loc = np.argsort(rn[s0:e0], kind="stable")
        rows = np.arange(s0, e0)[loc]              # parquet 全体での行 index(bin 順)
        nb_ = (T + STR - 1) // STR
        sel = np.clip(np.arange(nb_) * STR + STR // 2, 0, T - 1)

        # --- D(δ, bin中心行の値) / S(幅) ---
        Ds, Ss = [], []
        for dc, sc in zip(DELTA, STD):
            dv = DEL[dc][rows]
            Ds.append(dv[sel])
            if sc is None:
                Ss.append(np.full(nb_, 2.5))
            elif sc == "beam_std_d":
                Ss.append(np.clip(STw[sc][rows], 1.5, 15.0)[sel])
            else:
                Ss.append(STw[sc][rows][sel])
        Dm = np.stack(Ds, 1)
        Sm = np.clip(np.nan_to_num(np.stack(Ss, 1), nan=3.0), 1.0, 15.0)
        Dm = np.clip(np.nan_to_num(Dm, nan=0.0), -60, 60)

        # --- per-bin ターゲット dtrue / 重み wgt(= v62 の median(y_true - st) 相当。st -> last_known)
        #     bid = arange(T)//STR は連続 16 行チャンクなのでスライスで集計(高速) ---
        tw_ = tgt[rows]
        dtrue = np.full(nb_, np.nan)
        wgt = np.zeros(nb_)
        for b in range(nb_):
            vals = tw_[b * STR: min((b + 1) * STR, T)]
            fin = np.isfinite(vals)
            if fin.any():
                dtrue[b] = float(np.median(vals[fin]))
                wgt[b] = float(fin.sum())

        # --- C(文脈 3ch)。v62: [md_since/1000, (lk-st)/10, log1p(wgt)]
        #     struct-free では anchor==last_known なので (lk-anchor)=0(死にチャネルだが shape 保持) ---
        lk_minus_anchor = np.zeros(nb_)
        C = np.stack([mds[rows][sel] / 1000.0, lk_minus_anchor, np.log1p(wgt)], 1)
        if _cx:                                          # ★CEXTRA を bin中心(sel)で C に連結
            Cx = np.stack([CEX[c][rows][sel] for c in _cx], 1)
            C = np.concatenate([C, Cx], 1)

        bid = np.minimum(np.arange(T) // STR, nb_ - 1).astype(np.int32)
        OUTD[wn] = dict(
            D=Dm.astype(np.float32), S=Sm.astype(np.float32),
            C=np.nan_to_num(C).astype(np.float32),
            dtrue=dtrue.astype(np.float32), wgt=wgt.astype(np.float32),
            bid=bid, rows=rows.astype(np.int64), T=int(T))
        if CANDFEAT:                                      # ★候補別特徴 DX(nb,K,nF): 各候補トークンに loglik/vs_tw
            DXs = []
            for ci in range(len(DELTA)):
                ch = [(CF[fc][rows][sel] if fc else np.zeros(nb_)) for fc in CFMAP[ci]]
                DXs.append(np.stack(ch, 1))
            OUTD[wn]["DX"] = np.nan_to_num(np.stack(DXs, 1)).astype(np.float32)   # (nb,K,nF)
        if (wi + 1) % 150 == 0:
            print(f"  {wi+1}/{len(starts)} ({time.time()-t0:.0f}s)", flush=True)

    pickle.dump(OUTD, open(OUT, "wb"), protocol=4)
    import json as _json                                    # ★候補名sidecar(D の列順=DELTA)。L5 の PRUNE 剪定で名前→indexに使う
    _json.dump(list(DELTA), open(str(OUT) + ".cand.json", "w", encoding="utf-8"), ensure_ascii=False)
    k0 = next(iter(OUTD.values()))
    nbs = [v["D"].shape[0] for v in OUTD.values()]
    print(f"saved {OUT.name}: {len(OUTD)} wells  D{k0['D'].shape} (K=14 struct-free)")
    print(f"  N(total rows)={N}  nb: min={min(nbs)} max={max(nbs)} mean={np.mean(nbs):.1f}")
    print(f"  ({time.time()-t0:.0f}s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile build_forward_v102.py
# -*- coding: utf-8 -*-
"""v102 forward-PF feature generator(lean, PF専用パス)— build_backward_v100.py の順方向版。
create_v95 を通さない=CSV毎メソッド再読込 / imputer fit / build_well(beam/NCC/全タブラー)の無駄が無い。

build_backward からの差分:
  - 逆方向処理を全撤去(reverse_inputs / far-end warm-init / dip符号反転 なし)= 素の順方向 PF。
  - MODE(env)で照合GR加工: plain / affine(prefix較正 gr=(gr-b)/a)/ dec10(MD-DEC間引き→interp戻し)。
  - FWD_METHODS で表現を選択(既定5表現。★affine_tw は FWD_METHODS=tw の1表現のみ)。
  - FWD_SMOOTH_MODE で fixedlag(smooth16)/ full を切替。
  - emission ON(FWD_W_NN で config の w_nn を上書き。pf5 config=0.0 対策)。
  - CSV は最初に1回だけ読んでキャッシュ(HWS/TWS)=Kaggle の遅いFSでも再読込オーバーヘッド無し。
  - ★resume: per-(bank,rep) を STORE_CACHE_DIR に pkl 保存。完了済スキップ=中断再開可。

出力 = {id, well, <PRE>_*}(train と行同順, PF列のみ)。非PF特徴は base v102 と同一=生成しない。

env:
  LEG            v96|v97|v100(必須)
  MODE           plain|affine|dec10(既定 plain)
  FWD_METHODS    tw,self,nbr,nbr_gr5,self_graft のカンマ区切り(既定 全5。affine_tw=tw)
  FWD_SMOOTH_MODE fixedlag|full(既定 fixedlag)
  FWD_SMOOTH_LAG fixedlag のラグ(既定 16)
  DEC            間引き幅(MODE=dec10, 既定 10)
  OUT_PREFIX     出力列接頭(既定 MODE依存: aff/dec/fpf)
  ROGII_ART      レグ art dir(pf_banks_config.json / grfree_anchor_train.pkl / nn50 / SIM_PKL)
  SIM_PKL        emission sim pkl(あれば emission込み)
  FWD_W_NN       emission 強度(既定 0.02。全バンク強制)
  OUT_FWD        出力 parquet
  STORE_CACHE_DIR per-(bank,rep) キャッシュ dir(既定 OUT_FWD隣 fwdcache_{LEG}_{MODE})
  NWELLS         スモーク(先頭N井)
  ROGII_DATA     競技データ / PF_NGPU  run_smoother_ext が読む(Kaggle T4x2)
"""
import os, sys, io, glob, time, pickle
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False)
    sys.stderr = io.open(2, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np
import pandas as pd
import torch

_SELF = Path(os.path.abspath(__file__)).parent
sys.path.insert(0, str(_SELF))

LEG = os.environ["LEG"].lower()                      # v96 / v97 / v100
_PFMOD = {"v96": "pf_banks_v95", "v97": "pf_banks_v97", "v100": "pf_banks_v95"}[LEG]
_IMPMOD = {"v96": "imputers_v95", "v97": "imputers_v97", "v100": "imputers_v95"}[LEG]
_FEATMOD = {"v96": "features_v95", "v97": "features_v97", "v100": "features_v95"}[LEG]
pf = __import__(_PFMOD)
imp = __import__(_IMPMOD)
featv = __import__(_FEATMOD)
import affine_v102 as _aff

MODE = os.environ.get("MODE", "plain").lower()       # plain | affine | dec10 | affine_dec10(併用)
DO_AFFINE = "affine" in MODE                         # ★affine較正を掛ける
DO_DEC = "dec10" in MODE                             # ★MD-DEC 間引きを掛ける(affineと併用可)
DEC = int(os.environ.get("DEC", "10"))
_ALL_METHODS = ["tw", "self", "nbr", "nbr_gr5", "self_graft"]
METHODS = [m.strip() for m in os.environ.get("FWD_METHODS", ",".join(_ALL_METHODS)).split(",") if m.strip()]
METHODS = [m for m in _ALL_METHODS if m in METHODS]  # 正規順序
SMODE = os.environ.get("FWD_SMOOTH_MODE", "fixedlag").lower()   # fixedlag(smooth16) | full
_PRE_DEFAULT = (("aff" if DO_AFFINE else "") + ("dec" if DO_DEC else "")) or "fpf"
PRE = os.environ.get("OUT_PREFIX", _PRE_DEFAULT)

DATA_DIR = Path(os.environ.get("ROGII_DATA", str(pf.DATA_DIR)))
SEED = 4423098
N_SEED = pf.N_SEED
WELL_CHUNK = int(os.environ.get("PF_WELL_CHUNK", "40"))
BANK_ORDER = pf.BANK_ORDER
SUFFIXES = [f"_{i+1}" for i in range(len(BANK_ORDER))]
DEVICE = pf.DEVICE

SIM_PKL = Path(os.environ["SIM_PKL"]) if os.environ.get("SIM_PKL") else None
OUT_FWD = Path(os.environ.get("OUT_FWD", str(_SELF / "artifacts_v102" / "fwd" / f"{PRE}_{LEG}.parquet")))
STORE_CACHE = Path(os.environ.get("STORE_CACHE_DIR", str(OUT_FWD.parent / f"fwdcache_{LEG}_{MODE}")))
NWELLS = int(os.environ["NWELLS"]) if os.environ.get("NWELLS") else None
FWD_W_NN = float(os.environ.get("FWD_W_NN", "0.02"))
NEED_NBR = ("nbr" in METHODS) or ("nbr_gr5" in METHODS)
OFFS = featv.PF_OFFS
TAG = {"tw": "ancc", "self": "self", "nbr": "nbr", "nbr_gr5": "nbr_gr5", "self_graft": "self_graft"}


# ---------------- 照合GR加工(affine / dec10) ----------------
def _agg1(x, s):
    x = np.asarray(x, float); n = len(x); cuts = list(range(0, n, s)) + [n]
    return np.array([x[a:b].mean() for a, b in zip(cuts[:-1], cuts[1:])])


def _agg2(X, s):
    X = np.asarray(X, float); n = len(X); cuts = list(range(0, n, s)) + [n]
    return np.stack([X[a:b].mean(0) for a, b in zip(cuts[:-1], cuts[1:])], 0)


def _decimate(x):
    n = len(x["md"])
    if n < DEC * 2:
        xd = dict(x); xd["_dec_md"] = np.asarray(x["md"], float); xd["_full_md"] = xd["_dec_md"]; return xd
    xd = dict(md=_agg1(x["md"], DEC), z=_agg1(x["z"], DEC), gr=_agg1(x["gr"], DEC),
              gg=x["gg"], gmin=x["gmin"], gst=x["gst"], gs=x["gs"], ls=x["ls"], ir=x["ir"], N=x["N"], _wid=x.get("_wid"))
    if "anc" in x:
        xd["anc"] = _agg1(x["anc"], DEC); xd["ancs"] = _agg1(x["ancs"], DEC)
    if "_st" in x:
        xd["_st"] = _agg1(x["_st"], DEC); xd["_sim"] = _agg2(x["_sim"], DEC)
    xd["_full_md"] = np.asarray(x["md"], float); xd["_dec_md"] = xd["md"]
    return xd


def _attach_sim(x, wid, SIMD):
    sd = SIMD.get(wid)
    if sd is not None:
        x["_sim"] = sd["sim"]; x["_st"] = sd["st"]
    return x


def _build_rep(rep, hw, tt, tg, P, wid, NREFS, NREFS_GR5):
    if rep == "tw":
        return pf.build_smoother_inputs(hw, tt, tg, P)
    if rep == "self":
        return pf.build_inputs_self(hw, tt, tg, P)
    if rep == "nbr":
        return pf.build_inputs_nbr(hw, tt, tg, P, NREFS.get(wid, []))
    if rep == "nbr_gr5":
        return pf.build_inputs_nbr(hw, tt, tg, P, NREFS_GR5.get(wid, []))
    if rep == "self_graft":
        return pf.build_inputs_self_graft(hw, tt, tg, P)
    raise ValueError(rep)


def run_bank_rep(paths, names, bank, rep, SIMD, NREFS, NREFS_GR5, HWS, TWS):
    """1 バンク x 1 表現の順方向 smooth-PF。名前->dict(mean,std) 元行順(heel->toe)。"""
    P = pf.bank_param(bank); phys = P["_physics"]
    w_nn = FWD_W_NN                                     # ★全バンク emission 強制
    if SMODE == "full":
        P["smooth_mode"] = "full"                       # ★full平滑(config の smooth_lag を使用)
    else:
        P["smooth_lag"] = int(os.environ.get("FWD_SMOOTH_LAG", "16"))
        P["smooth_mode"] = "fixedlag"                   # ★smooth16
    inps = []; fmds = []
    _tb = time.perf_counter()
    for (hp, tp) in paths:
        wid = os.path.basename(str(hp)).split("__")[0]
        hw = HWS[wid]; tw = TWS[wid]
        tt = tw["TVT"].to_numpy(float); tg = tw["GR"].to_numpy(float)
        x = _build_rep(rep, hw, tt, tg, P, wid, NREFS, NREFS_GR5)
        if DO_AFFINE:
            x = _aff.apply_affine(pf, x, hw, tg, P)     # ★prefix較正(間引き前)
        x = pf.attach_anchor(x, wid, phys)
        _attach_sim(x, wid, SIMD)
        if DO_DEC:
            xd = _decimate(x)                            # ★affine後に間引き(併用時=affine+dec10)
            fmds.append((np.asarray(xd["_dec_md"], float), np.asarray(xd["_full_md"], float)))
            inps.append(xd)
        else:
            fmds.append(None)
            inps.append(x)
    t_build = time.perf_counter() - _tb
    _tp = time.perf_counter()
    outs = pf.run_smoother_ext(inps, P, SEED, N_SEED, WELL_CHUNK, w_nn=w_nn)
    t_pf = time.perf_counter() - _tp
    res = {}
    for i, w in enumerate(names):
        o = outs[i]
        if DO_DEC and fmds[i] is not None:
            dm, fm = fmds[i]
            if len(dm) != len(fm):
                mf = np.interp(fm, dm, np.asarray(o["mean"], float)).astype(np.float32)
                sf = np.interp(fm, dm, np.asarray(o["std"], float)).astype(np.float32)
                res[w] = dict(mean=mf, std=sf); continue
        res[w] = dict(mean=np.asarray(o["mean"], np.float32), std=np.asarray(o["std"], np.float32))
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return res, t_build, t_pf


def main():
    t_all = time.time()
    print(f"[fwd_v102] LEG={LEG} MODE={MODE} PRE={PRE} methods={METHODS} smooth={SMODE} "
          f"pf={_PFMOD} banks={BANK_ORDER} n_seed={N_SEED} "
          f"emission={'ON' if SIM_PKL and SIM_PKL.exists() else 'OFF'} w_nn={FWD_W_NN} "
          f"PF_NGPU={os.environ.get('PF_NGPU','1')} DEC={DEC if MODE=='dec10' else '-'}", flush=True)

    base = DATA_DIR / os.environ.get("FWD_SPLIT", "train")   # ★test対応: FWD_SPLIT=test で test/ を読む
    # (0) 有効井 + CSVキャッシュ(1回だけ読む)
    P0 = pf.bank_param(BANK_ORDER[0])
    names = []; paths = []; HWS = {}; TWS = {}
    hps = sorted(glob.glob(str(base / "*__horizontal_well.csv")))
    _t_read = time.time()
    for hp in hps:
        wid = os.path.basename(hp).split("__")[0]
        tp = str(base / f"{wid}__typewell.csv")
        if not os.path.exists(tp):
            continue
        hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
        if pf.build_smoother_inputs(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), P0) is None:
            continue
        names.append(wid); paths.append((hp, tp)); HWS[wid] = hw; TWS[wid] = tw
        if NWELLS and len(names) >= NWELLS:
            break
    print(f"[fwd_v102] CSVキャッシュ {len(names)}井 読込 {time.time()-_t_read:.0f}s / passes={len(BANK_ORDER)*len(METHODS)}", flush=True)

    # (1) emission sim(ストリーム読み)
    if SIM_PKL and SIM_PKL.exists():
        with open(SIM_PKL, "rb") as _sf:
            SIMD = pickle.load(_sf)
        print(f"[fwd_v102 emission] sim load: {len(SIMD)} wells (w_nn={FWD_W_NN})", flush=True)
    else:
        SIMD = {}
        print(f"[fwd_v102 emission] sim 無し -> emission OFF", flush=True)

    # (2) imputer + 近傍プール(nbr表現がある時だけ。tw限定=affine_tw では丸ごと skip=高速)
    NREFS = {}; NREFS_GR5 = {}
    if NEED_NBR:
        imp.build_imputers()
        for (hp, tp) in paths:
            wid = os.path.basename(str(hp)).split("__")[0]
            hw = HWS[wid]
            cx, cy = float(hw["X"].mean()), float(hw["Y"].mean())
            cz = float(hw["Z"].mean()) if "Z" in hw else 0.0
            refw = imp.neighbors_of(cx, cy, exclude_wid=wid)
            NREFS[wid] = [imp.ref_grtvt(r) for r, _ in refw] if len(refw) >= imp.NEED_REFS else []
            kn = hw.dropna(subset=["TVT_input", "GR"]) if "TVT_input" in hw else hw
            try:
                ng = imp.neighbors_gr_of(kn["TVT_input"].to_numpy(float), kn["GR"].to_numpy(float), cx, cy, cz, exclude_wid=wid)
                NREFS_GR5[wid] = [imp.ref_grtvt(w) for w, *_ in ng]
            except Exception:
                NREFS_GR5[wid] = []
        print(f"[fwd_v102] 近傍プール: nbr={sum(1 for v in NREFS.values() if v)} gr5={sum(1 for v in NREFS_GR5.values() if v)}", flush=True)
    else:
        print(f"[fwd_v102] nbr表現なし({METHODS})-> imputer/近傍プール skip", flush=True)

    # (3) 各バンク x 各表現 順方向 smooth-PF(★per-(bank,rep) キャッシュで復元可能)
    STORE_CACHE.mkdir(parents=True, exist_ok=True)
    STORE = {rep: {} for rep in METHODS}
    for k, bank in enumerate(BANK_ORDER):
        for rep in METHODS:
            cpath = STORE_CACHE / f"{bank}_{rep}.pkl"
            if cpath.exists():
                d = pickle.loads(cpath.read_bytes())
                if all(w in d for w in names):
                    for w in names:
                        STORE[rep][(w, bank)] = d[w]
                    print(f"[fwd-{rep}] {bank} [cache] 流用({len(names)}井)", flush=True); continue
            t0 = time.perf_counter()
            r, t_build, t_pf = run_bank_rep(paths, names, bank, rep, SIMD, NREFS, NREFS_GR5, HWS, TWS)
            for w in names:
                STORE[rep][(w, bank)] = r[w]
            cpath.write_bytes(pickle.dumps(r, protocol=4))
            print(f"[fwd-{rep}] {bank} done {time.perf_counter()-t0:.1f}s (入力構築{t_build:.1f}s / PF{t_pf:.1f}s)", flush=True)
    import gc
    del SIMD; gc.collect()

    # (4) per-well hgr/hmd/tw グリッド + id(eval行 index)
    HGR = {}; HMD = {}; TWG = {}; IDMAP = {}; LKMAP = {}
    for (hp, tp) in paths:
        wid = os.path.basename(str(hp)).split("__")[0]
        hw = HWS[wid]; tw = TWS[wid]
        tt = tw["TVT"].to_numpy(float); tg = tw["GR"].to_numpy(float)
        x0 = pf.build_smoother_inputs(hw, tt, tg, P0)
        if x0 is None:
            continue
        HGR[wid] = np.asarray(x0["gr"], np.float32); HMD[wid] = np.asarray(x0["md"], np.float32)
        TWG[wid] = (tt, tg)
        ev_idx = hw.index[hw["TVT_input"].isna()].to_numpy()
        IDMAP[wid] = np.array([f"{wid}_{int(i)}" for i in ev_idx], dtype=object)
        kn = hw[hw["TVT_input"].notna()]
        LKMAP[wid] = float(kn["TVT_input"].iloc[-1]) if len(kn) else np.nan

    # (5) <PRE>_ 特徴生成(METHODS 部分集合に対応。affine_tw=tw のみでも動く)
    present = [r for r in _ALL_METHODS if r in METHODS]
    has_tw = "tw" in present
    ref_rep = "tw" if has_tw else present[0]
    rows_id = []; rows_well = []; FEAT = {}
    def put(name, arr):
        FEAT.setdefault(name, []).append(np.asarray(arr, np.float32))

    for wid in names:
        bm_ref = {s: np.asarray(STORE[ref_rep][(wid, BANK_ORDER[k])]["mean"], np.float32) for k, s in enumerate(SUFFIXES)}
        m = min(len(v) for v in bm_ref.values())
        hgr = HGR.get(wid); hmd = HMD.get(wid)
        if hgr is not None: hgr = hgr[:m]
        if hmd is not None: hmd = hmd[:m]
        twp = TWG.get(wid); tw_tvt, tw_gr = (twp if twp else (None, None))
        lk = float(LKMAP.get(wid, np.nan))
        f = {}
        rep_stacks = {rep: [] for rep in present}
        for k, s in enumerate(SUFFIXES):
            bank = BANK_ORDER[k]
            for rep in present:
                tag = TAG[rep]
                bm = np.asarray(STORE[rep][(wid, bank)]["mean"], np.float32)[:m]
                bs = np.asarray(STORE[rep][(wid, bank)]["std"], np.float32)[:m]
                f[f"{PRE}_{tag}{s}"] = bm
                f[f"{PRE}_{tag}_std{s}"] = bs
                f[f"{PRE}_{tag}_delta{s}"] = (bm - lk).astype(np.float32)
                rep_stacks[rep].append(bm)
                if rep == "tw":
                    pref = f"{PRE}{s}"
                    if tw_tvt is not None and hgr is not None:
                        featv._add_gr_match_features(f, pref, bm, hgr, tw_tvt, tw_gr, OFFS)
                    if hmd is not None:
                        featv._add_path_shape_features(f, pref, bm, hmd)
            # 表現間比較(存在する表現のみ)
            if has_tw:
                twk = f[f"{PRE}_ancc{s}"]
                for rep in present:
                    if rep == "tw":
                        continue
                    tag = TAG[rep]
                    f[f"{PRE}_{tag}_vs_tw{s}"] = (f[f"{PRE}_{tag}{s}"] - twk).astype(np.float32)
            if "self" in present and "nbr" in present:
                f[f"{PRE}_self_vs_nbr{s}"] = (f[f"{PRE}_self{s}"] - f[f"{PRE}_nbr{s}"]).astype(np.float32)
            if len(present) >= 2:
                sstk = np.stack([f[f"{PRE}_{TAG[t]}{s}"].astype(float) for t in present], 1)
                f[f"{PRE}_agree_std{s}"] = np.nanstd(sstk, 1).astype(np.float32)
                f[f"{PRE}_agree_range{s}"] = (np.nanmax(sstk, 1) - np.nanmin(sstk, 1)).astype(np.float32)
        # cross-bank consensus(tw=ancc)
        if has_tw and len(SUFFIXES) >= 2:
            st = np.stack(rep_stacks["tw"], 1)
            pmean = st.mean(1).astype(np.float32); pmed = np.median(st, 1).astype(np.float32)
            f[f"{PRE}_ancc_mean"] = pmean; f[f"{PRE}_ancc_med"] = pmed
            f[f"{PRE}_ancc_std_between"] = st.std(1).astype(np.float32)
            f[f"{PRE}_ancc_range_between"] = (st.max(1) - st.min(1)).astype(np.float32)
            f[f"{PRE}_ancc_mean_delta"] = (pmean - lk).astype(np.float32)
            f[f"{PRE}_ancc_med_delta"] = (pmed - lk).astype(np.float32)
            if tw_tvt is not None and hgr is not None:
                featv._add_gr_match_features(f, f"{PRE}_mean", pmean, hgr, tw_tvt, tw_gr, OFFS)
            if hmd is not None:
                featv._add_path_shape_features(f, f"{PRE}_mean", pmean, hmd)
        # 非tw表現の集約(存在すれば)
        for rep in present:
            if rep == "tw":
                continue
            tag = TAG[rep]
            if len(rep_stacks[rep]) >= 2:
                RM = np.stack([x.astype(float) for x in rep_stacks[rep]], 1)
                f[f"{PRE}_{tag}_mean"] = np.nanmean(RM, 1).astype(np.float32)
                f[f"{PRE}_{tag}_pstd"] = np.nanstd(RM, 1).astype(np.float32)
        # 位置(heel からの距離=順方向)
        if hmd is not None:
            mdf = (hmd - hmd[0]).astype(np.float32)
            span = float(hmd[-1] - hmd[0]); span = span if span > 1e-6 else 1.0
            f[f"{PRE}_md_since"] = mdf; f[f"{PRE}_pos_frac"] = (mdf / span).astype(np.float32)
        ids = IDMAP.get(wid)
        if ids is not None and len(ids) >= m:
            rows_id.extend(list(ids[:m]))
        else:
            rows_id.extend([f"{wid}_{i}" for i in range(m)])
        rows_well.extend([wid] * m)
        for name, arr in f.items():
            put(name, arr)

    # (6) DataFrame 化・保存
    L = len(rows_id)
    cols = sorted(FEAT.keys())
    data = {"id": rows_id, "well": rows_well}
    for c in cols:
        parts = FEAT[c]
        arr = parts[0] if len(parts) == 1 else np.concatenate(parts)
        if len(arr) != L:
            a = np.full(L, np.nan, np.float32); a[:len(arr)] = arr; arr = a
        data[c] = arr
        FEAT[c] = None
    fdf = pd.DataFrame(data)
    OUT_FWD.parent.mkdir(parents=True, exist_ok=True)
    fdf.to_parquet(OUT_FWD, index=False)
    print(f"\nsaved: {OUT_FWD}  rows={len(fdf)} {PRE}_cols={len(cols)} wells={fdf['well'].nunique()} ({time.time()-t_all:.0f}s)", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile submit_v104.py
# -*- coding: utf-8 -*-
"""v104 提出オーケストレータ(Kaggle)。v103 submit を土台に、anchor-free dec10(sharp/smooth)を追加し、
剪定(PRUNE)を候補軸スライスで適用。v104 = v103候補 + sharp/smooth − PRUNE。
流れ:
 (1) v100の15バンク test 特徴生成(=v103 と同一)→ test_combo(15バンク)。
 (2) dec10 + tw-affine を test で生成(=v103)。
 (3) ★dec10af(anchor-free sharp/smooth)を test で生成(emission OFF, tw, lean=sharp/smooth列のみ)。
 (4) 結合 → v104 test_combo。
 (5) L5 推論: seqfeat(v103候補 + sharp/smooth)→ PRUNE で候補軸スライス → v104 L5 ckpt を平均。
 (6) TCN 推論(v104)→ blend(blend_v104.json, expansion 後処理)→ submission。
env: V104_L5_CKPTS(カンマ区切りの ckpt dir 名) / V104_PRUNE(剪定, カンマ区切り部分文字列) / V104_TCN
"""
import os, sys, io, json, pickle, subprocess, glob
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False); sys.stderr = io.open(2, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import torch, torch.nn as nn

WORK = Path(os.environ.get("ROGII_PROJ", "."))
SCR = Path(os.environ.get("ROGII_SCRIPTS", str(WORK)))
DATA = Path(os.environ["ROGII_DATA"])
PY = os.environ.get("ROGII_PY", sys.executable)
CB = Path(os.environ["V100SUB"])            # v100 submit_dataset(configs/錨/diff_cols)
V104DS = Path(os.environ["V104SUB"])        # v104 dataset(v100共有物 + dec10/dec10af config + l5_v104 ckpt + TCN + blend)
V96A = WORK / "v96_art"; V97A = WORK / "v97_art"; V100A = WORK / "v100_art"
FWD = WORK / "fwd"; FWD.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPS = ("ancc", "self", "nbr", "nbr_gr5", "self_graft")


def _run(script, env_extra, tag):
    env = dict(os.environ); env.update(env_extra); env["PYTHONUNBUFFERED"] = "1"
    print(f"\n==== {tag}: {script} ====", flush=True)
    r = subprocess.run([PY, str(SCR / script)], env=env)
    if r.returncode != 0:
        raise SystemExit(f"{tag} 失敗 ({r.returncode})")


# ============ (1) v100 の 15バンク(v103 submit と同一機構)============
def gen_v100_15banks():
    import shutil
    for a, src in [(V96A, "v95"), (V97A, "v97"), (V100A, "pf5")]:
        if not a.exists():
            shutil.copytree(V104DS / src, a)
    baseC = dict(ROGII_SCRIPTS=str(SCR), ROGII_DATA=str(DATA), ROGII_PY=PY,
                 GRFREE_GEN=str(SCR / "gen_grfree_anchor.py"), GRFREE_FOLD_ART=str(CB / "grfree_fold_art.pkl"))
    v96pq = V96A / "test.parquet"
    b = dict(baseC, ROGII_ART95=str(V96A), ROGII_ART95B=str(V96A))
    if not (V96A / "grfree_anchor_test.pkl").exists():
        _run("gen_grfree_test_anchor_v95.py", b, "v96錨")
    _run("create_v95.py", dict(b, SPLIT="test", V93_ANCHOR_PKL=str(V96A / "grfree_anchor_test.pkl"), OUT_PARQUET=str(v96pq)), "v96 create")
    v97pq = V97A / "test.parquet"; anc = str(V97A / "grfree_anchor_test.pkl"); sim = str(V97A / "sim_grfree_test_v97.pkl")
    b7 = dict(baseC, ROGII_ART97=str(V97A), ROGII_ART97B=str(V97A))
    if (V96A / "grfree_anchor_test.pkl").exists():
        __import__("shutil").copyfile(V96A / "grfree_anchor_test.pkl", anc)
    else:
        _run("gen_grfree_test_anchor_v97.py", b7, "v97錨")
    _run("nn_emission_v97.py", dict(b7, NN_SPLIT="test", V93_ANCHOR_PKL=anc), "v97 nn-emission")
    _run("create_v97.py", dict(b7, SPLIT="test", V93_ANCHOR_PKL=anc, SIM_PKL=sim, OUT_PARQUET=str(v97pq)), "v97 create")
    v100pq = V100A / "test.parquet"
    _run("create_v95.py", dict(baseC, ROGII_ART95=str(V100A), SPLIT="test",
         V93_ANCHOR_PKL=str(V96A / "grfree_anchor_test.pkl"), OUT_PARQUET=str(v100pq),
         PF_CACHE_DIR=str(V100A / "pfcache")), "v100 create(5バンク)")
    return v96pq, v97pq, v100pq


def build_combo15(v96pq, v97pq, v100pq):
    diff = json.loads((CB / "v97_diff_cols.json").read_text()); diff100 = json.loads((CB / "v100_diff_cols.json").read_text())
    id96 = pq.read_table(v96pq, columns=["id"]).to_pandas()["id"].astype(str).to_numpy()
    assert np.array_equal(id96, pq.read_table(v97pq, columns=["id"]).to_pandas()["id"].astype(str).to_numpy())
    assert np.array_equal(id96, pq.read_table(v100pq, columns=["id"]).to_pandas()["id"].astype(str).to_numpy())
    out = WORK / "data" / "test_combo15.parquet"; (WORK / "data").mkdir(parents=True, exist_ok=True)
    pf96 = pq.ParquetFile(v96pq); pf97 = pq.ParquetFile(v97pq); pf100 = pq.ParquetFile(v100pq); writer = None
    for b96, b97, b100 in zip(pf96.iter_batches(batch_size=300000), pf97.iter_batches(batch_size=300000, columns=diff),
                              pf100.iter_batches(batch_size=300000, columns=diff100)):
        cols = list(b96.columns); names = list(b96.schema.names)
        for j, c in enumerate(diff): cols.append(b97.column(j)); names.append("v97_" + c)
        for j, c in enumerate(diff100): cols.append(b100.column(j)); names.append("v100_" + c)
        tbl = pa.Table.from_arrays(cols, names=names)
        if writer is None: writer = pq.ParquetWriter(out, tbl.schema, compression="zstd")
        writer.write_table(tbl)
    writer.close(); print(f"[combo15] 列={len(pq.read_schema(out).names)}", flush=True); return out


# ============ (2)(3) dec10 + tw-affine + dec10af を test で生成 ============
def gen_v104_fwd():
    V95 = V104DS / "v95"; V97 = V104DS / "v97"; PF5 = V104DS / "pf5"; DEC = V104DS / "dec10"; DECAF = V104DS / "dec10af"
    tanc96 = str(V96A / "grfree_anchor_test.pkl"); tanc97 = str(V97A / "grfree_anchor_test.pkl")
    tasks = [  # (name, LEG, ARTENV, artdir, MODE, PRE, extra, lean_cols or None)
        ("dec10", "v96", "ROGII_ART95", str(DEC), "affine_dec10", "dec", {"DEC": "10", "FWD_METHODS": "tw,self,nbr,nbr_gr5,self_graft", "V93_ANCHOR_PKL": tanc96}, None),
        ("aff_v96", "v96", "ROGII_ART95", str(V95), "affine", "aff96", {"FWD_METHODS": "tw", "V93_ANCHOR_PKL": tanc96}, None),
        ("aff_v97", "v97", "ROGII_ART97", str(V97), "affine", "aff97", {"FWD_METHODS": "tw", "V93_ANCHOR_PKL": tanc97}, None),
        ("aff_pf5", "v100", "ROGII_ART95", str(PF5), "affine", "affpf5", {"FWD_METHODS": "tw", "V93_ANCHOR_PKL": tanc96}, None),
        # ★dec10af: anchor-free(physics_banks=[])、emission OFF、sharp(=1)/smooth(=2)を【6表現】
        #   raw dec10 = 5表現(tw/self/nbr/nbr_gr5/self_graft)+ affine_tw(affine_dec10, tw)= 6表現目
        ("dec10af", "v96", "ROGII_ART95", str(DECAF), "dec10", "decaf",
         {"DEC": "10", "FWD_METHODS": "tw,self,nbr,nbr_gr5,self_graft", "V93_ANCHOR_PKL": tanc96},
         [f"decaf_{r}_{k}_{b}" for r in ("ancc", "self", "nbr", "nbr_gr5", "self_graft") for k in ("delta", "std") for b in (1, 2)]),
        ("dec10af_aff", "v96", "ROGII_ART95", str(DECAF), "affine_dec10", "affdec",
         {"DEC": "10", "FWD_METHODS": "tw", "V93_ANCHOR_PKL": tanc96},
         [f"affdec_ancc_{k}_{b}" for k in ("delta", "std") for b in (1, 2)]),
    ]
    outs = {}
    for (name, leg, artenv, artdir, mode, pre, extra, lean) in tasks:
        outp = FWD / f"{name}_test.parquet"
        if not outp.exists():
            env = {artenv: artdir, "ROGII_DATA": str(DATA), "ROGII_PROJ": str(SCR), "LEG": leg, "MODE": mode,
                   "FWD_SMOOTH_MODE": "full", "FWD_SPLIT": "test", "OUT_PREFIX": pre, "OUT_FWD": str(outp),
                   "STORE_CACHE_DIR": str(FWD / f"cache_{name}"), "PF_NGPU": os.environ.get("PF_NGPU", "1"),
                   "FULL_VRAM_GB": os.environ.get("FULL_VRAM_GB", "8.0")}
            env.update(extra)
            _run("build_forward_v102.py", env, f"v104 test 生成 {name}")
            if lean:                       # ★lean 化: id/well + 必要列のみに絞る(dataset/メモリ節約)
                t = pq.read_table(outp, columns=["id", "well"] + lean)
                pq.write_table(t, outp, compression="zstd")
        outs[name] = (outp, lean)
    return outs


def merge_v104(combo15, fwd_outs):
    base = pq.read_table(combo15); bid = base.column("id").to_pandas().astype(str).to_numpy()
    for name, (p, lean) in fwd_outs.items():
        t = pq.read_table(p); tid = t.column("id").to_pandas().astype(str).to_numpy()
        assert len(tid) == len(bid) and bool((tid == bid).all()), f"{name} id不一致"
        add = lean if lean else [c for c in t.schema.names if c not in ("id", "well")]
        for c in add:
            base = base.append_column(c, t.column(c))
    out = WORK / "data" / "test_combo_v104.parquet"
    pq.write_table(base, out, compression="zstd"); print(f"[merge] v104 test_combo 列={len(base.schema.names)}", flush=True); return out


# ============ (5) L5 推論(v103候補 + sharp/smooth, PRUNE 適用)============
def _mk_l5(k, cdim, nf, dim):
    class L5(nn.Module):
        def __init__(s):
            super().__init__(); s.k = k
            s.enc = nn.Sequential(nn.Conv1d(2 + nf, dim, 9, padding=4), nn.GELU(), nn.Conv1d(dim, dim, 9, padding=16, dilation=4), nn.GELU(), nn.Conv1d(dim, dim, 9, padding=64, dilation=16))
            s.emb = nn.Embedding(k, dim); s.qry = nn.Sequential(nn.Linear(cdim + 4, dim), nn.GELU(), nn.Linear(dim, dim)); s.res = nn.Linear(dim, 1)
        def forward(s, D, S, C, m, DX=None):
            B, T, kk = D.shape; med = D.median(2, keepdim=True).values
            x = torch.stack([D - med, torch.log(S)] + ([DX[..., i] for i in range(DX.shape[-1])] if DX is not None else []), 2)
            x = x.permute(0, 3, 2, 1).reshape(B * kk, 2 + (DX.shape[-1] if DX is not None else 0), T)
            hc = s.enc(x).reshape(B, kk, -1, T).permute(0, 3, 1, 2) + s.emb.weight[None, None]
            stats = torch.cat([C, med / 10.0, D.std(2, keepdim=True), (D - med).abs().lt(2).float().mean(2, keepdim=True), torch.zeros_like(med)], -1)
            q = s.qry(stats); att = (hc @ q[..., None])[..., 0] / (hc.shape[-1] ** 0.5); w = att.softmax(-1)
            return (w * D).sum(-1) + 2.0 * torch.tanh(s.res(q)[..., 0])
    return L5().to(DEVICE)


def l5_v104_predict(pq_path):
    SF = WORK / "seqfeat_test_v104.pkl"
    # v103候補(51 + dec5 + aff15)+ sharp/smooth(2)
    ED = ",".join(["pf_nbr_gr5_delta_%d" % k for k in (1, 2, 3)] + ["pf_self_graft_delta_%d" % k for k in (1, 2, 3)]
                  + ["v97_pf_%s_delta_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4)]
                  + ["v100_pf_%s_delta_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4, 5)]
                  + ["dec_%s_delta_1" % t for t in REPS]
                  + ["aff96_ancc_delta_%d" % k for k in range(1, 7)] + ["aff97_ancc_delta_%d" % k for k in range(1, 5)]
                  + ["affpf5_ancc_delta_%d" % k for k in range(1, 6)]
                  + ["decaf_%s_delta_%d" % (t, b) for t in REPS for b in (1, 2)] + ["affdec_ancc_delta_1", "affdec_ancc_delta_2"])
    ES = ",".join(["pf_nbr_gr5_std_%d" % k for k in (1, 2, 3)] + ["pf_self_graft_std_%d" % k for k in (1, 2, 3)]
                  + ["v97_pf_%s_std_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4)]
                  + ["v100_pf_%s_std_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4, 5)]
                  + ["dec_%s_std_1" % t for t in REPS]
                  + ["aff96_ancc_std_%d" % k for k in range(1, 7)] + ["aff97_ancc_std_%d" % k for k in range(1, 5)]
                  + ["affpf5_ancc_std_%d" % k for k in range(1, 6)]
                  + ["decaf_%s_std_%d" % (t, b) for t in REPS for b in (1, 2)] + ["affdec_ancc_std_1", "affdec_ancc_std_2"])
    _run("build_seqfeat_v97.py", dict(SF_IN=str(pq_path), SF_OUT=str(SF), SF_NO_TARGET="1",
         CANDFEAT="1", NF_MAX="2", V97_NBANK="6", V93_INPUT="fwd", EXTRA_DELTA=ED, EXTRA_STD=ES, ROGII_SCRIPTS=str(SCR)), "v104 seqfeat")
    d = pickle.loads(SF.read_bytes()); wells = list(d.keys()); w0 = d[wells[0]]
    # ★PRUNE: 学習時と同一の候補軸スライス(cand.json の名前で index を決める)
    prune = [x.strip() for x in os.environ.get("V104_PRUNE", "").split(",") if x.strip()]
    if prune:
        names = json.loads((Path(str(SF) + ".cand.json")).read_text(encoding="utf-8"))
        keep = np.asarray([i for i, nm in enumerate(names) if not any(p in nm for p in prune)], dtype=np.int64)
        for w in wells:
            f = d[w]; f["D"] = f["D"][:, keep]; f["S"] = f["S"][:, keep]
            if f.get("DX") is not None: f["DX"] = f["DX"][:, keep]
        print(f"[L5 v104] PRUNE {prune}: 候補 {len(names)}→{len(keep)}", flush=True)
        w0 = d[wells[0]]
    K = w0["D"].shape[1]; CDIM = w0["C"].shape[1]; NF = w0["DX"].shape[2]
    N = int(max(w["rows"].max() for w in d.values())) + 1
    acc = np.zeros(N, np.float64); cnt = np.zeros(N, np.float64)
    ckdirs = [x.strip() for x in os.environ["V104_L5_CKPTS"].split(",") if x.strip()]
    nmodel = 0
    for cd in ckdirs:
        for fp in sorted(glob.glob(str(V104DS / cd / "l5_s*_f*_model.pt"))):
            ck = torch.load(fp, map_location=DEVICE)
            net = _mk_l5(K, CDIM, NF, ck.get("dim", 64) if isinstance(ck, dict) else 64)
            net.load_state_dict(ck["state"] if isinstance(ck, dict) and "state" in ck else ck); net.eval()
            with torch.no_grad():
                for w in wells:
                    f = d[w]; T = f["D"].shape[0]
                    D = torch.from_numpy(f["D"])[None].to(DEVICE); S = torch.from_numpy(f["S"])[None].to(DEVICE)
                    C = torch.from_numpy(f["C"])[None].to(DEVICE); m = torch.ones((1, T), device=DEVICE); DX = torch.from_numpy(f["DX"])[None].to(DEVICE)
                    o = net(D, S, C, m, DX)[0].cpu().numpy().astype(np.float64)
                    acc[f["rows"]] += o[f["bid"]]; cnt[f["rows"]] += 1
            nmodel += 1; del net
            if DEVICE == "cuda": torch.cuda.empty_cache()
    l5 = np.full(N, np.nan); ok = cnt > 0; l5[ok] = acc[ok] / cnt[ok]
    print(f"[L5 v104] cover%={100 * ok.mean():.1f} ({nmodel}model, ckdirs={ckdirs})", flush=True); return l5


def _warm_imp_cache():
    """★train参照構造を1回だけ構築し IMP_CACHE で全 subprocess に共有(setup 重複排除)。"""
    cp = str(WORK / "imp_cache.pkl")
    if not os.path.exists(cp):
        import importlib
        importlib.import_module("imputers_v95").save_cache(cp)
    os.environ["IMP_CACHE"] = cp
    print(f"[submit] IMP_CACHE={cp}(setup 重複排除)", flush=True)


def main():
    _warm_imp_cache()
    v96pq, v97pq, v100pq = gen_v100_15banks()
    c15 = build_combo15(v96pq, v97pq, v100pq)
    fwd = gen_v104_fwd()
    cpq = merge_v104(c15, fwd)
    l5 = l5_v104_predict(cpq)
    meta = pq.read_table(cpq, columns=["id", "last_known_tvt"]); ids = meta.column("id").to_pandas().to_numpy()
    lk = meta.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
    _bl = json.loads((V104DS / "blend_v104.json").read_text()) if (V104DS / "blend_v104.json").exists() else {}
    bwv = _bl.get("bw") or [0.0, 1.0]
    GG = _bl.get("gain_g"); GK = _bl.get("gain_k"); GS = _bl.get("gain_s"); GAIN = float(_bl.get("gain", 1.0))
    tcn = None
    if os.environ.get("V104_TCN", "1") == "1" and (V104DS / "tcn_v104_ckpt").exists():
        tcn = _tcn_v104_predict(cpq, ids)
    pred = l5 if tcn is None else (bwv[0] * tcn + bwv[1] * l5)
    if GG is not None and GK is not None and GS:      # ★後処理 expansion 較正(v100/v103 と同一式)
        pred = float(GG) * pred * (1.0 + float(GK) * np.abs(pred) / float(GS))
    elif GAIN != 1.0:
        pred = GAIN * pred
    fin = np.isfinite(pred)
    if tcn is not None:
        pred[~fin] = tcn[~fin]
    tvt = lk + pred
    sub = pd.DataFrame({"id": ids.astype(str), "tvt": tvt.astype(np.float64)})
    samp = pd.read_csv(DATA / "sample_submission.csv"); merged = samp[["id"]].merge(sub, on="id", how="left")
    out = WORK / "submission_v104.csv"; merged.to_csv(out, index=False)
    print(f"\n==== submission_v104.csv rows={len(merged)} finite%={100 * merged['tvt'].notna().mean():.2f} "
          f"(blend={bwv}, 後処理=({GG},{GK},{GS})|gain{GAIN}, PRUNE={os.environ.get('V104_PRUNE','')}) ====", flush=True)
    print(merged.head(6).to_string(index=False), flush=True)


# ---- TCN(SeqNet)推論(v103 と同一構造)----
class _ChanLN(nn.Module):
    def __init__(s, c): super().__init__(); s.ln = nn.LayerNorm(c)
    def forward(s, x): return s.ln(x.transpose(1, 2)).transpose(1, 2)
class _Block(nn.Module):
    def __init__(s, c, d, k, dr): super().__init__(); s.conv = nn.Conv1d(c, c, k, padding=d*(k-1)//2, dilation=d); s.ln = _ChanLN(c); s.pw = nn.Conv1d(c, c, 1); s.do = nn.Dropout(dr)
    def forward(s, x, m): h = torch.nn.functional.gelu(s.conv(x*m)); h = s.ln(h); h = s.do(s.pw(h)); return (x+h)*m
class _SelfAttn(nn.Module):
    def __init__(s, c, h, dr): super().__init__(); s.h = h; s.dk = c//h; s.qkv = nn.Linear(c, 3*c); s.proj = nn.Linear(c, c)
    def forward(s, x, km):
        B, L, C = x.shape; q, k, v = s.qkv(x).chunk(3, -1)
        q = q.view(B, L, s.h, s.dk).transpose(1, 2); k = k.view(B, L, s.h, s.dk).transpose(1, 2); v = v.view(B, L, s.h, s.dk).transpose(1, 2)
        o = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=km[:, None, None, :], dropout_p=0.0)
        return s.proj(o.transpose(1, 2).reshape(B, L, C))
class _PrefixEnc(nn.Module):
    def __init__(s, ci, ctx): super().__init__(); s.c1 = nn.Conv1d(ci, 64, 5, padding=2); s.c2 = nn.Conv1d(64, 64, 5, padding=8, dilation=4); s.fc = nn.Linear(128, ctx)
    def forward(s, x, m):
        h = torch.nn.functional.gelu(s.c1(x*m)); h = torch.nn.functional.gelu(s.c2(h))*m; ss = m.sum(-1)
        mean = h.sum(-1)/ss.clamp(min=1.0); mx = (h+(m-1.0)*1e4).amax(-1); mx = torch.where(ss > 0, mx, torch.zeros_like(mx))
        return s.fc(torch.cat([mean, mx], -1))*(ss > 0).float()
class _SeqNet(nn.Module):
    def __init__(s, kin, head, ch, ks, dils, dr, ua, ah, cd, uc=False, pc=4, ctx=64, nm=0):
        super().__init__(); s.head = head; s.use_ctx = uc; s.inp = nn.Conv1d(kin, ch, 1); s.indrop = nn.Dropout(cd)
        if uc: s.pre = _PrefixEnc(pc, ctx); s.ctxp = nn.Linear(ctx, ch)
        s.blocks = nn.ModuleList([_Block(ch, d, ks, dr) for d in dils]); s.use_attn = ua
        if ua: s.attn = _SelfAttn(ch, ah, dr); s.aln = nn.LayerNorm(ch)
        s.out = nn.Conv1d(ch, 5 if head == "mdn2" else 1, 1)
    def forward(s, x, m, px=None, pm=None):
        h = s.inp(s.indrop(x))*m
        if s.use_ctx and px is not None: h = (h+s.ctxp(s.pre(px, pm))[:, :, None])*m
        for b in s.blocks: h = b(h, m)
        if s.use_attn:
            q = s.aln(h.transpose(1, 2)); a = s.attn(q, (m[:, 0, :] > 0.5)); h = (h+a.transpose(1, 2))*m
        return s.out(h), None
def _mdn2_mean(o): pl = torch.sigmoid(o[:, 2]); return pl*o[:, 0]+(1-pl)*o[:, 1]


def _tcn_v104_predict(pq_path, ids_ref):
    pack = json.loads((V104DS / "submit_pack_v104.json").read_text())
    feats = pack["features"]; arch = pack["arch"]; t_mean = float(pack["t_mean"]); t_scale = float(pack["t_scale"])
    gmed = pack["prefix_stats"]["gmed"]; giqr = pack["prefix_stats"]["giqr"]
    use_flip = bool(pack["cfg"]["flip"]); use_prefix = bool(pack["cfg"]["prefix"])
    cols = list(dict.fromkeys(feats + ["well", "id", "last_known_tvt", "md_since"]))
    tb = pq.read_table(pq_path, columns=cols); N = tb.num_rows
    well = tb.column("well").to_pandas().to_numpy(); ids = tb.column("id").to_pandas().to_numpy()
    lk = tb.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
    mds = tb.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
    X = np.empty((N, len(feats)), np.float32)
    for j, c in enumerate(feats): X[:, j] = tb.column(c).to_numpy(zero_copy_only=False).astype(np.float32)
    del tb
    a2r = np.asarray(pack["abs2res"], bool); X[:, a2r] -= lk.astype(np.float32)[:, None]
    X -= np.asarray(pack["med"], np.float32); X /= np.asarray(pack["scale"], np.float32)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=8.0, neginf=-8.0); np.clip(X, -8, 8, out=X)
    K_IN = X.shape[1]
    wid_order = pd.unique(well); wr = {w: np.where(well == w)[0] for w in wid_order}
    for w in wid_order: wr[w] = wr[w][np.argsort(mds[wr[w]], kind="stable")]
    PREF = {}
    if use_prefix:
        for w in wid_order:
            f = DATA / "test" / f"{w}__horizontal_well.csv"
            if not f.exists(): continue
            df = pd.read_csv(f, usecols=["MD", "Z", "GR", "TVT_input"]).dropna(subset=["MD", "Z", "GR", "TVT_input"]).tail(arch["pre_len"])
            if len(df) < 8: continue
            a = df.to_numpy(np.float64)
            gr_n = np.clip((a[:, 2]-gmed)/giqr, -8, 8); tvt_rel = (a[:, 3]-a[-1, 3])/t_scale; z_rel = (a[:, 1]-a[-1, 1])/50.0; md_rel = (a[:, 0]-a[-1, 0])/1000.0
            PREF[w] = np.nan_to_num(np.stack([gr_n, tvt_rel, z_rel, md_rel], 1), nan=0.0, posinf=8.0, neginf=-8.0).astype(np.float32)
    def new_model(): return _SeqNet(K_IN, "mdn2", arch["ch"], arch["ksize"], arch["dilations"], arch["drop"], arch["use_attn"], arch["attn_heads"], arch["ch_drop"], uc=use_prefix, pc=arch["pre_c"], ctx=arch["ctx_dim"], nm=0).to(DEVICE)
    @torch.no_grad()
    def pw(model, r, flip):
        rr = r[::-1] if flip else r; L = len(rr); x = torch.from_numpy(np.ascontiguousarray(X[rr][None])).to(DEVICE).transpose(1, 2); m = torch.ones((1, 1, L), device=DEVICE); px = pm = None
        if use_prefix:
            a = PREF.get(well[r[0]]); a = a if a is not None else np.zeros((1, arch["pre_c"]), np.float32)
            px = torch.from_numpy(np.ascontiguousarray(a[None])).to(DEVICE).transpose(1, 2); pm = torch.ones((1, 1, len(a)), device=DEVICE)
        o = model(x, m, px, pm)[0]; p = _mdn2_mean(o.float())[0].cpu().numpy(); out = np.empty(len(r), np.float64); out[:] = p[::-1] if flip else p; return out
    acc = np.zeros(N, np.float64); nck = 0
    for s in pack.get("seeds", [0, 1, 2, 3, 4]):
        for k in range(pack.get("n_splits", 5)):
            fp = V104DS / "tcn_v104_ckpt" / f"mdn2_s{s}_fold{k}.pt"
            if not fp.exists(): print(f"  WARN missing {fp.name}"); continue
            model = new_model(); model.load_state_dict(torch.load(fp, map_location=DEVICE)); model.eval()
            pr = np.zeros(N, np.float64)
            for w in wid_order:
                r = wr[w]; p = pw(model, r, False)
                if use_flip: p = 0.5*(p+pw(model, r, True))
                pr[r] = p
            acc += pr; nck += 1; del model
            if DEVICE == "cuda": torch.cuda.empty_cache()
    nn_pred = (acc/max(nck, 1))*t_scale+t_mean
    assert np.array_equal(ids, ids_ref), "TCN と combo の id 不一致"
    print(f"[TCN v104] {nck}ckpt pred[mean]={nn_pred.mean():.3f}", flush=True); return nn_pred


if __name__ == "__main__":
    main()


In [ ]:
%%writefile submit_v103.py
# -*- coding: utf-8 -*-
"""v103 提出オーケストレータ(Kaggle)。v100 submit を土台に、脱相関の新バンク(dec10 + tw-affine, emission OFF)を追加。
流れ:
 (1) v100の15バンク test 特徴生成(gen_v96/v97/v100 = v100 submit 同一)→ test_combo(15バンク)。
 (2) dec10 + tw-affine を **test で build_forward 生成**(emission OFF, full平滑, FWD_SPLIT=test)。
 (3) test_combo に dec/aff 列を **結合**(test は小さいので安価)→ v103 test_combo。
 (4) L5 推論: v103 seqfeat(v100 ED + dec/aff)→ CV grid 全構成(gseed×K)の (seed,fold) model を平均。
 (5) TCN 推論: v103 TCN(dec/aff込み features)→ 平均。 (6) blend(blend_v103.json)→ submission。
env: V103_L5_CONFIGS(既定 全grid) / V103_TCN_CKPT / SKIP_GEN
"""
import os, sys, io, json, pickle, subprocess
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False); sys.stderr = io.open(2, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import torch, torch.nn as nn

WORK = Path(os.environ.get("ROGII_PROJ", "."))
SCR = Path(os.environ.get("ROGII_SCRIPTS", str(WORK)))
DATA = Path(os.environ["ROGII_DATA"])
PY = os.environ.get("ROGII_PY", sys.executable)
CB = Path(os.environ["V100SUB"])            # v100 submit_dataset(configs/錨/l5_fcombo_ckpt/nn/pack/diff_cols/blend)
V103DS = Path(os.environ["V103SUB"])        # v103 dataset(dec10 config/錨, v95/v97/pf5 config/錨, l5_v103*_ckpt, v103 TCN)
V96A = WORK / "v96_art"; V97A = WORK / "v97_art"; V100A = WORK / "v100_art"     # 書込可 art(configコピー先)
FWD = WORK / "fwd"; FWD.mkdir(parents=True, exist_ok=True)
COMBO = WORK; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPS = ("ancc", "self", "nbr", "nbr_gr5", "self_graft")


def _run(script, env_extra, tag):
    env = dict(os.environ); env.update(env_extra); env["PYTHONUNBUFFERED"] = "1"
    print(f"\n==== {tag}: {script} ====", flush=True)
    r = subprocess.run([PY, str(SCR / script)], env=env)
    if r.returncode != 0:
        raise SystemExit(f"{tag} 失敗 ({r.returncode})")


# ============ (1) v100 の 15バンク test 特徴(v100 submit と同一機構) ============
def gen_v100_15banks():
    import shutil
    for a, src in [(V96A, "v95"), (V97A, "v97"), (V100A, "pf5")]:
        if not a.exists():
            shutil.copytree(V103DS / src, a)                 # config+錨(+nn50)を書込可へ
    baseC = dict(ROGII_SCRIPTS=str(SCR), ROGII_DATA=str(DATA), ROGII_PY=PY,
                 GRFREE_GEN=str(SCR / "gen_grfree_anchor.py"), GRFREE_FOLD_ART=str(CB / "grfree_fold_art.pkl"))
    # v96(6バンク, emission OFF)
    v96pq = V96A / "test.parquet"
    b = dict(baseC, ROGII_ART95=str(V96A), ROGII_ART95B=str(V96A))
    if not (V96A / "grfree_anchor_test.pkl").exists():
        _run("gen_grfree_test_anchor_v95.py", b, "v96錨")
    _run("create_v95.py", dict(b, SPLIT="test", V93_ANCHOR_PKL=str(V96A / "grfree_anchor_test.pkl"), OUT_PARQUET=str(v96pq)), "v96 create")
    # v97(4バンク, emission ON=v100と同一)
    v97pq = V97A / "test.parquet"; anc = str(V97A / "grfree_anchor_test.pkl"); sim = str(V97A / "sim_grfree_test_v97.pkl")
    b7 = dict(baseC, ROGII_ART97=str(V97A), ROGII_ART97B=str(V97A))
    if (V96A / "grfree_anchor_test.pkl").exists():
        __import__("shutil").copyfile(V96A / "grfree_anchor_test.pkl", anc)
    else:
        _run("gen_grfree_test_anchor_v97.py", b7, "v97錨")
    _run("nn_emission_v97.py", dict(b7, NN_SPLIT="test", V93_ANCHOR_PKL=anc), "v97 nn-emission")
    _run("create_v97.py", dict(b7, SPLIT="test", V93_ANCHOR_PKL=anc, SIM_PKL=sim, OUT_PARQUET=str(v97pq)), "v97 create")
    # v100/pf5(5バンク, emission OFF, use_anchor=False)
    v100pq = V100A / "test.parquet"
    _run("create_v95.py", dict(baseC, ROGII_ART95=str(V100A), SPLIT="test",
         V93_ANCHOR_PKL=str(V96A / "grfree_anchor_test.pkl"), OUT_PARQUET=str(v100pq),
         PF_CACHE_DIR=str(V100A / "pfcache")), "v100 create(5バンク)")
    return v96pq, v97pq, v100pq


def build_combo15(v96pq, v97pq, v100pq):
    diff = json.loads((CB / "v97_diff_cols.json").read_text()); diff100 = json.loads((CB / "v100_diff_cols.json").read_text())
    id96 = pq.read_table(v96pq, columns=["id"]).to_pandas()["id"].astype(str).to_numpy()
    assert np.array_equal(id96, pq.read_table(v97pq, columns=["id"]).to_pandas()["id"].astype(str).to_numpy())
    assert np.array_equal(id96, pq.read_table(v100pq, columns=["id"]).to_pandas()["id"].astype(str).to_numpy())
    out = WORK / "data" / "test_combo15.parquet"; (WORK / "data").mkdir(parents=True, exist_ok=True)
    pf96 = pq.ParquetFile(v96pq); pf97 = pq.ParquetFile(v97pq); pf100 = pq.ParquetFile(v100pq); writer = None
    for b96, b97, b100 in zip(pf96.iter_batches(batch_size=300000), pf97.iter_batches(batch_size=300000, columns=diff),
                              pf100.iter_batches(batch_size=300000, columns=diff100)):
        cols = list(b96.columns); names = list(b96.schema.names)
        for j, c in enumerate(diff): cols.append(b97.column(j)); names.append("v97_" + c)
        for j, c in enumerate(diff100): cols.append(b100.column(j)); names.append("v100_" + c)
        tbl = pa.Table.from_arrays(cols, names=names)
        if writer is None: writer = pq.ParquetWriter(out, tbl.schema, compression="zstd")
        writer.write_table(tbl)
    writer.close(); print(f"[combo15] 列={len(pq.read_schema(out).names)}", flush=True); return out


# ============ (2) dec10 + tw-affine を test で生成(build_forward, emission OFF) ============
def gen_v103_fwd():
    import shutil
    V99v95 = V103DS / "v95"; V99v97 = V103DS / "v97"; PF5 = V103DS / "pf5"; DEC = V103DS / "dec10"
    tanc96 = str(V96A / "grfree_anchor_test.pkl")           # gen_v100_15banks が生成した test 錨(geometry錨=バンク非依存)
    tanc97 = str(V97A / "grfree_anchor_test.pkl")
    tasks = [  # (name, LEG, ARTENV, artdir, MODE, PRE, extra)  ★test 錨を V93_ANCHOR_PKL で渡す(physicsバンクが使用)
        ("dec10", "v96", "ROGII_ART95", str(DEC), "affine_dec10", "dec", {"DEC": "10", "FWD_METHODS": "tw,self,nbr,nbr_gr5,self_graft", "V93_ANCHOR_PKL": tanc96}),
        ("aff_v96", "v96", "ROGII_ART95", str(V99v95), "affine", "aff96", {"FWD_METHODS": "tw", "V93_ANCHOR_PKL": tanc96}),
        ("aff_v97", "v97", "ROGII_ART97", str(V99v97), "affine", "aff97", {"FWD_METHODS": "tw", "V93_ANCHOR_PKL": tanc97}),
        ("aff_pf5", "v100", "ROGII_ART95", str(PF5), "affine", "affpf5", {"FWD_METHODS": "tw", "V93_ANCHOR_PKL": tanc96}),
    ]
    outs = {}
    for (name, leg, artenv, artdir, mode, pre, extra) in tasks:
        outp = FWD / f"{name}_test.parquet"
        if not outp.exists():
            env = {artenv: artdir, "ROGII_DATA": str(DATA), "ROGII_PROJ": str(SCR), "LEG": leg, "MODE": mode,
                   "FWD_SMOOTH_MODE": "full", "FWD_SPLIT": "test", "OUT_PREFIX": pre, "OUT_FWD": str(outp),
                   "STORE_CACHE_DIR": str(FWD / f"cache_{name}"), "PF_NGPU": os.environ.get("PF_NGPU", "1"),
                   "FULL_VRAM_GB": os.environ.get("FULL_VRAM_GB", "8.0")}
            env.update(extra)
            _run("build_forward_v102.py", env, f"v103 test 生成 {name}")
        outs[name] = outp
    return outs


def merge_v103(combo15, fwd_outs):
    """test_combo15 に dec/aff の pf列を id整列で結合 → v103 test_combo(全特徴)。test小=一括で可。"""
    base = pq.read_table(combo15); bid = base.column("id").to_pandas().astype(str).to_numpy()
    for name, p in fwd_outs.items():
        t = pq.read_table(p); tid = t.column("id").to_pandas().astype(str).to_numpy()
        assert len(tid) == len(bid) and bool((tid == bid).all()), f"{name} id不一致"
        for c in t.schema.names:
            if c in ("id", "well"):
                continue
            base = base.append_column(c, t.column(c))
    out = WORK / "data" / "test_combo_v103.parquet"
    pq.write_table(base, out, compression="zstd"); print(f"[merge] v103 test_combo 列={len(base.schema.names)}", flush=True); return out


# ============ (3) L5 grid アンサンブル推論 ============
def _mk_l5(k, cdim, nf, dim):
    class L5(nn.Module):
        def __init__(s):
            super().__init__(); s.k = k
            s.enc = nn.Sequential(nn.Conv1d(2 + nf, dim, 9, padding=4), nn.GELU(), nn.Conv1d(dim, dim, 9, padding=16, dilation=4), nn.GELU(), nn.Conv1d(dim, dim, 9, padding=64, dilation=16))
            s.emb = nn.Embedding(k, dim); s.qry = nn.Sequential(nn.Linear(cdim + 4, dim), nn.GELU(), nn.Linear(dim, dim)); s.res = nn.Linear(dim, 1)
        def forward(s, D, S, C, m, DX=None):
            B, T, kk = D.shape; med = D.median(2, keepdim=True).values
            x = torch.stack([D - med, torch.log(S)] + ([DX[..., i] for i in range(DX.shape[-1])] if DX is not None else []), 2)
            x = x.permute(0, 3, 2, 1).reshape(B * kk, 2 + (DX.shape[-1] if DX is not None else 0), T)
            hc = s.enc(x).reshape(B, kk, -1, T).permute(0, 3, 1, 2) + s.emb.weight[None, None]
            stats = torch.cat([C, med / 10.0, D.std(2, keepdim=True), (D - med).abs().lt(2).float().mean(2, keepdim=True), torch.zeros_like(med)], -1)
            q = s.qry(stats); att = (hc @ q[..., None])[..., 0] / (hc.shape[-1] ** 0.5); w = att.softmax(-1)
            return (w * D).sum(-1) + 2.0 * torch.tanh(s.res(q)[..., 0])
    return L5().to(DEVICE)


def l5_v103_predict(pq_path):
    SF = WORK / "seqfeat_test_v103.pkl"
    ED = ",".join(["pf_nbr_gr5_delta_%d" % k for k in (1, 2, 3)] + ["pf_self_graft_delta_%d" % k for k in (1, 2, 3)]
                  + ["v97_pf_%s_delta_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4)]
                  + ["v100_pf_%s_delta_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4, 5)]
                  + ["dec_%s_delta_1" % t for t in REPS]
                  + ["aff96_ancc_delta_%d" % k for k in range(1, 7)] + ["aff97_ancc_delta_%d" % k for k in range(1, 5)]
                  + ["affpf5_ancc_delta_%d" % k for k in range(1, 6)])
    ES = ",".join(["pf_nbr_gr5_std_%d" % k for k in (1, 2, 3)] + ["pf_self_graft_std_%d" % k for k in (1, 2, 3)]
                  + ["v97_pf_%s_std_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4)]
                  + ["v100_pf_%s_std_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4, 5)]
                  + ["dec_%s_std_1" % t for t in REPS]
                  + ["aff96_ancc_std_%d" % k for k in range(1, 7)] + ["aff97_ancc_std_%d" % k for k in range(1, 5)]
                  + ["affpf5_ancc_std_%d" % k for k in range(1, 6)])
    _run("build_seqfeat_v97.py", dict(SF_IN=str(pq_path), SF_OUT=str(SF), SF_NO_TARGET="1",
         CANDFEAT="1", NF_MAX="2", V97_NBANK="6", V93_INPUT="fwd", EXTRA_DELTA=ED, EXTRA_STD=ES, ROGII_SCRIPTS=str(SCR)), "v103 seqfeat")
    d = pickle.loads(SF.read_bytes()); wells = list(d.keys()); w0 = d[wells[0]]
    K = w0["D"].shape[1]; CDIM = w0["C"].shape[1]; NF = w0["DX"].shape[2]
    N = int(max(w["rows"].max() for w in d.values())) + 1
    acc = np.zeros(N, np.float64); cnt = np.zeros(N, np.float64)
    configs = json.loads(os.environ["V103_L5_CONFIGS"])   # [[gseed,K],...] gs<0 = 決定版GroupKFold(l5_v103all_ckpt)
    for gs, Kf in configs:
        ckdir = (V103DS / "l5_v103all_ckpt") if gs < 0 else (V103DS / f"l5_v103g{gs}k{Kf}_ckpt")
        for s in range(int(os.environ.get("NN_SEEDS", "5"))):
            for k in range(Kf):
                fp = ckdir / f"l5_s{s}_f{k}_model.pt"
                if not fp.exists():
                    continue
                ck = torch.load(fp, map_location=DEVICE)
                net = _mk_l5(K, CDIM, NF, ck.get("dim", 64) if isinstance(ck, dict) else 64)
                net.load_state_dict(ck["state"] if isinstance(ck, dict) and "state" in ck else ck); net.eval()
                with torch.no_grad():
                    for w in wells:
                        f = d[w]; T = f["D"].shape[0]
                        D = torch.from_numpy(f["D"])[None].to(DEVICE); S = torch.from_numpy(f["S"])[None].to(DEVICE)
                        C = torch.from_numpy(f["C"])[None].to(DEVICE); m = torch.ones((1, T), device=DEVICE); DX = torch.from_numpy(f["DX"])[None].to(DEVICE)
                        o = net(D, S, C, m, DX)[0].cpu().numpy().astype(np.float64)
                        acc[f["rows"]] += o[f["bid"]]; cnt[f["rows"]] += 1
                del net
                if DEVICE == "cuda": torch.cuda.empty_cache()
    l5 = np.full(N, np.nan); ok = cnt > 0; l5[ok] = acc[ok] / cnt[ok]
    print(f"[L5 v103] cover%={100 * ok.mean():.1f} (grid {len(configs)}構成)", flush=True); return l5


def _warm_imp_cache():
    """★train参照構造(773井)を1回だけ構築し IMP_CACHE で全 subprocess に共有(create/build_forward の setup 重複排除)。"""
    cp = str(WORK / "imp_cache.pkl")
    if not os.path.exists(cp):
        import importlib
        _imp = importlib.import_module("imputers_v95")
        _imp.save_cache(cp)
    os.environ["IMP_CACHE"] = cp
    print(f"[submit] IMP_CACHE={cp}(setup 重複排除)", flush=True)


def main():
    _warm_imp_cache()
    v96pq, v97pq, v100pq = gen_v100_15banks()
    c15 = build_combo15(v96pq, v97pq, v100pq)
    fwd = gen_v103_fwd()
    cpq = merge_v103(c15, fwd)
    l5 = l5_v103_predict(cpq)
    meta = pq.read_table(cpq, columns=["id", "last_known_tvt"]); ids = meta.column("id").to_pandas().to_numpy()
    lk = meta.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
    # TCN: v103 TCN(GroupKFold=det 学習のみ)があれば blend、無ければ L5 単独。blend/後処理gain は mode 別(strong/base)
    _bl = json.loads((V103DS / "blend_v103.json").read_text()) if (V103DS / "blend_v103.json").exists() else {}
    tagk = (os.environ.get("V103_OUT_TAG", "") or "base").lstrip("_") or "base"
    _m = _bl.get(tagk) or (_bl if "bw" in _bl else {})    # mode別 {bw, gain_g/k/s, gain}。旧形式(トップに bw)も許容
    bwv = _m.get("bw") or [0.0, 1.0]                       # [w_tcn, w_l5]
    GG = _m.get("gain_g"); GK = _m.get("gain_k"); GS = _m.get("gain_s"); GAIN = float(_m.get("gain", 1.0))
    tcn = None
    if os.environ.get("V103_TCN", "1") == "1" and (V103DS / "tcn_v103_ckpt").exists():
        tcn = _tcn_v103_predict(cpq, ids)   # TCN は det 学習のみ→strong/base 両方に同じ TCN を blend
    pred = l5 if tcn is None else (bwv[0] * tcn + bwv[1] * l5)
    # ★後処理: v100 と同一の expansion 較正 p→g·p·(1+k|p|/s)(δを非線形 de-shrink, OOF で mode別 fit)。無ければ線形 gain。
    if GG is not None and GK is not None and GS:
        pred = float(GG) * pred * (1.0 + float(GK) * np.abs(pred) / float(GS))
    elif GAIN != 1.0:
        pred = GAIN * pred
    fin = np.isfinite(pred)
    if tcn is not None:
        pred[~fin] = tcn[~fin]
    tvt = lk + pred
    sub = pd.DataFrame({"id": ids.astype(str), "tvt": tvt.astype(np.float64)})
    samp = pd.read_csv(DATA / "sample_submission.csv"); merged = samp[["id"]].merge(sub, on="id", how="left")
    out = WORK / f"submission_v103{os.environ.get('V103_OUT_TAG', '')}.csv"; merged.to_csv(out, index=False)
    print(f"\n==== {out.name} rows={len(merged)} finite%={100 * merged['tvt'].notna().mean():.2f} "
          f"(blend={bwv}, 後処理=({GG},{GK},{GS})|gain{GAIN}, L5構成={len(json.loads(os.environ['V103_L5_CONFIGS']))}) ====", flush=True)
    print(merged.head(6).to_string(index=False), flush=True)


# ---- TCN(SeqNet)推論クラス(v100 submit と同一構造)----
class _ChanLN(nn.Module):
    def __init__(s, c): super().__init__(); s.ln = nn.LayerNorm(c)
    def forward(s, x): return s.ln(x.transpose(1, 2)).transpose(1, 2)
class _Block(nn.Module):
    def __init__(s, c, d, k, dr): super().__init__(); s.conv = nn.Conv1d(c, c, k, padding=d*(k-1)//2, dilation=d); s.ln = _ChanLN(c); s.pw = nn.Conv1d(c, c, 1); s.do = nn.Dropout(dr)
    def forward(s, x, m): h = torch.nn.functional.gelu(s.conv(x*m)); h = s.ln(h); h = s.do(s.pw(h)); return (x+h)*m
class _SelfAttn(nn.Module):
    def __init__(s, c, h, dr): super().__init__(); s.h = h; s.dk = c//h; s.qkv = nn.Linear(c, 3*c); s.proj = nn.Linear(c, c)
    def forward(s, x, km):
        B, L, C = x.shape; q, k, v = s.qkv(x).chunk(3, -1)
        q = q.view(B, L, s.h, s.dk).transpose(1, 2); k = k.view(B, L, s.h, s.dk).transpose(1, 2); v = v.view(B, L, s.h, s.dk).transpose(1, 2)
        o = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=km[:, None, None, :], dropout_p=0.0)
        return s.proj(o.transpose(1, 2).reshape(B, L, C))
class _PrefixEnc(nn.Module):
    def __init__(s, ci, ctx): super().__init__(); s.c1 = nn.Conv1d(ci, 64, 5, padding=2); s.c2 = nn.Conv1d(64, 64, 5, padding=8, dilation=4); s.fc = nn.Linear(128, ctx)
    def forward(s, x, m):
        h = torch.nn.functional.gelu(s.c1(x*m)); h = torch.nn.functional.gelu(s.c2(h))*m; ss = m.sum(-1)
        mean = h.sum(-1)/ss.clamp(min=1.0); mx = (h+(m-1.0)*1e4).amax(-1); mx = torch.where(ss > 0, mx, torch.zeros_like(mx))
        return s.fc(torch.cat([mean, mx], -1))*(ss > 0).float()
class _SeqNet(nn.Module):
    def __init__(s, kin, head, ch, ks, dils, dr, ua, ah, cd, uc=False, pc=4, ctx=64, nm=0):
        super().__init__(); s.head = head; s.use_ctx = uc; s.inp = nn.Conv1d(kin, ch, 1); s.indrop = nn.Dropout(cd)
        if uc: s.pre = _PrefixEnc(pc, ctx); s.ctxp = nn.Linear(ctx, ch)
        s.blocks = nn.ModuleList([_Block(ch, d, ks, dr) for d in dils]); s.use_attn = ua
        if ua: s.attn = _SelfAttn(ch, ah, dr); s.aln = nn.LayerNorm(ch)
        s.out = nn.Conv1d(ch, 5 if head == "mdn2" else 1, 1)
    def forward(s, x, m, px=None, pm=None):
        h = s.inp(s.indrop(x))*m
        if s.use_ctx and px is not None: h = (h+s.ctxp(s.pre(px, pm))[:, :, None])*m
        for b in s.blocks: h = b(h, m)
        if s.use_attn:
            q = s.aln(h.transpose(1, 2)); a = s.attn(q, (m[:, 0, :] > 0.5)); h = (h+a.transpose(1, 2))*m
        return s.out(h), None
def _mdn2_mean(o): pl = torch.sigmoid(o[:, 2]); return pl*o[:, 0]+(1-pl)*o[:, 1]


def _tcn_v103_predict(pq_path, ids_ref):
    """v103 TCN(dec/aff込み features)推論。pack=V103DS/submit_pack_v103.json, ckpt=V103DS/tcn_v103_ckpt/mdn2_s{s}_fold{k}.pt。
       v100 tcn_combo_predict 準拠(features に dec/aff を含む前提)。返り: nn_pred(δ=TVT−last_known)。"""
    pack = json.loads((V103DS / "submit_pack_v103.json").read_text())
    feats = pack["features"]; arch = pack["arch"]; t_mean = float(pack["t_mean"]); t_scale = float(pack["t_scale"])
    gmed = pack["prefix_stats"]["gmed"]; giqr = pack["prefix_stats"]["giqr"]
    use_flip = bool(pack["cfg"]["flip"]); use_prefix = bool(pack["cfg"]["prefix"])
    cols = list(dict.fromkeys(feats + ["well", "id", "last_known_tvt", "md_since"]))
    tb = pq.read_table(pq_path, columns=cols); N = tb.num_rows
    well = tb.column("well").to_pandas().to_numpy(); ids = tb.column("id").to_pandas().to_numpy()
    lk = tb.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
    mds = tb.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
    X = np.empty((N, len(feats)), np.float32)
    for j, c in enumerate(feats): X[:, j] = tb.column(c).to_numpy(zero_copy_only=False).astype(np.float32)
    del tb
    a2r = np.asarray(pack["abs2res"], bool); X[:, a2r] -= lk.astype(np.float32)[:, None]
    X -= np.asarray(pack["med"], np.float32); X /= np.asarray(pack["scale"], np.float32)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=8.0, neginf=-8.0); np.clip(X, -8, 8, out=X)
    K_IN = X.shape[1]
    wid_order = pd.unique(well); wr = {w: np.where(well == w)[0] for w in wid_order}
    for w in wid_order: wr[w] = wr[w][np.argsort(mds[wr[w]], kind="stable")]
    PREF = {}
    if use_prefix:
        for w in wid_order:
            f = DATA / "test" / f"{w}__horizontal_well.csv"
            if not f.exists(): continue
            df = pd.read_csv(f, usecols=["MD", "Z", "GR", "TVT_input"]).dropna(subset=["MD", "Z", "GR", "TVT_input"]).tail(arch["pre_len"])
            if len(df) < 8: continue
            a = df.to_numpy(np.float64)
            gr_n = np.clip((a[:, 2]-gmed)/giqr, -8, 8); tvt_rel = (a[:, 3]-a[-1, 3])/t_scale; z_rel = (a[:, 1]-a[-1, 1])/50.0; md_rel = (a[:, 0]-a[-1, 0])/1000.0
            PREF[w] = np.nan_to_num(np.stack([gr_n, tvt_rel, z_rel, md_rel], 1), nan=0.0, posinf=8.0, neginf=-8.0).astype(np.float32)
    def new_model(): return _SeqNet(K_IN, "mdn2", arch["ch"], arch["ksize"], arch["dilations"], arch["drop"], arch["use_attn"], arch["attn_heads"], arch["ch_drop"], uc=use_prefix, pc=arch["pre_c"], ctx=arch["ctx_dim"], nm=0).to(DEVICE)
    @torch.no_grad()
    def pw(model, r, flip):
        rr = r[::-1] if flip else r; L = len(rr); x = torch.from_numpy(np.ascontiguousarray(X[rr][None])).to(DEVICE).transpose(1, 2); m = torch.ones((1, 1, L), device=DEVICE); px = pm = None
        if use_prefix:
            a = PREF.get(well[r[0]]); a = a if a is not None else np.zeros((1, arch["pre_c"]), np.float32)
            px = torch.from_numpy(np.ascontiguousarray(a[None])).to(DEVICE).transpose(1, 2); pm = torch.ones((1, 1, len(a)), device=DEVICE)
        o = model(x, m, px, pm)[0]; p = _mdn2_mean(o.float())[0].cpu().numpy(); out = np.empty(len(r), np.float64); out[:] = p[::-1] if flip else p; return out
    acc = np.zeros(N, np.float64); nck = 0
    for s in pack.get("seeds", [0, 1, 2, 3, 4]):
        for k in range(pack.get("n_splits", 5)):
            fp = V103DS / "tcn_v103_ckpt" / f"mdn2_s{s}_fold{k}.pt"
            if not fp.exists(): print(f"  WARN missing {fp.name}"); continue
            model = new_model(); model.load_state_dict(torch.load(fp, map_location=DEVICE)); model.eval()
            pr = np.zeros(N, np.float64)
            for w in wid_order:
                r = wr[w]; p = pw(model, r, False)
                if use_flip: p = 0.5*(p+pw(model, r, True))
                pr[r] = p
            acc += pr; nck += 1; del model
            if DEVICE == "cuda": torch.cuda.empty_cache()
    nn_pred = (acc/max(nck, 1))*t_scale+t_mean
    assert np.array_equal(ids, ids_ref), "TCN と combo の id 不一致"
    print(f"[TCN v103] {nck}ckpt pred[mean]={nn_pred.mean():.3f}", flush=True); return nn_pred


if __name__ == "__main__":
    main()


In [ ]:
%%writefile submit_final.py
# -*- coding: utf-8 -*-
"""最終提出オーケストレータ(Kaggle, A/B 2モード)。全脚を test で再現し、モード別 blend_final.json で合成。
生成は submit_v104(dec10af=sharp/smooth 込み)を流用 → greedy が学習時と同じ103候補を再現できる。
脚:
 - L5grid  = v103 grid(gseed×K)アンサンブル。**91候補 seqfeat**(v103 ED)で学習=submit_v103.l5_v103_predict をそのまま流用。
 - greedy  = v104 貪欲サブセット(keep27, decaf 4本含む)。**103候補 seqfeat**(v104 ED)で keep → plain L5。
 - TCN     = v103 TCN(出力ブレンド用)。v104 combo の列部分集合で推論。
 - anchor  = GRフリー物理錨 δ(st − last_known)。sim_grfree_test_v97 + 103-seqfeat rows。
モード: A={L5grid,TCN,anchor} / B=A+greedy(blend_final.json の重み)。各モード後に expansion 後処理。
env SUBMIT_MODE=A/B を2回実行して submission_final_A.csv / _B.csv を作る。
"""
import os, sys, io, json, pickle, glob
try:
    sys.stdout = io.open(1, "w", encoding="utf-8", closefd=False); sys.stderr = io.open(2, "w", encoding="utf-8", closefd=False)
except Exception:
    pass
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq
import torch
import submit_v104 as S104          # gen(dec10af込み)+ _mk_l5 + _tcn(未使用)
import submit_v103 as S103          # l5_v103_predict(L5grid, 91候補)+ _tcn_v103_predict(v103 TCN)

WORK = S104.WORK; DATA = S104.DATA; V104DS = S104.V104DS; V97A = S104.V97A
SCR = S104.SCR; DEVICE = S104.DEVICE; REPS = S104.REPS


def _v104_ed_es():
    """submit_v104.l5_v104_predict と厳密一致の 103候補 ED/ES(v103 71 + sharp/smooth 12)。"""
    ED = ",".join(["pf_nbr_gr5_delta_%d" % k for k in (1, 2, 3)] + ["pf_self_graft_delta_%d" % k for k in (1, 2, 3)]
                  + ["v97_pf_%s_delta_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4)]
                  + ["v100_pf_%s_delta_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4, 5)]
                  + ["dec_%s_delta_1" % t for t in REPS]
                  + ["aff96_ancc_delta_%d" % k for k in range(1, 7)] + ["aff97_ancc_delta_%d" % k for k in range(1, 5)]
                  + ["affpf5_ancc_delta_%d" % k for k in range(1, 6)]
                  + ["decaf_%s_delta_%d" % (t, b) for t in REPS for b in (1, 2)] + ["affdec_ancc_delta_1", "affdec_ancc_delta_2"])
    ES = ",".join(["pf_nbr_gr5_std_%d" % k for k in (1, 2, 3)] + ["pf_self_graft_std_%d" % k for k in (1, 2, 3)]
                  + ["v97_pf_%s_std_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4)]
                  + ["v100_pf_%s_std_%d" % (r, k) for r in REPS for k in (1, 2, 3, 4, 5)]
                  + ["dec_%s_std_1" % t for t in REPS]
                  + ["aff96_ancc_std_%d" % k for k in range(1, 7)] + ["aff97_ancc_std_%d" % k for k in range(1, 5)]
                  + ["affpf5_ancc_std_%d" % k for k in range(1, 6)]
                  + ["decaf_%s_std_%d" % (t, b) for t in REPS for b in (1, 2)] + ["affdec_ancc_std_1", "affdec_ancc_std_2"])
    return ED, ES


def _build_seqfeat103(cpq):
    """103候補 seqfeat(v104 ED)= greedy 学習時と同一。C も 103 で構築。"""
    SF = WORK / "seqfeat_test_final103.pkl"
    ED, ES = _v104_ed_es()
    S104._run("build_seqfeat_v97.py", dict(SF_IN=str(cpq), SF_OUT=str(SF), SF_NO_TARGET="1",
              CANDFEAT="1", NF_MAX="2", V97_NBANK="6", V93_INPUT="fwd", EXTRA_DELTA=ED, EXTRA_STD=ES,
              ROGII_SCRIPTS=str(SCR)), "final 103-seqfeat")
    sf = pickle.loads(SF.read_bytes())
    names = json.loads((Path(str(SF) + ".cand.json")).read_text(encoding="utf-8"))
    return sf, names


def _l5_keep_predict(sf, names, keep_names, ckpt_dir):
    """103-seqfeat を keep_names で候補軸スライス(名前一致, 学習時と同順)→ plain L5(ckpt_dir)平均。"""
    ck = V104DS / ckpt_dir
    if not ck.exists():
        print(f"[greedy] {ckpt_dir} 無し → skip"); return None
    ks = set(keep_names); keep = [i for i, nm in enumerate(names) if nm in ks]
    miss = [nm for nm in keep_names if nm not in set(names)]
    assert not miss, f"greedy keep が 103候補に無い: {miss}"
    assert len(keep) == len(keep_names), f"keep 数不一致 {len(keep)}!={len(keep_names)}"
    keep = np.asarray(keep, dtype=np.int64)
    d = {w: dict(f) for w, f in sf.items()}
    for w in d:
        f = d[w]; f["D"] = f["D"][:, keep]; f["S"] = f["S"][:, keep]
        if f.get("DX") is not None: f["DX"] = f["DX"][:, keep]
    wells = list(d.keys()); w0 = d[wells[0]]
    K = w0["D"].shape[1]; CDIM = w0["C"].shape[1]; NF = w0["DX"].shape[2]
    N = int(max(w["rows"].max() for w in d.values())) + 1
    acc = np.zeros(N, np.float64); cnt = np.zeros(N, np.float64); nm = 0
    for fp in sorted(glob.glob(str(ck / "l5_s*_f*_model.pt"))):
        c = torch.load(fp, map_location=DEVICE)
        assert (not isinstance(c, dict)) or c.get("K", K) == K, f"greedy ckpt K={c.get('K')}≠{K}"
        net = S103._mk_l5(K, CDIM, NF, c.get("dim", 64) if isinstance(c, dict) else 64)
        net.load_state_dict(c["state"] if isinstance(c, dict) and "state" in c else c); net.eval()
        with torch.no_grad():
            for w in wells:
                f = d[w]; T = f["D"].shape[0]
                D = torch.from_numpy(f["D"])[None].to(DEVICE); Sd = torch.from_numpy(f["S"])[None].to(DEVICE)
                C = torch.from_numpy(f["C"])[None].to(DEVICE); m = torch.ones((1, T), device=DEVICE); DX = torch.from_numpy(f["DX"])[None].to(DEVICE)
                o = net(D, Sd, C, m, DX)[0].cpu().numpy().astype(np.float64)
                acc[f["rows"]] += o[f["bid"]]; cnt[f["rows"]] += 1
        nm += 1; del net
        if DEVICE == "cuda": torch.cuda.empty_cache()
    out = np.full(N, np.nan); ok = cnt > 0; out[ok] = acc[ok] / cnt[ok]
    print(f"[greedy] {nm}model cover%={100*ok.mean():.1f} K={K}", flush=True); return out


def _anchor_delta(sf103, lk, N, eval_mask):
    """GRフリー物理錨 δ = st − last_known。sim_grfree_test の st(=eval行のみ, nn_emission_v97:151)を
       103-seqfeat rows へ整列。train は全行=eval で一致するが、test は prefix(既知TVT)+ eval を含むため
       st(eval行)を **eval行サブセット(bin順)** に整列する(prefix行は提出対象外=NaNのままで可)。"""
    sim_p = V97A / "sim_grfree_test_v97.pkl"; anc = np.full(N, np.nan)
    if not sim_p.exists():
        print("[anchor] sim_grfree_test_v97.pkl 無し → anchor=nan", flush=True); return anc
    SIM = pickle.loads(sim_p.read_bytes())
    nkey = nfull = neval = nskip = 0
    for w, dd in SIM.items():
        if w not in sf103:
            continue
        nkey += 1
        st = np.asarray(dd["st"], np.float64); rows = np.asarray(sf103[w]["rows"])
        if len(st) == len(rows):                          # 全行一致(train型)
            anc[rows] = st - lk[rows]; nfull += 1
        else:                                             # eval行のみ(test型): eval subset(bin順)に整列
            erows = rows[eval_mask[rows]]                 # sf103 rows は既に bin(MD)順 → eval subset も MD順
            if len(erows) == len(st):
                anc[erows] = st - lk[erows]; neval += 1
            else:
                nskip += 1
    print(f"[anchor] key一致={nkey} 全行整列={nfull} eval整列={neval} 長さ不一致skip={nskip} "
          f"finite%={100*np.isfinite(anc).mean():.1f}", flush=True)
    return anc


def main():
    mode = os.environ.get("SUBMIT_MODE", "B").upper(); assert mode in ("A", "B")
    S104._warm_imp_cache()
    v96pq, v97pq, v100pq = S104.gen_v100_15banks()
    c15 = S104.build_combo15(v96pq, v97pq, v100pq)
    fwd = S104.gen_v104_fwd()               # ★dec10 + aff + dec10af(sharp/smooth)を test 生成
    cpq = S104.merge_v104(c15, fwd)         # v104 test_combo(103候補分の全列)
    meta = pq.read_table(cpq, columns=["id", "last_known_tvt"]); ids = meta.column("id").to_pandas().to_numpy()
    lk = meta.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64); N = len(ids)
    samp = pd.read_csv(DATA / "sample_submission.csv")
    _subids = set(samp["id"].astype(str)); eval_mask = np.array([str(i) in _subids for i in ids])  # 提出対象(eval)行
    legs = {}
    # L5grid(91候補=submit_v103 と厳密一致。cpq の decaf 列は ED に無いので無視)
    legs["L5grid"] = S103.l5_v103_predict(cpq)
    # 103-seqfeat(greedy + anchor rows 用)
    sf103, names = _build_seqfeat103(cpq)
    # greedy(mode=B のみ必要。A では計算不要=時短)
    if mode == "B":
        gk = V104DS / "greedy_keep.json"
        if gk.exists():
            kj = json.loads(gk.read_text()); kn = kj["keep"] if isinstance(kj, dict) else kj
            legs["greedy"] = _l5_keep_predict(sf103, names, kn, "l5_v104greedyg-1_ckpt")
    legs["anchor"] = _anchor_delta(sf103, lk, N, eval_mask)
    legs["TCN"] = S103._tcn_v103_predict(cpq, ids)
    # ---- blend(SUBMIT_MODE で A/B)----
    bl = json.loads((V104DS / "blend_final.json").read_text())[mode]
    W = bl["weights"]; pred = np.zeros(N, np.float64); wsum = 0.0; used = {}
    for nm, wv in W.items():
        a = legs.get(nm)
        if a is None or not np.isfinite(a).any():
            raise SystemExit(f"[blend {mode}] 脚 {nm} が欠測(weight={wv})=提出不能。生成/ckpt を確認")
        pred += wv * np.nan_to_num(a, nan=0.0); wsum += wv; used[nm] = wv
    pred /= max(wsum, 1e-9)
    GG = bl.get("g"); GK = bl.get("k"); GS = bl.get("s")
    if GG is not None and GK is not None and GS:
        pred = float(GG) * pred * (1.0 + float(GK) * np.abs(pred) / float(GS))
    for nm in ("L5grid", "TCN"):            # 欠測行の保険
        a = legs.get(nm)
        if a is None: continue
        bad = ~np.isfinite(pred) & np.isfinite(a); pred[bad] = a[bad]
    tvt = lk + pred
    sub = pd.DataFrame({"id": ids.astype(str), "tvt": tvt.astype(np.float64)})
    merged = samp[["id"]].merge(sub, on="id", how="left")
    out = WORK / f"submission_final_{mode}.csv"; merged.to_csv(out, index=False)
    print(f"\n==== {out.name} rows={len(merged)} finite%={100*merged['tvt'].notna().mean():.2f} "
          f"mode={mode} used={used} 後処理=({GG},{GK},{GS}) ====", flush=True)
    print(merged.head(6).to_string(index=False), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# ===== 実行(SUBMIT_MODE のモードで submission.csv 生成)=====
import subprocess, sys, shutil, os
r=subprocess.run([sys.executable,"submit_final.py"],env=dict(os.environ))
if r.returncode!=0: raise SystemExit(f"submit_final 異常終了 ({r.returncode})")
import pandas as pd
_f=f"submission_final_{SUBMIT_MODE}.csv"; shutil.copyfile(_f,"submission.csv")
s=pd.read_csv(_f); print(_f, len(s), "行  finite%=", round(100*s['tvt'].notna().mean(),2), " → submission.csv")
print(s.head())
